# Data Extraction Pipeline (Stages 1–4)
This notebook extracts workflow metadata, run-level metrics, step telemetry (TTFTS), and workload signatures (including artifact-based executed-test evidence).

## Stage 1 — Verify workflows and label execution-style evidence

In [1]:
# ============================================================
# Stage 1 (REWIRED): Fetch workflows from GitHub for URL_List repos,
# follow called scripts/local actions, and emit verified_workflows_v16.csv
#
# CHANGES ONLY (per request):
# 1) FIX missed Gradle connected*AndroidTest when tasks include GHA expressions
#    => sanitize ${{ ... }} expressions before regex matching.
# 2) ADD a column for step name(s) that include the test invocation
#    => test_invocation_step_names
# 3) Flutter integration tests are labeled "Flutter Integration Test" instead of "3P-CLI"
#    => flutter_project_hint dropped
# 4) ADD Detox Android E2E detection
#    => invocation_types includes "Detox"
#    => looks_like_instru=yes only when Detox + Android runtime/style evidence is present
#    => test_invocation_step_names captures the step with yarn/npx detox test
# 5) FIX Python DeprecationWarning ("Flags not at the start...")
#    => no inline flags inside shared regex fragments
# 6) ADD workflow-structure anchor position proxies (NEW)
#    => anchor_job_ordinal
#    => anchor_step_ordinal_in_job
#    (based on declared YAML order of the first detected invocation step)
#
# 7) Add STRICT emulator.wtf "indirect invocation" detection (already in your latest)
# 8) NEW (ONLY): Add STRICT BrowserStack "indirect invocation" detection
#    - Requires BrowserStack signal AND credible execution trigger (API/script/CLI/gradle task)
#    - Prevents FP from mere env/setup mentions
#
# 9) NEW (ONLY): Persist called-file instrumentation evidence for downstream stages
#    => called_instru_signal
#    => called_instru_file_paths
#    => called_instru_origin_refs
#    => called_instru_origin_step_names
#    => called_instru_file_types
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union
from urllib.parse import urlparse

import requests

try:
    import yaml  # PyYAML (not required)
except Exception:
    yaml = None

# =========================
# CONFIG (KEEP THESE AS YOUR STAGE-1 CONTRACT)
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_URL_LIST_CSV = ROOT_DIR / "URL_List.csv"               # input list of repos
OUT_STAGE1_CSV  = ROOT_DIR / "verified_workflows_v16.csv" # Stage-1 output name (original)

MAX_TOKENS_TO_USE = 7

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# follow local files referenced by workflow:
FOLLOW_CALLED_FILES = True
MAX_FOLLOW_DEPTH = 2
MAX_FOLLOW_BYTES = 1_500_000  # skip huge files

# =========================
# Helpers
# =========================
BOM = "\ufeff"
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")  # sanitize expressions like ${{ matrix.flavor }}

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()

def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            clean_row = {}
            for k, v in r.items():
                ck = _clean_key(k)
                clean_row[ck] = (v or "")
            rows.append(clean_row)
    return rows, fields

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        x = (x or "").strip()
        if not x:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join(items: List[str], max_len: int = 1500) -> str:
    s = ",".join(unique_preserve(items))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."

def safe_join_pipe(items: List[str], max_len: int = 3000) -> str:
    s = "|".join(unique_preserve(items))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."

def parse_repo_full_name(url: str) -> str:
    u = (url or "").strip()
    if not u:
        return ""
    if re.match(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$", u):
        return u
    if u.startswith("git@github.com:"):
        u2 = u.split("git@github.com:", 1)[1]
        u2 = u2[:-4] if u2.endswith(".git") else u2
        return u2.strip("/")
    if "github.com" in u:
        try:
            p = urlparse(u)
            parts = [x for x in (p.path or "").split("/") if x]
            if len(parts) >= 2:
                return f"{parts[0]}/{parts[1].replace('.git','')}"
        except Exception:
            return ""
    return ""

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def sanitize_gha_expr(text: str) -> str:
    # "connected${{ matrix.flavor }}DebugAndroidTest" -> "connectedDebugAndroidTest"
    return GHA_EXPR_RE.sub("", text or "")

def normalize_repo_rel_path(ref: str) -> str:
    rr = (ref or "").replace("\\", "/").strip()
    rr = rr[2:] if rr.startswith("./") else rr
    rr = rr.lstrip("/")
    while "//" in rr:
        rr = rr.replace("//", "/")
    return rr

# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "stage1-v16-workflow-scan/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# =========================
# GitHub endpoints
# =========================
def get_repo_meta(gh: GitHubClient, full_name: str) -> Dict[str, str]:
    url = f"https://api.github.com/repos/{full_name}"
    data = gh.request_json("GET", url, params={})
    if not isinstance(data, dict):
        return {}
    return {
        "default_branch": (data.get("default_branch") or "").strip(),
        "archived": str(bool(data.get("archived"))).lower(),
        "private": str(bool(data.get("private"))).lower(),
    }

def list_workflows(gh: GitHubClient, full_name: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows"
    data = gh.request_json("GET", url, params={"per_page": 100})
    if not isinstance(data, dict):
        return []
    wfs = data.get("workflows", [])
    return wfs if isinstance(wfs, list) else []

def fetch_file_text_at_ref(gh: GitHubClient, full_name: str, path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref} if ref else {})
    if not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text or ""
        except requests.exceptions.RequestException:
            return ""
    return ""

def file_size_at_ref(gh: GitHubClient, full_name: str, path: str, ref: str) -> Optional[int]:
    url = f"https://api.github.com/repos/{full_name}/contents/{path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref} if ref else {})
    if not isinstance(data, dict):
        return None
    sz = data.get("size")
    try:
        return int(sz)
    except Exception:
        return None

# =========================
# Signal detection patterns (Stage-1)
# =========================
GRADLE_INVOKE_PREFIX = r"(?:^|[ \t\r\n;&|()\"'`])"

GRADLE_CMD_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradlew\.bat\b|gradle\s+)",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# Flutter integration tests (hint + android-targeted)
FLUTTER_IT_RE = re.compile(r"\bflutter\s+(?:test|drive)\b", flags=re.IGNORECASE | re.DOTALL)
FLUTTER_IT_ANDROID_HINT_RE = re.compile(r"\b(integration_test|--driver\b|test_driver)\b", flags=re.IGNORECASE | re.DOTALL)
FLUTTER_DEVICE_FLAG_RE = re.compile(r"\s+-d\s+(?P<dev>\"[^\"]+\"|'[^']+'|\S+)", flags=re.IGNORECASE | re.DOTALL)
FLUTTER_DEVICE_IS_ANDROID_RE = re.compile(
    r"\b(android|emulator-\d+|sdk\s+gphone|android\s+sdk\s+built\s+for|pixel)\b",
    flags=re.IGNORECASE | re.DOTALL,
)

# Detox Android E2E invocation (CLI via yarn/npm/pnpm/npx)
DETOX_INVOKE_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}("
    r"(?:yarn|npm|pnpm)\s+[^\n\r]*\bdetox(?::[a-z0-9:_-]+)?\b|"
    r"(?:npx\s+detox\s+test\b)|"
    r"(?:detox\s+test\b)"
    r")",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# --- GMD tasks ---
GMD_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"manageddevice[\w:-]*(check|androidtest|test|setup)\b|"
    r":[\w:-]*manageddevice[\w:-]*(check|androidtest|test|setup)\b"
    r")",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

GMD_MANAGEDDEV_PROP_RE = re.compile(
    r"\B-Pandroid\.(?:testoptions\.manageddevices|experimental\.testOptions\.managedDevices)\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

GENERIC_ANDROIDTEST_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b(?!connected)\w+androidtest\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

CONNECTED_ANDROIDTEST_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"connected\w*androidtest|connectedcheck|devicecheck|alldevicescheck|"
    r"device\w*androidtest"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

BASELINE_PROFILE_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"generate\w*baselineprofile|collect\w*baselineprofile|baselineprofile"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

ADB_INSTR_RE = re.compile(r"\badb\s+shell\s+am\s+instrument\b|\bam\s+instrument\b", flags=re.IGNORECASE | re.DOTALL)

EMU_COMMUNITY_ACTION_RE = re.compile(
    r"\b("
    r"reactivecircus/android-emulator-runner|"
    r"malinskiy/action-android/emulator-run-cmd|"
    r"android-emulator-runner"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

EMU_CUSTOM_RUNTIME_RE = re.compile(
    r"\b("
    r"\bemulator\b.*\b-avd\b|"
    r"\bavdmanager\b|"
    r"adb\s+wait[- ]?for[- ]?device"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

REAL_DEVICE_ADB_RE = re.compile(
    r"\badb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b",
    flags=re.IGNORECASE | re.MULTILINE,
)

THIRD_PARTY_PROVIDER_NAME_RE = re.compile(
    r"\b("
    r"firebase\s+test\s+lab|gcloud\s+firebase|"
    r"browserstack|bstack|hub\.browserstack\.com|"
    r"sauce(labs)?|saucectl|"
    r"appcenter|microsoft/appcenter|"
    r"emulator\.wtf|"
    r"maestro\s+cloud"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

THIRD_PARTY_INVOKE_RE = re.compile(
    r"\b("
    r"(gcloud\s+firebase\s+test\s+android\s+run\b)|"
    r"(firebase\s+test\s+android\s+run\b)|"
    r"(flank\s+android\s+run\b)|"
    r"(appcenter\s+test\s+run\s+android\b)|"
    r"(appcenter\s+test\s+run\s+espresso\b)|"
    r"(saucectl\s+(run|test)\b)|"
    r"(emulator-wtf/run-tests@)|"
    r"(maestro\s+cloud\b)"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

THIRD_PARTY_SETUP_ONLY_RE = re.compile(
    r"\b("
    r"google-github-actions/(auth|setup-gcloud)|"
    r"gcloud\s+auth|"
    r"gcloud\s+config\s+set"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# =========================
# STRICT indirect 3P invocation detectors
# =========================

# --- emulator.wtf (already added previously) ---
EMULATOR_WTF_SIGNAL_RE = re.compile(
    r"\b("
    r"emulator\.wtf|"
    r"emulator_wtf|"
    r"ew_api_token|"
    r"emulatorwtf_token|"
    r"emulator_wtf_token"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

EMULATOR_WTF_GRADLE_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"(?:[:\w.-]+)?emulatorwtf(?:[:\w.-]+)?"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

def _is_emulator_wtf_indirect_invoke(text: str) -> bool:
    low = sanitize_gha_expr(text or "").lower()
    return bool(EMULATOR_WTF_SIGNAL_RE.search(low)) and bool(EMULATOR_WTF_GRADLE_TASK_RE.search(low))

# --- NEW: BrowserStack strict indirect invoke ---
# Signal: domain, bs:// app ids, common secrets/envs, local tunnel, sdk token
BROWSERSTACK_SIGNAL_RE = re.compile(
    r"\b("
    r"api-cloud\.browserstack\.com|"
    r"hub\.browserstack\.com|"
    r"browserstack(local)?|"
    r"\bbs://|"
    r"browserstack_username|browserstack_access(_)?key|"
    r"bstack(_)?(username|access(_)?key)|"
    r"browserstack-sdk"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# Execution trigger: API call, bstack CLI, explicit scripts, or gradle tasks containing browserstack
BROWSERSTACK_EXEC_TRIGGER_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}("
    r"curl\s+[^\n\r]*(api-cloud\.browserstack\.com|hub\.browserstack\.com)|"
    r"(?:python|python3)\s+[^\n\r]*browserstack[^\s]*\.(?:py)\b|"
    r"node\s+[^\n\r]*browserstack[^\s]*\.(?:js|mjs|cjs)\b|"
    r"(?:yarn|npm|pnpm)\s+[^\n\r]*\bbrowserstack\b|"
    r"\bbstack\b\s+[^\n\r]*(?:run|execute|test|app-automate|appautomate|espresso|xcuitest|appium)\b|"
    r"(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\bbrowserstack\b"
    r")",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

def _is_browserstack_indirect_invoke(text: str) -> bool:
    low = sanitize_gha_expr(text or "").lower()
    return bool(BROWSERSTACK_SIGNAL_RE.search(low)) and bool(BROWSERSTACK_EXEC_TRIGGER_RE.search(low))

# =========================
# Follow-called-files extraction (UNCHANGED)
# =========================
LOCAL_USES_RE = re.compile(r'(?mi)^\s*uses\s*:\s*(?P<ref>\./\S+?)(?:\s+#.*)?$')
WORKDIR_RE = re.compile(r'(?mi)^\s*working-directory\s*:\s*(?P<wd>[^\n#]+)')

SCRIPT_CALL_RE = re.compile(r'''(?mix)
(?:^|[;&|()\s"'`])
(?:(?:bash|sh|pwsh|powershell|python|python3|node|ruby)\s+)?
(?P<path>(?:\./|\.\\)?[\w./\\-]+\.(?:sh|ps1|bat|cmd|py|js|rb|pl))
(?:\s|$)
''')

GENERIC_REL_EXEC_RE = re.compile(r'(?m)(?:^|[;&|()\s"\'`])(?P<path>\./[A-Za-z0-9_./\\-]+)(?:\s|$)')
CONFIG_ARG_RE = re.compile(r'(?mi)\b--config(?:=|\s+)(?P<path>[^\s"\']+)')

NO_FOLLOW_BASENAMES = {"gradlew", "gradlew.bat", "gradle", "adb", "flutter", "gcloud", "java", "python", "python3"}

def _strip_quotes(s: str) -> str:
    return (s or "").strip().strip('"').strip("'").strip("`")

def is_dynamic_ref(ref: str) -> bool:
    r = ref or ""
    return ("${{" in r) or ("${" in r) or ("$(" in r) or ("%{" in r)

def extract_workdirs(text: str) -> List[str]:
    wds = []
    for m in WORKDIR_RE.finditer(text or ""):
        wd = _strip_quotes(m.group("wd"))
        if wd:
            wd = wd.replace("\\", "/").lstrip("./")
            wds.append(wd)
    return unique_preserve(wds)

def extract_references(text: str) -> List[str]:
    refs: List[str] = []

    for m in LOCAL_USES_RE.finditer(text or ""):
        ref = _strip_quotes(m.group("ref"))
        if "@" in ref:
            ref = ref.split("@", 1)[0]
        refs.append(ref)

    for m in SCRIPT_CALL_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))

    for m in CONFIG_ARG_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))

    for m in GENERIC_REL_EXEC_RE.finditer(text or ""):
        p = _strip_quotes(m.group("path"))
        base = Path(p.replace("\\", "/")).name.lower()
        if base in NO_FOLLOW_BASENAMES:
            continue
        refs.append(p)

    out = []
    for r in refs:
        if not r:
            continue
        out.append(r.replace("\\", "/").strip())
    return unique_preserve(out)

def normalize_ref_path(ref: str) -> str:
    rr = (ref or "").replace("\\", "/").strip()
    rr = rr[2:] if rr.startswith("./") else rr
    rr = rr.lstrip("/")
    return rr

def candidate_paths_for_ref(ref: str, workdirs: List[str]) -> List[str]:
    rr = normalize_ref_path(ref)
    prefixes = [""] + [wd.strip("/").replace("\\", "/") for wd in (workdirs or []) if wd.strip()]
    out = []
    for pref in prefixes:
        p = f"{pref}/{rr}" if pref else rr
        out.append(p.strip("/"))
    return unique_preserve(out)

def possible_action_ymls(path: str) -> List[str]:
    p = path.strip("/")
    return unique_preserve([f"{p}/action.yml", f"{p}/action.yaml"])

def classify_called_file_type(path: str) -> str:
    low = (path or "").lower().replace("\\", "/")
    if low.endswith("action.yml") or low.endswith("action.yaml"):
        return "local_action"
    if "/.github/workflows/" in f"/{low}":
        return "local_workflow"
    return "script"

# =========================
# Step parsing for invocation step names + anchor ordinals (UNCHANGED)
# =========================
STEP_NAME_LINE_RE = re.compile(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", re.MULTILINE)

def _count_leading_spaces(s: str) -> int:
    return len(s) - len(s.lstrip(" "))

def parse_workflow_step_records(yaml_text: str) -> List[Dict[str, Union[str, int]]]:
    if not yaml_text:
        return []

    lines = yaml_text.splitlines()
    n = len(lines)
    out: List[Dict[str, Union[str, int]]] = []

    jobs_idx = None
    for i, line in enumerate(lines):
        if re.match(r"^\s*jobs\s*:\s*$", line):
            jobs_idx = i
            break
    if jobs_idx is None:
        return out

    jobs_indent = _count_leading_spaces(lines[jobs_idx])
    i = jobs_idx + 1
    job_ordinal = 0

    while i < n:
        line = lines[i]
        if line.strip() == "":
            i += 1
            continue

        indent = _count_leading_spaces(line)
        if indent <= jobs_indent:
            break

        m_job = re.match(r"^\s*([A-Za-z0-9_.-]+)\s*:\s*$", line)
        if not m_job or indent != jobs_indent + 2:
            i += 1
            continue

        job_id = m_job.group(1).strip()
        job_ordinal += 1

        block_start = i + 1
        j = block_start
        while j < n:
            nxt = lines[j]
            if nxt.strip() == "":
                j += 1
                continue
            nxt_indent = _count_leading_spaces(nxt)
            if nxt_indent <= indent:
                break
            j += 1
        job_block_lines = lines[block_start:j]
        job_block = "\n".join(job_block_lines)

        m_name = re.search(r"(?mi)^\s*name\s*:\s*(.+?)\s*$", job_block)
        job_name = m_name.group(1).strip().strip('"').strip("'") if m_name else ""

        job_lines = job_block_lines
        steps_idx = None
        steps_indent = None
        for k, jl in enumerate(job_lines):
            if re.match(r"^\s*steps\s*:\s*$", jl):
                steps_idx = k
                steps_indent = _count_leading_spaces(jl)
                break

        if steps_idx is not None and steps_indent is not None:
            step_ordinal = 0
            k = steps_idx + 1
            while k < len(job_lines):
                cur = job_lines[k]
                if cur.strip() == "":
                    k += 1
                    continue
                cur_indent = _count_leading_spaces(cur)
                if cur_indent <= steps_indent:
                    break

                m_step = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", cur)
                if not m_step:
                    k += 1
                    continue

                base_indent = len(m_step.group(1))
                step_name = m_step.group(2).strip().strip('"').strip("'")
                block = [cur]
                kk = k + 1
                while kk < len(job_lines):
                    nxt = job_lines[kk]
                    m2 = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", nxt)
                    if m2 and len(m2.group(1)) == base_indent:
                        break
                    if nxt.strip() and _count_leading_spaces(nxt) <= steps_indent:
                        break
                    block.append(nxt)
                    kk += 1

                step_ordinal += 1
                out.append({
                    "job_id": job_id,
                    "job_name": job_name,
                    "job_ordinal": job_ordinal,
                    "step_name": step_name,
                    "step_ordinal_in_job": step_ordinal,
                    "step_block": "\n".join(block),
                })

                k = kk
        i = j

    return out

def build_origin_ref_to_step_names(workflow_yaml_text: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for rec in parse_workflow_step_records(workflow_yaml_text):
        step_name = str(rec.get("step_name") or "").strip()
        step_block = str(rec.get("step_block") or "")
        refs = extract_references(step_block)
        for r in refs:
            rr = normalize_repo_rel_path(r)
            if not rr:
                continue
            out.setdefault(rr, [])
            out[rr] = unique_preserve(out[rr] + [step_name])
    return out

def _flutter_androidish_from_text(text: str, runtime_ev: Dict[str, bool]) -> bool:
    t = text or ""
    t2 = sanitize_gha_expr(t)
    low = t2.lower()

    if not (FLUTTER_IT_RE.search(low) and FLUTTER_IT_ANDROID_HINT_RE.search(low)):
        return False

    targeted = False
    for m in FLUTTER_DEVICE_FLAG_RE.finditer(low):
        dev = (m.group("dev") or "").strip().strip('"').strip("'").lower()
        if FLUTTER_DEVICE_IS_ANDROID_RE.search(dev):
            targeted = True
            break
    if FLUTTER_DEVICE_IS_ANDROID_RE.search(low):
        targeted = True

    if targeted:
        return True

    if runtime_ev.get("emu_comm") or runtime_ev.get("emu_custom") or runtime_ev.get("real_device") or runtime_ev.get("third_party_invoke"):
        return True

    return False

def _detox_androidish_from_text(text: str, runtime_ev: Dict[str, bool]) -> bool:
    t = text or ""
    t2 = sanitize_gha_expr(t)
    low = t2.lower()

    if not DETOX_INVOKE_RE.search(low):
        return False

    if runtime_ev.get("emu_comm") or runtime_ev.get("emu_custom") or runtime_ev.get("real_device") or runtime_ev.get("third_party_invoke"):
        return True

    return False

def _is_third_party_invoke_non_flutter(text: str) -> bool:
    low = sanitize_gha_expr(text or "").lower()
    return bool(THIRD_PARTY_INVOKE_RE.search(low)) and not bool(
        THIRD_PARTY_SETUP_ONLY_RE.search(low) and not THIRD_PARTY_INVOKE_RE.search(low)
    )

def extract_test_invocation_step_names_and_anchor(
    gh: GitHubClient,
    full_name: str,
    base_ref: str,
    workflow_yaml_text: str,
) -> Tuple[List[str], Optional[int], Optional[int]]:
    if not workflow_yaml_text:
        return [], None, None

    step_records = parse_workflow_step_records(workflow_yaml_text)
    if not step_records:
        return [], None, None

    step_names: List[str] = []
    anchor_job_ordinal: Optional[int] = None
    anchor_step_ordinal_in_job: Optional[int] = None

    ev_full = scan_text_for_evidence(workflow_yaml_text)
    runtime_ev = {
        "emu_comm": bool(ev_full.get("emu_comm")),
        "emu_custom": bool(ev_full.get("emu_custom")),
        "real_device": bool(ev_full.get("real_device")),
        "third_party_invoke": bool(ev_full.get("third_party_invoke")),
    }

    for rec in step_records:
        step_name = str(rec.get("step_name") or "")
        blk = str(rec.get("step_block") or "")
        blk_s = sanitize_gha_expr(blk)
        low = blk_s.lower()

        direct_connected = bool(CONNECTED_ANDROIDTEST_RE.search(low))
        direct_gmd_task = bool(GMD_TASK_RE.search(low))
        direct_gmd_prop = bool(GMD_MANAGEDDEV_PROP_RE.search(low))
        direct_generic_androidtest = bool(GENERIC_ANDROIDTEST_TASK_RE.search(low))
        direct_baseline = bool(BASELINE_PROFILE_TASK_RE.search(low))
        direct_adb = bool(ADB_INSTR_RE.search(low))

        direct_3p = _is_third_party_invoke_non_flutter(low)
        direct_emu_wtf_indirect = _is_emulator_wtf_indirect_invoke(low)
        direct_bs_indirect = _is_browserstack_indirect_invoke(low)  # NEW

        direct_flutter_androidish = _flutter_androidish_from_text(low, runtime_ev)
        direct_detox_androidish = _detox_androidish_from_text(low, runtime_ev)

        direct_gmd = bool(direct_gmd_task or (direct_gmd_prop and (direct_baseline or direct_generic_androidtest)))

        matched_here = False
        if (
            direct_connected
            or direct_gmd
            or direct_baseline
            or direct_adb
            or direct_3p
            or direct_emu_wtf_indirect
            or direct_bs_indirect          # NEW
            or direct_flutter_androidish
            or direct_detox_androidish
        ):
            matched_here = True
        else:
            refs = extract_references(blk)
            wds = extract_workdirs(blk)

            for r in refs:
                if not r or is_dynamic_ref(r):
                    continue
                candidates = candidate_paths_for_ref(r, wds)
                is_prob_action = bool(r.strip().startswith("./")) and (r.endswith("/") or "/" in r)

                found_invoke_in_called = False

                for c in candidates:
                    if is_prob_action and (not c.lower().endswith((".yml", ".yaml", ".sh", ".ps1", ".py", ".js", ".rb", ".pl", ".bat", ".cmd"))):
                        for ay in possible_action_ymls(c):
                            txt = fetch_file_text_at_ref(gh, full_name, ay, base_ref)
                            if not txt:
                                continue
                            ev = scan_text_for_evidence(txt)
                            if compute_looks_like_instru(ev) == "yes":
                                found_invoke_in_called = True
                                break
                        if found_invoke_in_called:
                            break

                    txt = fetch_file_text_at_ref(gh, full_name, c, base_ref)
                    if not txt:
                        continue
                    ev = scan_text_for_evidence(txt)
                    if compute_looks_like_instru(ev) == "yes":
                        found_invoke_in_called = True
                        break

                if found_invoke_in_called:
                    matched_here = True
                    break

        if matched_here:
            step_names.append(step_name)
            if anchor_job_ordinal is None:
                try:
                    anchor_job_ordinal = int(rec.get("job_ordinal"))  # type: ignore[arg-type]
                except Exception:
                    anchor_job_ordinal = None
                try:
                    anchor_step_ordinal_in_job = int(rec.get("step_ordinal_in_job"))  # type: ignore[arg-type]
                except Exception:
                    anchor_step_ordinal_in_job = None

    return unique_preserve(step_names), anchor_job_ordinal, anchor_step_ordinal_in_job

# =========================
# Scan logic
# =========================
def detect_provider_names(text: str) -> List[str]:
    t = (text or "").lower()
    names = []
    if re.search(r"\b(gcloud\s+firebase|firebase\s+test\s+lab|firebase\s+test\s+android\s+run)\b", t):
        names.append("Firebase Test Lab")
    if re.search(r"\bbrowserstack|bstack|hub\.browserstack\.com\b", t):
        names.append("BrowserStack")
    if re.search(r"\bsauce(labs)?|saucectl\b", t):
        names.append("Sauce Labs")
    if re.search(r"\b(appcenter|microsoft/appcenter)\b", t):
        names.append("App Center")
    if re.search(r"\bemulator\.wtf|emulator-wtf/run-tests@\b", t):
        names.append("emulator.wtf")
    if re.search(r"\bmaestro\s+cloud\b", t):
        names.append("Maestro Cloud")
    return unique_preserve(names)

def scan_text_for_evidence(text: str) -> Dict[str, Union[bool, List[str]]]:
    txt = sanitize_gha_expr(text or "")
    low = txt.lower()

    has_gradle = bool(GRADLE_CMD_RE.search(low))

    baseline = bool(BASELINE_PROFILE_TASK_RE.search(low))
    adb = bool(ADB_INSTR_RE.search(low))

    gmd_prop = bool(GMD_MANAGEDDEV_PROP_RE.search(low))
    generic_androidtest = bool(GENERIC_ANDROIDTEST_TASK_RE.search(low))

    gmd_task = bool(GMD_TASK_RE.search(low))
    gmd = bool(gmd_task or (gmd_prop and (baseline or generic_androidtest)))

    connected = bool(CONNECTED_ANDROIDTEST_RE.search(low))

    emu_comm = bool(EMU_COMMUNITY_ACTION_RE.search(low))
    emu_custom = bool(EMU_CUSTOM_RUNTIME_RE.search(low))
    real_device = bool(REAL_DEVICE_ADB_RE.search(low))

    # Direct 3P invoke
    tp_invoke_direct = bool(THIRD_PARTY_INVOKE_RE.search(low)) and not bool(
        THIRD_PARTY_SETUP_ONLY_RE.search(low) and not THIRD_PARTY_INVOKE_RE.search(low)
    )

    # Strict indirect invokes
    tp_invoke_emu_wtf_indirect = _is_emulator_wtf_indirect_invoke(low)
    tp_invoke_bs_indirect = _is_browserstack_indirect_invoke(low)  # NEW

    tp_invoke = bool(tp_invoke_direct or tp_invoke_emu_wtf_indirect or tp_invoke_bs_indirect)

    tp_providers = detect_provider_names(low) if THIRD_PARTY_PROVIDER_NAME_RE.search(low) else []

    runtime_ev = {
        "emu_comm": emu_comm,
        "emu_custom": emu_custom,
        "real_device": real_device,
        "third_party_invoke": tp_invoke,
    }

    flutter_androidish = _flutter_androidish_from_text(txt, runtime_ev)
    detox_androidish = _detox_androidish_from_text(txt, runtime_ev)

    return {
        "has_gradle": has_gradle,
        "gmd": gmd,
        "connected": connected,
        "baseline": baseline,
        "adb": adb,
        "emu_comm": emu_comm,
        "emu_custom": emu_custom,
        "real_device": real_device,
        "third_party_invoke": tp_invoke,
        "third_party_providers": tp_providers,
        "flutter_androidish_invoke": flutter_androidish,
        "detox_androidish_invoke": detox_androidish,
    }

def merge_evidence(a: Dict, b: Dict) -> Dict:
    out = dict(a)
    for k, v in b.items():
        if isinstance(v, bool):
            out[k] = bool(out.get(k, False) or v)
        elif isinstance(v, list):
            out[k] = unique_preserve((out.get(k, []) or []) + v)
        else:
            out[k] = v
    return out

def compute_invocation_types(ev: Dict) -> List[str]:
    inv: List[str] = []

    if ev.get("detox_androidish_invoke"):
        inv.append("Detox")

    if ev.get("flutter_androidish_invoke"):
        inv.append("Flutter Integration Test")
    else:
        if ev.get("third_party_invoke"):
            inv.append("3P-CLI")

    if ev.get("adb"):
        inv.append("ADB")
    if ev.get("gmd"):
        inv.append("Gradle_GMD")
    if ev.get("connected"):
        inv.append("Gradle_Connected")
    if ev.get("baseline"):
        inv.append("Gradle_BaselineProfile")
    if ev.get("has_gradle") and (ev.get("gmd") or ev.get("connected") or ev.get("baseline")):
        inv.append("Gradle")

    return sorted(set(inv))

def compute_styles(ev: Dict) -> List[str]:
    styles: List[str] = []
    if ev.get("third_party_invoke"):
        styles.append("Third-Party")
    if ev.get("gmd"):
        styles.append("GMD")
    if ev.get("real_device"):
        styles.append("Real-Device")

    if ev.get("emu_comm"):
        styles.append("Emu_Community")
    else:
        if ev.get("emu_custom"):
            styles.append("Emu_Custom")

    return sorted(set(styles))

def compute_looks_like_instru(ev: Dict) -> str:
    if (
        ev.get("gmd")
        or ev.get("connected")
        or ev.get("baseline")
        or ev.get("adb")
        or ev.get("third_party_invoke")
        or ev.get("flutter_androidish_invoke")
        or ev.get("detox_androidish_invoke")
    ):
        return "yes"
    return "no"

def infer_instru_detect_method(styles: List[str], inv: List[str]) -> str:
    if "Third-Party" in styles:
        return "third_party_cli"
    if "GMD" in styles:
        return "gradle_gmd"
    if "Emu_Community" in styles or "Emu_Custom" in styles:
        if any(x in inv for x in ["Gradle_Connected", "Gradle"]):
            return "gradle_connected"
        if "Detox" in inv:
            return "invocation_signal"
    if "Real-Device" in styles:
        return "real_device_adb"
    if inv:
        return "invocation_signal"
    return "none"

# =========================
# Called-file following via GitHub API (ADJUSTED ONLY TO PERSIST CALLED-FILE INSTRU EVIDENCE)
# =========================
def follow_called_files(
    gh: GitHubClient,
    full_name: str,
    base_ref: str,
    root_text: str,
    origin_ref_to_step_names: Optional[Dict[str, List[str]]] = None,
    max_depth: int = MAX_FOLLOW_DEPTH,
) -> Tuple[Dict, int, int, List[str], bool, List[str], List[str], List[str], List[str]]:
    if not FOLLOW_CALLED_FILES:
        return scan_text_for_evidence(root_text), 0, 0, [], False, [], [], [], []

    agg_evidence = scan_text_for_evidence(root_text)
    unresolved_dynamic = 0
    followed_paths: List[str] = []
    visited: Set[str] = set()

    called_instru_signal = False
    called_instru_file_paths: List[str] = []
    called_instru_origin_refs: List[str] = []
    called_instru_origin_step_names: List[str] = []
    called_instru_file_types: List[str] = []

    origin_ref_to_step_names = origin_ref_to_step_names or {}

    def fetch_and_scan(path: str) -> Optional[Tuple[str, Dict]]:
        sz = file_size_at_ref(gh, full_name, path, base_ref)
        if sz is not None and sz > MAX_FOLLOW_BYTES:
            return None
        txt = fetch_file_text_at_ref(gh, full_name, path, base_ref)
        if not txt:
            return None
        ev = scan_text_for_evidence(txt)
        return txt, ev

    def register_called_instru(path: str, origin_ref: str) -> None:
        nonlocal called_instru_signal, called_instru_file_paths, called_instru_origin_refs, called_instru_origin_step_names, called_instru_file_types
        called_instru_signal = True
        norm_path = normalize_repo_rel_path(path)
        norm_origin = normalize_repo_rel_path(origin_ref)
        called_instru_file_paths.append(norm_path)
        called_instru_origin_refs.append(norm_origin)
        called_instru_file_types.append(classify_called_file_type(norm_path))
        called_instru_origin_step_names.extend(origin_ref_to_step_names.get(norm_origin, []))

    def walk(text: str, depth: int) -> None:
        nonlocal agg_evidence, unresolved_dynamic, followed_paths, visited
        if depth > max_depth:
            return

        refs = extract_references(text)
        wds = extract_workdirs(text)

        for r in refs:
            if not r:
                continue
            if is_dynamic_ref(r):
                unresolved_dynamic += 1
                continue

            norm_origin_ref = normalize_repo_rel_path(r)
            candidates = candidate_paths_for_ref(r, wds)
            is_prob_action = bool(r.strip().startswith("./")) and (r.endswith("/") or "/" in r)

            for c in candidates:
                if c in visited:
                    continue

                if is_prob_action and (not c.lower().endswith((".yml", ".yaml", ".sh", ".ps1", ".py", ".js", ".rb", ".pl", ".bat", ".cmd"))):
                    for ay in possible_action_ymls(c):
                        if ay in visited:
                            continue
                        got = fetch_and_scan(ay)
                        if got:
                            visited.add(ay)
                            followed_paths.append(ay)
                            txt2, ev2 = got
                            agg_evidence = merge_evidence(agg_evidence, ev2)
                            if compute_looks_like_instru(ev2) == "yes":
                                register_called_instru(ay, norm_origin_ref)
                            walk(txt2, depth + 1)

                got = fetch_and_scan(c)
                if got:
                    visited.add(c)
                    followed_paths.append(c)
                    txt2, ev2 = got
                    agg_evidence = merge_evidence(agg_evidence, ev2)
                    if compute_looks_like_instru(ev2) == "yes":
                        register_called_instru(c, norm_origin_ref)
                    walk(txt2, depth + 1)

    walk(root_text, 0)
    return (
        agg_evidence,
        len(unique_preserve(followed_paths)),
        int(unresolved_dynamic),
        unique_preserve(followed_paths),
        bool(called_instru_signal),
        unique_preserve(called_instru_file_paths),
        unique_preserve(called_instru_origin_refs),
        unique_preserve(called_instru_origin_step_names),
        unique_preserve(called_instru_file_types),
    )

# =========================
# Stage-1 processing
# =========================
def build_stage1_rows_for_repo(gh: GitHubClient, full_name: str, repo_url: str) -> List[Dict[str, str]]:
    meta = get_repo_meta(gh, full_name)
    default_branch = meta.get("default_branch") or "main"

    workflows = list_workflows(gh, full_name)
    out_rows: List[Dict[str, str]] = []

    for wf in workflows:
        wf_name = (wf.get("name") or "").strip()
        wf_path = (wf.get("path") or "").strip()
        wf_state = (wf.get("state") or "").strip()
        wf_id = str(wf.get("id") or "")

        if not wf_path:
            continue

        yaml_text = fetch_file_text_at_ref(gh, full_name, wf_path, default_branch)
        if not yaml_text:
            continue

        origin_ref_to_step_names = build_origin_ref_to_step_names(yaml_text)

        (
            ev0,
            followed_count,
            unresolved_dyn,
            followed_paths,
            called_instru_signal,
            called_instru_file_paths,
            called_instru_origin_refs,
            called_instru_origin_step_names,
            called_instru_file_types,
        ) = follow_called_files(
            gh=gh,
            full_name=full_name,
            base_ref=default_branch,
            root_text=yaml_text,
            origin_ref_to_step_names=origin_ref_to_step_names,
            max_depth=MAX_FOLLOW_DEPTH,
        )

        invocation_types = compute_invocation_types(ev0)
        styles = compute_styles(ev0)
        looks_like = compute_looks_like_instru(ev0)

        tp_names = ev0.get("third_party_providers", []) if isinstance(ev0.get("third_party_providers"), list) else []
        tp_name_str = safe_join(tp_names) if ("Third-Party" in styles) else ""

        step_inv_names, anchor_job_ordinal, anchor_step_ordinal_in_job = extract_test_invocation_step_names_and_anchor(
            gh=gh,
            full_name=full_name,
            base_ref=default_branch,
            workflow_yaml_text=yaml_text,
        )

        row = {
            "repo_url": repo_url,
            "full_name": full_name,
            "workflow_id": wf_id,
            "workflow_identifier": wf_name,
            "workflow_path": wf_path,
            "workflow_state": wf_state,
            "styles": ",".join(styles),
            "invocation_types": ",".join(invocation_types),
            "looks_like_instru": looks_like,
            "instru_detect_method": infer_instru_detect_method(styles, invocation_types),
            "third_party_provider_name": tp_name_str,
            "test_invocation_step_names": safe_join(step_inv_names, max_len=1200),
            "anchor_job_ordinal": "" if anchor_job_ordinal is None else str(anchor_job_ordinal),
            "anchor_step_ordinal_in_job": "" if anchor_step_ordinal_in_job is None else str(anchor_step_ordinal_in_job),
            "followed_files_count": str(followed_count),
            "unresolved_dynamic_refs_count": str(unresolved_dyn),
            "followed_paths": safe_join(followed_paths, max_len=1500),

            "called_instru_signal": "True" if called_instru_signal else "False",
            "called_instru_file_paths": safe_join_pipe(called_instru_file_paths, max_len=3000),
            "called_instru_origin_refs": safe_join_pipe(called_instru_origin_refs, max_len=3000),
            "called_instru_origin_step_names": safe_join_pipe(called_instru_origin_step_names, max_len=3000),
            "called_instru_file_types": safe_join_pipe(called_instru_file_types, max_len=1000),

            "stage1_extracted_at_utc": now_utc_iso(),
        }
        out_rows.append(row)

    return out_rows

def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    url_rows, url_fields = read_csv_rows(IN_URL_LIST_CSV)
    if not url_rows:
        raise RuntimeError("URL_List.csv is empty.")

    candidates = ["repo_urls", "repo_url", "url", "repo"]

    def get_url(r: Dict[str, str]) -> str:
        for c in candidates:
            if (r.get(c) or "").strip():
                return (r.get(c) or "").strip()
        if url_fields:
            return (r.get(url_fields[0]) or "").strip()
        return ""

    repo_urls = unique_preserve([get_url(r) for r in url_rows])

    stage1_rows: List[Dict[str, str]] = []

    for u in repo_urls:
        full_name = parse_repo_full_name(u)
        if not full_name:
            continue
        try:
            rows = build_stage1_rows_for_repo(gh, full_name, u)
            stage1_rows.extend(rows)
        except Exception as e:
            stage1_rows.append({
                "repo_url": u,
                "full_name": full_name,
                "workflow_id": "",
                "workflow_identifier": "",
                "workflow_path": "",
                "workflow_state": "",
                "styles": "",
                "invocation_types": "",
                "looks_like_instru": "no",
                "instru_detect_method": "error",
                "third_party_provider_name": "",
                "test_invocation_step_names": "",
                "anchor_job_ordinal": "",
                "anchor_step_ordinal_in_job": "",
                "followed_files_count": "0",
                "unresolved_dynamic_refs_count": "0",
                "followed_paths": "",
                "called_instru_signal": "False",
                "called_instru_file_paths": "",
                "called_instru_origin_refs": "",
                "called_instru_origin_step_names": "",
                "called_instru_file_types": "",
                "stage1_extracted_at_utc": now_utc_iso(),
            })
            print(f"[warn] {full_name}: {e}")

    out_fields = [
        "repo_url",
        "full_name",
        "workflow_id",
        "workflow_identifier",
        "workflow_path",
        "workflow_state",
        "styles",
        "invocation_types",
        "looks_like_instru",
        "instru_detect_method",
        "third_party_provider_name",
        "test_invocation_step_names",
        "anchor_job_ordinal",
        "anchor_step_ordinal_in_job",
        "followed_files_count",
        "unresolved_dynamic_refs_count",
        "followed_paths",
        "called_instru_signal",
        "called_instru_file_paths",
        "called_instru_origin_refs",
        "called_instru_origin_step_names",
        "called_instru_file_types",
        "stage1_extracted_at_utc",
    ]

    write_csv(OUT_STAGE1_CSV, out_fields, stage1_rows)
    print("[done] Stage 1:", OUT_STAGE1_CSV, f"(rows={len(stage1_rows)})")

if __name__ == "__main__":
    main()

[done] Stage 1: C:\Android Mobile App\ICST2026_Ext\verified_workflows_v16.csv (rows=1635)


## Stage 2 — Extract run-level metrics and attach style labels

In [2]:
# ============================================================
# Stage 2 (ADJUSTED): Run inventory + anchored fallback metrics (S2_)
#
# What changed in this version
# - Uses workflow_id (NOT workflow_identifier) to list runs (more reliable)
# - Removes Stage-2 fields that are not present in current Stage-1 output
#   (gmd_capable, gmd_reasons, evidence_labels, label_source, workflow_name)
# - Adds anchored runtime-step matching using Stage-1:
#     test_invocation_step_names
#   This improves fallback timing (especially TTFTS)
# - Keeps regex fallback (step/job) for coverage
# - Stage-2 heuristics updated to better align with Stage-1 labels
#   (includes Detox / Flutter wording in runtime step-name heuristics)
#
# NEW (for modified TTFTS readiness)
# - Pass-through Stage-1 extra anchor-position field if present:
#     jobs_before_anchor_count
# - Adds anchor-job timing outputs for job-based TTFTS:
#     anchor_job_name
#     anchor_job_started_at
#     anchor_job_start_source
#     time_to_first_instru_from_anchor_job_seconds
#     time_to_first_instru_from_anchor_job_quality
# - Adds S2_ mirrors for the above so Stage 3 can consume them as fallback
#
# NEW (ONLY): carries Stage-1 called-file instrumentation evidence fields
#   so Stage 3 can use them directly without re-following every file:
#     called_instru_signal
#     called_instru_file_paths
#     called_instru_origin_refs
#     called_instru_origin_step_names
#     called_instru_file_types
#
# Output:
# - Keeps legacy metrics columns
# - Adds S2_ mirror/fallback fields for Stage 3 consumption
# ============================================================

import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Union, Tuple

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_VERIFIED_WORKFLOWS_CSV = ROOT_DIR / "verified_workflows_v16.csv"
OUT_RUN_INVENTORY_CSV = ROOT_DIR / "run_inventory.csv"

DEFAULT_BRANCH_ONLY = True
PROCESS_ONLY_LOOKS_LIKE_INSTRU = True
FETCH_JOBS_FOR_EACH_RUN = True

MAX_RUNS_PER_WORKFLOW: Optional[int] = None
RUN_CREATED_AT_AFTER: Optional[str] = None

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

MAX_TOKENS_TO_USE = 7
SLEEP_BETWEEN_WORKFLOWS_SEC = 0.05


# =========================
# Helpers
# =========================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(str(iso).replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = (row.get(key_field) or "").strip()
            if k:
                keys.add(k)
    return keys

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break

    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x is None:
            continue
        s = str(x)
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out

def safe_join_names(names: List[str], max_len: int = 700) -> str:
    s = ",".join(unique_preserve([n.strip() for n in names if n and str(n).strip()]))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."


# =========================
# Name normalization / matching
# =========================
_NORM_WS_RE = re.compile(r"\s+")
_NORM_PUNCT_RE = re.compile(r"[\[\]\(\)\{\}:;|]+")

def normalize_name(s: str) -> str:
    x = (s or "").strip().strip('"').strip("'").lower()
    x = _NORM_PUNCT_RE.sub(" ", x)
    x = _NORM_WS_RE.sub(" ", x).strip()
    return x

def parse_anchor_step_names(csv_value: str) -> List[str]:
    if not csv_value:
        return []
    raw = [p.strip() for p in str(csv_value).split(",")]
    out = [r for r in raw if r]
    return unique_preserve(out)

def anchored_step_match(runtime_step_name: str, anchor_names: List[str]) -> bool:
    """
    Robust but conservative:
    - exact normalized match
    - contains match either direction for matrix/appended labels
    """
    rn = normalize_name(runtime_step_name)
    if not rn:
        return False
    for a in anchor_names:
        an = normalize_name(a)
        if not an:
            continue
        if rn == an:
            return True
        if an in rn or rn in an:
            return True
    return False


# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-inventory-stage2-v16/1.5",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# =========================
# GitHub endpoints
# =========================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    data = gh.request_json("GET", f"https://api.github.com/repos/{full_name}")
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_workflow_runs(gh: GitHubClient, full_name: str, workflow_id_or_file: str, branch: Optional[str]) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_id_or_file}/runs"
    params = {"branch": branch} if branch else {}
    return list(gh.paginate(url, params=params, item_key="workflow_runs"))

def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))


# =========================
# Run timing from jobs window
# =========================
def compute_run_window_from_jobs(jobs: List[Dict]) -> Tuple[str, str, Optional[int]]:
    if not jobs:
        return "", "", None
    starts: List[datetime] = []
    ends: List[datetime] = []
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt:
            starts.append(sdt)
        if edt:
            ends.append(edt)
    if not starts or not ends:
        return "", "", None
    smin = min(starts)
    emax = max(ends)
    return (
        smin.isoformat().replace("+00:00", "Z"),
        emax.isoformat().replace("+00:00", "Z"),
        dt_to_seconds(smin, emax),
    )


# =========================
# Stage-2 runtime heuristics (aligned better with Stage 1)
# =========================
INSTRU_STEP_NAME_RE = re.compile(
    r"("
    r"instrument|"
    r"connected.*androidtest|androidtest|manageddevice|gmd|"
    r"baseline.?profile|"
    r"adb|"
    r"emulator runner|android emulator|avd|"
    r"firebase test|test lab|device farm|"
    r"uiautomator|espresso|"
    r"detox|"
    r"flutter.*(integration|drive)|integration test"
    r")",
    re.IGNORECASE,
)
INSTRU_JOB_NAME_RE = re.compile(
    r"("
    r"instrument|androidtest|connected|manageddevice|gmd|"
    r"baseline|"
    r"emulator|avd|"
    r"firebase|test lab|device farm|"
    r"detox|flutter.*integration"
    r")",
    re.IGNORECASE,
)


# =========================
# Instrumentation metrics from jobs/steps
# =========================
def infer_instru_metrics_from_jobs(
    jobs: List[Dict],
    run_created_at: str,
    run_started_at: str,
    anchor_step_names: Optional[List[str]] = None,
) -> Dict[str, Union[str, int, float, None]]:
    """
    Priority:
      1) Stage-1 anchored runtime step-name match (step_name_anchor)
      2) Regex step-name match (step_regex)
      3) Regex job-name match (job_regex)

    NEW:
      - Captures anchor-job timing for modified TTFTS:
          anchor_job_name
          anchor_job_started_at
          anchor_job_start_source
          time_to_first_instru_from_anchor_job_seconds
          time_to_first_instru_from_anchor_job_quality
    """
    anchor_step_names = anchor_step_names or []

    out = {
        "instru_conclusion": "unknown",
        "instru_detect_method": "none",
        "instru_duration_seconds": None,
        "run_duration_seconds": None,
        "runner_labels_union": "",
        "queue_seconds": None,
        "time_to_first_instru_seconds": None,
        "instru_job_count": 0,
        "instru_step_count": 0,
        "instru_job_names": "",
        "instru_step_names": "",
        "instru_total_seconds": None,
        "instru_window_seconds": None,
        "instru_first_started_at": "",
        "instru_last_completed_at": "",
        "instru_share_of_run": None,

        # NEW anchor-job fields
        "anchor_job_name": "",
        "anchor_job_started_at": "",
        "anchor_job_start_source": "missing",
        "time_to_first_instru_from_anchor_job_seconds": None,
        "time_to_first_instru_from_anchor_job_quality": "missing",
    }

    out["queue_seconds"] = dt_to_seconds(iso_to_dt(run_created_at), iso_to_dt(run_started_at))
    if not jobs:
        return out

    starts, ends = [], []
    labels_union: Set[str] = set()
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt:
            starts.append(sdt)
        if edt:
            ends.append(edt)
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and lab.strip():
                labels_union.add(lab.strip())

    run_dur = dt_to_seconds(min(starts) if starts else None, max(ends) if ends else None)
    out["run_duration_seconds"] = run_dur
    out["runner_labels_union"] = ",".join(sorted(labels_union))

    candidates: List[Dict[str, Union[int, str, datetime, None]]] = []
    instru_job_names: List[str] = []
    instru_step_names: List[str] = []

    def add_candidate(
        priority: int,
        method: str,
        conclusion: str,
        dur: Optional[int],
        sdt: Optional[datetime],
        edt: Optional[datetime],
        job_name: str,
        step_name: str,
        job_sdt: Optional[datetime],
        job_earliest_step_sdt: Optional[datetime],
    ) -> None:
        candidates.append({
            "priority": priority,
            "method": method,
            "conclusion": conclusion or "unknown",
            "dur": dur,
            "sdt": sdt,
            "edt": edt,
            "job_name": job_name,
            "step_name": step_name,
            "job_sdt": job_sdt,
            "job_earliest_step_sdt": job_earliest_step_sdt,
        })

    for j in jobs:
        job_name = (j.get("name") or "").strip()
        job_is_instru_regex = bool(INSTRU_JOB_NAME_RE.search(job_name))
        job_sdt = iso_to_dt(j.get("started_at"))
        job_edt = iso_to_dt(j.get("completed_at"))
        job_dur = dt_to_seconds(job_sdt, job_edt)

        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        # earliest actual step start in this job (fallback if job.started_at missing)
        job_earliest_step_sdt: Optional[datetime] = None
        for stx in steps:
            s0 = iso_to_dt(stx.get("started_at"))
            if s0 and (job_earliest_step_sdt is None or s0 < job_earliest_step_sdt):
                job_earliest_step_sdt = s0

        any_step_candidate_here = False
        for st in steps:
            step_name = (st.get("name") or "").strip()
            if not step_name:
                continue

            is_anchor = anchored_step_match(step_name, anchor_step_names) if anchor_step_names else False
            is_regex = bool(INSTRU_STEP_NAME_RE.search(step_name))

            if not (is_anchor or is_regex):
                continue

            any_step_candidate_here = True
            if job_name:
                instru_job_names.append(job_name)
            instru_step_names.append(step_name)

            st_sdt = iso_to_dt(st.get("started_at")) or job_sdt or job_earliest_step_sdt
            st_edt = iso_to_dt(st.get("completed_at")) or job_edt
            st_dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
            if st_dur is None:
                st_dur = job_dur

            if is_anchor:
                add_candidate(
                    priority=0,
                    method="step_name_anchor",
                    conclusion=(st.get("conclusion") or st.get("status") or "unknown"),
                    dur=st_dur,
                    sdt=st_sdt,
                    edt=st_edt,
                    job_name=job_name,
                    step_name=step_name,
                    job_sdt=job_sdt,
                    job_earliest_step_sdt=job_earliest_step_sdt,
                )
            else:
                add_candidate(
                    priority=1,
                    method="step_regex",
                    conclusion=(st.get("conclusion") or st.get("status") or "unknown"),
                    dur=st_dur,
                    sdt=st_sdt,
                    edt=st_edt,
                    job_name=job_name,
                    step_name=step_name,
                    job_sdt=job_sdt,
                    job_earliest_step_sdt=job_earliest_step_sdt,
                )

        if job_is_instru_regex and not any_step_candidate_here:
            if job_name:
                instru_job_names.append(job_name)
            add_candidate(
                priority=2,
                method="job_regex",
                conclusion=(j.get("conclusion") or j.get("status") or "unknown"),
                dur=job_dur,
                sdt=job_sdt or job_earliest_step_sdt,
                edt=job_edt,
                job_name=job_name,
                step_name="",
                job_sdt=job_sdt,
                job_earliest_step_sdt=job_earliest_step_sdt,
            )

    instru_first_start: Optional[datetime] = None
    instru_last_end: Optional[datetime] = None
    total_seconds = 0
    total_seconds_any = False

    if candidates:
        def _cand_sort_key(c: Dict[str, Union[int, str, datetime, None]]) -> Tuple[int, str]:
            p = int(c.get("priority") or 99)
            sdt = c.get("sdt")
            s = sdt.isoformat() if isinstance(sdt, datetime) else "9999-12-31T23:59:59+00:00"
            return (p, s)

        candidates_sorted = sorted(candidates, key=_cand_sort_key)

        first = candidates_sorted[0]
        out["instru_detect_method"] = str(first.get("method") or "none")
        out["instru_conclusion"] = str(first.get("conclusion") or "unknown")
        out["instru_duration_seconds"] = first.get("dur") if isinstance(first.get("dur"), int) else None

        # NEW: anchor-job timing for modified TTFTS
        anchor_job_name = str(first.get("job_name") or "")
        anchor_step_start = first.get("sdt") if isinstance(first.get("sdt"), datetime) else None
        anchor_job_sdt = first.get("job_sdt") if isinstance(first.get("job_sdt"), datetime) else None
        anchor_job_earliest_step_sdt = first.get("job_earliest_step_sdt") if isinstance(first.get("job_earliest_step_sdt"), datetime) else None

        job_start_basis: Optional[datetime] = None
        job_start_source = "missing"
        quality = "missing"

        if anchor_job_sdt:
            job_start_basis = anchor_job_sdt
            job_start_source = "job_started_at"
            quality = "exact"
        elif anchor_job_earliest_step_sdt:
            job_start_basis = anchor_job_earliest_step_sdt
            job_start_source = "earliest_step_in_job"
            quality = "inferred"

        if anchor_job_name:
            out["anchor_job_name"] = anchor_job_name
        if job_start_basis:
            out["anchor_job_started_at"] = job_start_basis.isoformat().replace("+00:00", "Z")
            out["anchor_job_start_source"] = job_start_source

        if job_start_basis and anchor_step_start:
            out["time_to_first_instru_from_anchor_job_seconds"] = dt_to_seconds(job_start_basis, anchor_step_start)
            out["time_to_first_instru_from_anchor_job_quality"] = quality
        elif anchor_step_start:
            # last-resort if we know the step start but not the job start
            base_run = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)
            if base_run:
                out["time_to_first_instru_from_anchor_job_seconds"] = dt_to_seconds(base_run, anchor_step_start)
                out["time_to_first_instru_from_anchor_job_quality"] = "run_started_fallback"
                out["anchor_job_start_source"] = "run_started_fallback"

        for c in candidates_sorted:
            sdt = c.get("sdt") if isinstance(c.get("sdt"), datetime) else None
            edt = c.get("edt") if isinstance(c.get("edt"), datetime) else None
            dur = c.get("dur") if isinstance(c.get("dur"), int) else None

            if sdt and (instru_first_start is None or sdt < instru_first_start):
                instru_first_start = sdt
            if edt and (instru_last_end is None or edt > instru_last_end):
                instru_last_end = edt

            if dur is not None:
                total_seconds += dur
                total_seconds_any = True

    out["instru_job_names"] = safe_join_names(instru_job_names)
    out["instru_step_names"] = safe_join_names(instru_step_names)
    out["instru_job_count"] = len(unique_preserve(instru_job_names))
    out["instru_step_count"] = len(unique_preserve(instru_step_names))

    if total_seconds_any:
        out["instru_total_seconds"] = total_seconds

    if instru_first_start:
        out["instru_first_started_at"] = instru_first_start.isoformat().replace("+00:00", "Z")
        base_start = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)
        out["time_to_first_instru_seconds"] = dt_to_seconds(base_start, instru_first_start)

    if instru_last_end:
        out["instru_last_completed_at"] = instru_last_end.isoformat().replace("+00:00", "Z")

    out["instru_window_seconds"] = dt_to_seconds(instru_first_start, instru_last_end)

    if out["instru_window_seconds"] is not None and run_dur:
        try:
            out["instru_share_of_run"] = round(float(out["instru_window_seconds"]) / float(run_dur), 6)
        except Exception:
            out["instru_share_of_run"] = None

    return out


# =========================
# Read verified workflows
# =========================
def load_verified_workflows(path: Path) -> List[Dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(f"Verified workflows CSV not found: {path}")
    rows: List[Dict[str, str]] = []
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for r in rdr:
            rows.append({(k or ""): (v or "") for k, v in r.items()})
    return rows


# =========================
# MAIN
# =========================
def main() -> None:
    if not IN_VERIFIED_WORKFLOWS_CSV.exists():
        raise FileNotFoundError(f"Missing input: {IN_VERIFIED_WORKFLOWS_CSV}")

    if OUT_RUN_INVENTORY_CSV.exists():
        OUT_RUN_INVENTORY_CSV.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    rows = load_verified_workflows(IN_VERIFIED_WORKFLOWS_CSV)
    if PROCESS_ONLY_LOOKS_LIKE_INSTRU:
        rows = [r for r in rows if (r.get("looks_like_instru", "").strip().lower() == "yes")]

    if not rows:
        raise RuntimeError("No workflows found to process (check verified CSV or filter).")

    after_dt = iso_to_dt(RUN_CREATED_AT_AFTER) if RUN_CREATED_AT_AFTER else None

    out_fields = [
        # workflow identity (consistent with current Stage 1)
        "full_name",
        "default_branch",
        "workflow_identifier",
        "workflow_id",
        "workflow_path",

        # workflow-level labels from Stage 1 (current schema)
        "looks_like_instru",
        "styles",
        "invocation_types",
        "third_party_provider_name",
        "test_invocation_step_names",

        # NEW: pass-through Stage-1 noise indicator if present
        "jobs_before_anchor_count",

        # NEW: pass-through called-file instrumentation evidence from Stage 1
        "called_instru_signal",
        "called_instru_file_paths",
        "called_instru_origin_refs",
        "called_instru_origin_step_names",
        "called_instru_file_types",

        # run
        "run_id",
        "run_number",
        "run_attempt",
        "head_sha",
        "created_at",
        "run_started_at",
        "status",
        "run_conclusion",
        "event",
        "head_branch",
        "html_url",
        "extracted_at_utc",

        # legacy metrics (kept)
        "queue_seconds",
        "time_to_first_instru_seconds",
        "instru_conclusion",
        "instru_detect_method",
        "instru_duration_seconds",
        "run_duration_seconds",
        "runner_labels_union",
        "instru_job_count",
        "instru_step_count",
        "instru_job_names",
        "instru_step_names",
        "instru_total_seconds",
        "instru_window_seconds",
        "instru_first_started_at",
        "instru_last_completed_at",
        "instru_share_of_run",

        # NEW: anchor-job timing (modified TTFTS readiness)
        "anchor_job_name",
        "anchor_job_started_at",
        "anchor_job_start_source",
        "time_to_first_instru_from_anchor_job_seconds",
        "time_to_first_instru_from_anchor_job_quality",

        # S2 run timing fallback
        "S2_run_started_at_jobs_min",
        "S2_run_ended_at_jobs_max",
        "S2_run_duration_seconds_jobs_window",
        "S2_run_timing_source",

        # S2 mirrored fallback metrics
        "S2_queue_seconds",
        "S2_time_to_first_instru_seconds",
        "S2_instru_conclusion",
        "S2_instru_detect_method",
        "S2_instru_duration_seconds",
        "S2_run_duration_seconds",
        "S2_runner_labels_union",
        "S2_instru_job_count",
        "S2_instru_step_count",
        "S2_instru_job_names",
        "S2_instru_step_names",
        "S2_instru_total_seconds",
        "S2_instru_window_seconds",
        "S2_instru_first_started_at",
        "S2_instru_last_completed_at",
        "S2_instru_share_of_run",

        # NEW S2 mirrors for modified TTFTS
        "S2_anchor_job_name",
        "S2_anchor_job_started_at",
        "S2_anchor_job_start_source",
        "S2_time_to_first_instru_from_anchor_job_seconds",
        "S2_time_to_first_instru_from_anchor_job_quality",
    ]

    ensure_csv_header(OUT_RUN_INVENTORY_CSV, out_fields)
    existing_run_ids = load_existing_keys(OUT_RUN_INVENTORY_CSV, "run_id")

    default_branch_cache: Dict[str, str] = {}

    wf_iter = rows
    if tqdm is not None:
        wf_iter = tqdm(rows, desc="Stage2: workflows -> runs")

    for wf in wf_iter:
        full_name = (wf.get("full_name") or "").strip()
        workflow_identifier = (wf.get("workflow_identifier") or "").strip()
        workflow_id = (wf.get("workflow_id") or "").strip()
        workflow_path = (wf.get("workflow_path") or "").strip()

        if not full_name:
            continue

        workflow_key_for_runs = workflow_id or workflow_identifier or workflow_path
        if not workflow_key_for_runs:
            continue

        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        branch = default_branch if DEFAULT_BRANCH_ONLY else None

        runs = list_workflow_runs(gh, full_name, workflow_key_for_runs, branch=branch) or []
        if MAX_RUNS_PER_WORKFLOW is not None:
            runs = runs[:MAX_RUNS_PER_WORKFLOW]

        anchor_step_names = parse_anchor_step_names(wf.get("test_invocation_step_names") or "")

        for run in runs:
            run_id = str(run.get("id") or "").strip()
            if not run_id or run_id in existing_run_ids:
                continue

            created_at = run.get("created_at") or ""
            if after_dt:
                cdt = iso_to_dt(created_at)
                if cdt and cdt < after_dt:
                    continue

            head_branch = run.get("head_branch") or ""
            if DEFAULT_BRANCH_ONLY and head_branch and head_branch != default_branch:
                continue

            run_started_at = run.get("run_started_at") or ""
            head_sha = run.get("head_sha") or ""

            jobs = list_run_jobs(gh, full_name, int(run_id)) if FETCH_JOBS_FOR_EACH_RUN else []

            metrics = infer_instru_metrics_from_jobs(
                jobs=jobs,
                run_created_at=created_at,
                run_started_at=run_started_at,
                anchor_step_names=anchor_step_names,
            )

            # Jobs-window run timing (S2_)
            s2_run_start, s2_run_end, s2_run_dur = compute_run_window_from_jobs(jobs)
            s2_run_src = "jobs_window" if s2_run_start and s2_run_end else "missing"

            # Broad workflow-label fallback only when no runtime signal exists
            if (
                int(metrics.get("instru_job_count") or 0) == 0
                and int(metrics.get("instru_step_count") or 0) == 0
                and (wf.get("looks_like_instru") or "").strip().lower() == "yes"
            ):
                metrics["instru_detect_method"] = "workflow_label"
                metrics["instru_conclusion"] = (run.get("conclusion") or "unknown")
                if run_started_at:
                    metrics["instru_first_started_at"] = run_started_at
                    metrics["time_to_first_instru_seconds"] = 0

                # Modified TTFTS fallback consistency (explicitly flagged)
                metrics["anchor_job_start_source"] = "run_started_fallback"
                metrics["time_to_first_instru_from_anchor_job_seconds"] = 0
                metrics["time_to_first_instru_from_anchor_job_quality"] = "workflow_label_proxy"
                metrics["anchor_job_started_at"] = run_started_at

            # Build S2_ mirror for Stage 3 fallback
            s2 = {
                "S2_queue_seconds": metrics["queue_seconds"],
                "S2_time_to_first_instru_seconds": metrics["time_to_first_instru_seconds"],
                "S2_instru_conclusion": metrics["instru_conclusion"],
                "S2_instru_detect_method": metrics["instru_detect_method"],
                "S2_instru_duration_seconds": metrics["instru_duration_seconds"],
                "S2_run_duration_seconds": metrics["run_duration_seconds"],
                "S2_runner_labels_union": metrics["runner_labels_union"],
                "S2_instru_job_count": metrics["instru_job_count"],
                "S2_instru_step_count": metrics["instru_step_count"],
                "S2_instru_job_names": metrics["instru_job_names"],
                "S2_instru_step_names": metrics["instru_step_names"],
                "S2_instru_total_seconds": metrics["instru_total_seconds"],
                "S2_instru_window_seconds": metrics["instru_window_seconds"],
                "S2_instru_first_started_at": metrics["instru_first_started_at"],
                "S2_instru_last_completed_at": metrics["instru_last_completed_at"],
                "S2_instru_share_of_run": metrics["instru_share_of_run"],

                # NEW S2 mirrors
                "S2_anchor_job_name": metrics["anchor_job_name"],
                "S2_anchor_job_started_at": metrics["anchor_job_started_at"],
                "S2_anchor_job_start_source": metrics["anchor_job_start_source"],
                "S2_time_to_first_instru_from_anchor_job_seconds": metrics["time_to_first_instru_from_anchor_job_seconds"],
                "S2_time_to_first_instru_from_anchor_job_quality": metrics["time_to_first_instru_from_anchor_job_quality"],
            }

            append_row(OUT_RUN_INVENTORY_CSV, out_fields, {
                # workflow identity
                "full_name": full_name,
                "default_branch": default_branch,
                "workflow_identifier": workflow_identifier,
                "workflow_id": workflow_id,
                "workflow_path": workflow_path,

                # workflow labels from Stage 1
                "looks_like_instru": (wf.get("looks_like_instru") or ""),
                "styles": (wf.get("styles") or ""),
                "invocation_types": (wf.get("invocation_types") or ""),
                "third_party_provider_name": (wf.get("third_party_provider_name") or ""),
                "test_invocation_step_names": (wf.get("test_invocation_step_names") or ""),

                # NEW pass-through field from Stage 1 (if absent, stays blank)
                "jobs_before_anchor_count": (wf.get("jobs_before_anchor_count") or ""),

                # NEW pass-through called-file instrumentation evidence from Stage 1
                "called_instru_signal": (wf.get("called_instru_signal") or ""),
                "called_instru_file_paths": (wf.get("called_instru_file_paths") or ""),
                "called_instru_origin_refs": (wf.get("called_instru_origin_refs") or ""),
                "called_instru_origin_step_names": (wf.get("called_instru_origin_step_names") or ""),
                "called_instru_file_types": (wf.get("called_instru_file_types") or ""),

                # run metadata
                "run_id": run_id,
                "run_number": run.get("run_number") or "",
                "run_attempt": run.get("run_attempt") or "",
                "head_sha": head_sha,
                "created_at": created_at,
                "run_started_at": run_started_at,
                "status": run.get("status") or "",
                "run_conclusion": run.get("conclusion") or "",
                "event": run.get("event") or "",
                "head_branch": head_branch,
                "html_url": run.get("html_url") or "",
                "extracted_at_utc": now_utc_iso(),

                # legacy metrics
                "queue_seconds": "" if metrics["queue_seconds"] is None else str(metrics["queue_seconds"]),
                "time_to_first_instru_seconds": "" if metrics["time_to_first_instru_seconds"] is None else str(metrics["time_to_first_instru_seconds"]),
                "instru_conclusion": metrics["instru_conclusion"],
                "instru_detect_method": metrics["instru_detect_method"],
                "instru_duration_seconds": "" if metrics["instru_duration_seconds"] is None else str(metrics["instru_duration_seconds"]),
                "run_duration_seconds": "" if metrics["run_duration_seconds"] is None else str(metrics["run_duration_seconds"]),
                "runner_labels_union": metrics["runner_labels_union"],
                "instru_job_count": str(metrics["instru_job_count"]),
                "instru_step_count": str(metrics["instru_step_count"]),
                "instru_job_names": metrics["instru_job_names"],
                "instru_step_names": metrics["instru_step_names"],
                "instru_total_seconds": "" if metrics["instru_total_seconds"] is None else str(metrics["instru_total_seconds"]),
                "instru_window_seconds": "" if metrics["instru_window_seconds"] is None else str(metrics["instru_window_seconds"]),
                "instru_first_started_at": metrics["instru_first_started_at"],
                "instru_last_completed_at": metrics["instru_last_completed_at"],
                "instru_share_of_run": "" if metrics["instru_share_of_run"] is None else str(metrics["instru_share_of_run"]),

                # NEW anchor-job timing metrics
                "anchor_job_name": metrics["anchor_job_name"],
                "anchor_job_started_at": metrics["anchor_job_started_at"],
                "anchor_job_start_source": metrics["anchor_job_start_source"],
                "time_to_first_instru_from_anchor_job_seconds": "" if metrics["time_to_first_instru_from_anchor_job_seconds"] is None else str(metrics["time_to_first_instru_from_anchor_job_seconds"]),
                "time_to_first_instru_from_anchor_job_quality": metrics["time_to_first_instru_from_anchor_job_quality"],

                # S2 timing fallback
                "S2_run_started_at_jobs_min": s2_run_start,
                "S2_run_ended_at_jobs_max": s2_run_end,
                "S2_run_duration_seconds_jobs_window": "" if s2_run_dur is None else str(s2_run_dur),
                "S2_run_timing_source": s2_run_src,

                # S2 mirrored metrics
                "S2_queue_seconds": "" if s2["S2_queue_seconds"] is None else str(s2["S2_queue_seconds"]),
                "S2_time_to_first_instru_seconds": "" if s2["S2_time_to_first_instru_seconds"] is None else str(s2["S2_time_to_first_instru_seconds"]),
                "S2_instru_conclusion": s2["S2_instru_conclusion"],
                "S2_instru_detect_method": s2["S2_instru_detect_method"],
                "S2_instru_duration_seconds": "" if s2["S2_instru_duration_seconds"] is None else str(s2["S2_instru_duration_seconds"]),
                "S2_run_duration_seconds": "" if s2["S2_run_duration_seconds"] is None else str(s2["S2_run_duration_seconds"]),
                "S2_runner_labels_union": s2["S2_runner_labels_union"],
                "S2_instru_job_count": "" if s2["S2_instru_job_count"] is None else str(s2["S2_instru_job_count"]),
                "S2_instru_step_count": "" if s2["S2_instru_step_count"] is None else str(s2["S2_instru_step_count"]),
                "S2_instru_job_names": s2["S2_instru_job_names"],
                "S2_instru_step_names": s2["S2_instru_step_names"],
                "S2_instru_total_seconds": "" if s2["S2_instru_total_seconds"] is None else str(s2["S2_instru_total_seconds"]),
                "S2_instru_window_seconds": "" if s2["S2_instru_window_seconds"] is None else str(s2["S2_instru_window_seconds"]),
                "S2_instru_first_started_at": s2["S2_instru_first_started_at"],
                "S2_instru_last_completed_at": s2["S2_instru_last_completed_at"],
                "S2_instru_share_of_run": "" if s2["S2_instru_share_of_run"] is None else str(s2["S2_instru_share_of_run"]),

                # NEW S2 mirrors for modified TTFTS
                "S2_anchor_job_name": s2["S2_anchor_job_name"],
                "S2_anchor_job_started_at": s2["S2_anchor_job_started_at"],
                "S2_anchor_job_start_source": s2["S2_anchor_job_start_source"],
                "S2_time_to_first_instru_from_anchor_job_seconds": "" if s2["S2_time_to_first_instru_from_anchor_job_seconds"] is None else str(s2["S2_time_to_first_instru_from_anchor_job_seconds"]),
                "S2_time_to_first_instru_from_anchor_job_quality": s2["S2_time_to_first_instru_from_anchor_job_quality"],
            })

            existing_run_ids.add(run_id)

        time.sleep(SLEEP_BETWEEN_WORKFLOWS_SEC)

    print("Done.")
    print("Wrote:", OUT_RUN_INVENTORY_CSV)


if __name__ == "__main__":
    main()

Stage2: workflows -> runs: 100%|██████████| 311/311 [3:56:28<00:00, 45.62s/it]    

Done.
Wrote: C:\Android Mobile App\ICST2026_Ext\run_inventory.csv


## Stage 3 — Extract step telemetry, derive TTFTS, and enhance run metrics

In [3]:
# ============================================================
# Stage 3 (FULLY ADJUSTED: canonical style names + stronger 3P alignment
#            + Stage1-confirmed called-file recovery only)
#
# Main adjustments in this version
# 1) Preserves existing output file names / locations
# 2) Preserves TTFTS logic and provenance behavior
# 3) Preserves run-level output and run×style output
# 4) Keeps:
#      - instru_duration_seconds = FULL instrumentation-path window
#      - core_instru_window_seconds / instru_exec_window_seconds = CORE execution span only
# 5) Uses canonical style names everywhere:
#      - Community
#      - Custom
#      - GMD
#      - Third-Party
#      - Real-Devices
# 6) Fixes multi-style segmentation by normalizing inferred style names
#    before comparing against declared styles
# 7) Strengthens Third-Party detection to align better with Stage 1
# 8) Keeps style-aware instrumentation end selection
# 9) IMPORTANT CHANGE:
#      - Stage 3 no longer blindly re-follows local files for every candidate step
#      - It now uses Stage 1 / Stage 2 pass-through fields:
#           called_instru_signal
#           called_instru_file_paths
#           called_instru_origin_refs
#           called_instru_origin_step_names
#           called_instru_file_types
#      - Called-file recovery is attempted ONLY when Stage 1 already confirmed
#        instrumentation evidence in followed files
# 10) Keeps Option B only for Custom-capable workflows:
#      - guarded Stage-1-supported fallback to rescue Custom wrapper anchor/execution detection
#
# 11) ONLY NEW CHANGE IN THIS REVISION:
#      - Custom no longer uses the same-job constrained instrumentation-end rule
#      - Custom now uses the broader continuation logic like Third-Party
#      - Community / GMD / Real-Devices remain same-job constrained
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_STAGE2_CSV = ROOT_DIR / "run_inventory.csv"

OUT_STAGE3A_RUNS_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
OUT_STAGE3B_STEPS_CSV = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"
OUT_STAGE3C_RUN_PER_STYLE_CSV = ROOT_DIR / "run_per_style_v1_stage3.csv"

MAX_TOKENS_TO_USE = 7
PROCESS_ONLY_RELEVANT_ROWS = True

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 7000

# targeted Stage1-confirmed followed-file fetch only
MAX_FOLLOW_BYTES_STAGE3 = 1_500_000

# =========================
# Helpers
# =========================
BOM = "\ufeff"
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join_names(names: List[str], max_len: int = 800) -> str:
    s = ",".join(unique_preserve([n for n in names if n]))
    return s[:max_len]

def safe_int_from_str(x: Optional[str]) -> Optional[int]:
    try:
        if x is None or str(x).strip() == "":
            return None
        return int(float(str(x).strip()))
    except Exception:
        return None

def norm(s: Optional[str]) -> str:
    return (s or "").strip()

def low(s: Optional[str]) -> str:
    return norm(s).lower()

def canon_key(s: Optional[str]) -> str:
    s = low(s)
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def bool_from_any(v) -> bool:
    if isinstance(v, bool):
        return v
    s = low(str(v))
    return s in {"1", "true", "yes", "y"}

def read_env_tokens(path: Path) -> List[str]:
    toks: List[str] = []
    if not path.exists():
        return toks
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        if k.strip().startswith("GITHUB_TOKEN"):
            tok = v.strip().strip('"').strip("'")
            if tok:
                toks.append(tok)
    return toks[:MAX_TOKENS_TO_USE]

def read_csv_rows(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        rdr = csv.DictReader(f)
        out = []
        for r in rdr:
            clean = {}
            for k, v in r.items():
                kk = (k or "").replace(BOM, "").strip()
                clean[kk] = v
            out.append(clean)
        return out

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in fieldnames})

# =========================
# Canonical style normalization
# =========================
STYLE_CANONICAL = ["Community", "Custom", "GMD", "Third-Party", "Real-Devices"]

STYLE_ALIASES = {
    "community": "Community",
    "custom": "Custom",
    "gmd": "GMD",
    "third party": "Third-Party",
    "third-party": "Third-Party",
    "third_party": "Third-Party",
    "thirdparty": "Third-Party",
    "3p": "Third-Party",
    "real devices": "Real-Devices",
    "real-devices": "Real-Devices",
    "real_devices": "Real-Devices",
    "realdevices": "Real-Devices",
}

def normalize_style_label(s: Optional[str]) -> str:
    key = canon_key(s)
    return STYLE_ALIASES.get(key, norm(s))

def split_styles(s: Optional[str]) -> List[str]:
    raw = norm(s)
    if not raw:
        return []
    parts = [normalize_style_label(x) for x in re.split(r"[|,;/]+", raw) if norm(x)]
    return unique_preserve([p for p in parts if p in STYLE_CANONICAL])

# =========================
# GitHub client
# =========================
@dataclass
class GitHubClient:
    tokens: List[str]
    idx: int = 0

    def __post_init__(self):
        if not self.tokens:
            raise RuntimeError("No GitHub tokens found.")

    def _headers(self) -> Dict[str, str]:
        return {
            "Authorization": f"token {self.tokens[self.idx]}",
            "Accept": "application/vnd.github+json",
            "User-Agent": "ICST2026-Stage3-StepTelemetry",
        }

    def _rotate(self):
        self.idx = (self.idx + 1) % len(self.tokens)

    def get(self, url: str, stream: bool = False) -> requests.Response:
        last_exc = None
        for attempt in range(MAX_RETRIES_PER_REQUEST):
            try:
                r = requests.get(
                    url,
                    headers=self._headers(),
                    timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                    stream=stream,
                )
                if r.status_code in (403, 429):
                    self._rotate()
                    time.sleep(min(BACKOFF_CAP_S, BACKOFF_BASE_S ** attempt) + random.random())
                    continue
                if r.status_code >= 500:
                    time.sleep(min(BACKOFF_CAP_S, BACKOFF_BASE_S ** attempt) + random.random())
                    continue
                return r
            except requests.RequestException as e:
                last_exc = e
                time.sleep(min(BACKOFF_CAP_S, BACKOFF_BASE_S ** attempt) + random.random())
        if last_exc:
            raise last_exc
        raise RuntimeError(f"Failed GET: {url}")

# =========================
# YAML fetch/cache
# =========================
WORKFLOW_YAML_CACHE: Dict[Tuple[str, str, str], str] = {}

def parse_repo(repo_full_name: str) -> Tuple[str, str]:
    parts = repo_full_name.split("/")
    if len(parts) != 2:
        raise ValueError(f"Bad full_name: {repo_full_name}")
    return parts[0], parts[1]

def gh_contents_raw(gh: GitHubClient, owner: str, repo: str, path: str, ref: str) -> Optional[str]:
    key = (f"{owner}/{repo}", path, ref)
    if key in WORKFLOW_YAML_CACHE:
        return WORKFLOW_YAML_CACHE[key]
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    r = gh.get(url)
    if r.status_code == 404:
        return None
    if r.status_code != 200:
        return None
    js = r.json()
    if not isinstance(js, dict):
        return None
    if js.get("type") != "file":
        return None
    content = js.get("content", "")
    encoding = js.get("encoding", "")
    if encoding == "base64":
        try:
            txt = base64.b64decode(content).decode("utf-8", errors="ignore")
        except Exception:
            return None
    else:
        dl = js.get("download_url")
        if not dl:
            return None
        rr = gh.get(dl)
        if rr.status_code != 200:
            return None
        txt = rr.text
    if len(WORKFLOW_YAML_CACHE) < WORKFLOW_YAML_CACHE_MAX:
        WORKFLOW_YAML_CACHE[key] = txt
    return txt

# =========================
# Stage 1 / Stage 2 carried-file parsing helpers
# =========================
def split_multi_value_cell(s: Optional[str]) -> List[str]:
    raw = norm(s)
    if not raw:
        return []
    if "||" in raw:
        parts = [x.strip() for x in raw.split("||")]
    elif "|" in raw:
        parts = [x.strip() for x in raw.split("|")]
    elif ";" in raw:
        parts = [x.strip() for x in raw.split(";")]
    else:
        parts = [x.strip() for x in raw.split(",")]
    return [p for p in parts if p]

def parse_stage1_confirmed_called_file_paths(row: Dict[str, str]) -> List[str]:
    signal = low(row.get("called_instru_signal"))
    if signal not in {"true", "1", "yes", "y"}:
        return []
    paths = split_multi_value_cell(row.get("called_instru_file_paths"))
    cleaned: List[str] = []
    for p in paths:
        p2 = p.strip().replace("\\", "/")
        if p2.startswith("./"):
            p2 = p2[2:]
        if p2:
            cleaned.append(p2)
    return unique_preserve(cleaned)

def parse_stage1_confirmed_called_origins(row: Dict[str, str]) -> Set[str]:
    vals = split_multi_value_cell(row.get("called_instru_origin_step_names"))
    return set(low(v) for v in vals if norm(v))

def parse_stage1_confirmed_called_refs(row: Dict[str, str]) -> Set[str]:
    vals = split_multi_value_cell(row.get("called_instru_origin_refs"))
    return set(norm(v) for v in vals if norm(v))

# =========================
# Workflow YAML extraction
# =========================
STEP_NAME_RE = re.compile(r"^\s*-\s*name\s*:\s*(.+?)\s*$")
STEP_USES_RE = re.compile(r"^\s*uses\s*:\s*(.+?)\s*$")
STEP_RUN_RE = re.compile(r"^\s*run\s*:\s*(.*)$")
JOB_ID_RE = re.compile(r"^\s{2}([A-Za-z0-9_.-]+)\s*:\s*$")
WORKFLOW_CALL_RE = re.compile(
    r"uses:\s*([./A-Za-z0-9_\-]+(?:/[A-Za-z0-9_\-\.]+)+\.ya?ml)(?:@([A-Za-z0-9_.\-/]+))?"
)

def strip_quotes(s: str) -> str:
    s = s.strip()
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        return s[1:-1]
    return s

def looks_like_local_workflow_ref(s: str) -> bool:
    return low(s).startswith("./.github/workflows/") and (s.endswith(".yml") or s.endswith(".yaml"))

def looks_like_local_action_ref(s: str) -> bool:
    return low(s).startswith("./") and not looks_like_local_workflow_ref(s)

def normalize_local_repo_path(s: str) -> str:
    s = s.strip().replace("\\", "/")
    if s.startswith("./"):
        s = s[2:]
    return s

def extract_steps_from_workflow_yaml(text: str) -> List[Dict[str, str]]:
    lines = text.splitlines()
    out: List[Dict[str, str]] = []
    current_job = ""
    current_step_name = ""
    i = 0
    in_steps_block = False
    while i < len(lines):
        line = lines[i]
        m_job = JOB_ID_RE.match(line)
        if m_job:
            current_job = m_job.group(1).strip()
            in_steps_block = False
        if re.match(r"^\s*steps\s*:\s*$", line):
            in_steps_block = True
            i += 1
            continue
        if in_steps_block:
            m_name = STEP_NAME_RE.match(line)
            if m_name:
                current_step_name = strip_quotes(m_name.group(1).strip())
                out.append({"job_name": current_job, "step_name": current_step_name, "uses": "", "run": ""})
                i += 1
                continue
            if out:
                m_uses = STEP_USES_RE.match(line.strip())
                if m_uses:
                    out[-1]["uses"] = strip_quotes(m_uses.group(1).strip())
                m_run = STEP_RUN_RE.match(line.strip())
                if m_run and out[-1].get("run", "") == "":
                    out[-1]["run"] = m_run.group(1)
        i += 1
    return out

# =========================
# Heuristics
# =========================
ANDROIDISH_RE = re.compile(
    r"(adb|avd|emulator|uiautomator|espresso|androidtest|connectedcheck|connectedandroidtest|manageddevice|gmd|"
    r"detox|baseline profile|macrobenchmark|instrumentation)",
    re.I,
)

THIRD_PARTY_PROVIDER_RE = re.compile(
    r"(browserstack|sauce\s*labs|saucelabs|kobiton|headspin|bitbar|perfecto|lambdatest|genymotion\s*cloud|firebase\s*test\s*lab)",
    re.I,
)

THIRD_PARTY_LIFECYCLE_RE = re.compile(
    r"(start|stop|upload|download|results?|report|session|app url|build id|device logs?)",
    re.I,
)

INSTRU_TASK_NAME_HINT_RE = re.compile(
    r"(androidtest|connectedcheck|connectedandroidtest|instrumentation|managed\s*device|gmd|detox|integration\s*test|"
    r"baseline\s*profile|macrobenchmark|uiautomator|espresso)",
    re.I,
)

CUSTOM_SCRIPT_HINT_RE = re.compile(
    r"(./gradlew|gradlew|python|bash|sh |pwsh|powershell|node |npm |yarn |ruby |bundle exec)",
    re.I,
)

FILE_HINT_INSTRU_RE = re.compile(
    r"(androidtest|connectedcheck|connectedandroidtest|detox|integration[\s_-]*test|managed[\s_-]*device|gmd|baseline[\s_-]*profile|macrobenchmark|espresso|uiautomator|instrumentation)",
    re.I,
)

ARTIFACT_HINT_RE = re.compile(
    r"(upload-artifact|download-artifact|artifact|test-results|results|report|reports|logs?)",
    re.I,
)

ENV_SETUP_HINT_RE = re.compile(
    r"(checkout|setup java|setup-jdk|setup android sdk|sdkmanager|avdmanager|create avd|start emulator|cache|restore cache)",
    re.I,
)

# =========================
# Step / flags inference
# =========================
def infer_flags_from_step(step_name: str, uses: str, run_cmd: str, target_style: str = "") -> Dict[str, Union[bool, str]]:
    sname = norm(step_name)
    suse = norm(uses)
    srun = norm(run_cmd)
    combo = " | ".join([sname, suse, srun])

    flags: Dict[str, Union[bool, str]] = {
        "yaml_match": False,
        "yaml_match_reason": "",

        "stage1_anchor_match": False,
        "stage1_anchor_match_reason": "",

        "explicit_instru": False,
        "explicit_instru_reason": "",

        "env_setup": False,
        "artifact": False,

        "third_party_provider": False,
        "third_party_provider_name": "",
        "third_party_provider_action": False,
        "third_party_config_hint": False,
        "third_party_instru_invoke": False,
        "third_party_lifecycle": False,

        "gmd_setup": False,
        "gmd_lifecycle_task": False,

        "flutter_integration_androidish": False,
        "detox_androidish": False,
        "gradle_androidtest": False,
        "baseline_profile": False,

        "custom_followed_file_instru": False,
        "custom_stage1_supported_exec": False,
    }

    if ENV_SETUP_HINT_RE.search(combo):
        flags["env_setup"] = True
    if ARTIFACT_HINT_RE.search(combo):
        flags["artifact"] = True

    if THIRD_PARTY_PROVIDER_RE.search(combo):
        flags["third_party_provider"] = True
        m = THIRD_PARTY_PROVIDER_RE.search(combo)
        flags["third_party_provider_name"] = m.group(1) if m else ""
        if THIRD_PARTY_LIFECYCLE_RE.search(combo):
            flags["third_party_lifecycle"] = True
        if ANDROIDISH_RE.search(combo):
            flags["third_party_instru_invoke"] = True
        if re.search(r"(app|apk|ipa|test-suite|project|build)", combo, re.I):
            flags["third_party_config_hint"] = True
        if re.search(r"(run|execute|trigger|start session|espresso|detox|instrumentation)", combo, re.I):
            flags["third_party_provider_action"] = True

    if re.search(r"(manageddevice|gmd|gradle managed device)", combo, re.I):
        flags["gmd_setup"] = True
        if re.search(r"(connectedcheck|androidtest|manageddevicecheck|gmd)", combo, re.I):
            flags["gmd_lifecycle_task"] = True

    if re.search(r"(flutter.*integration|integration.*flutter)", combo, re.I) and ANDROIDISH_RE.search(combo):
        flags["flutter_integration_androidish"] = True

    if re.search(r"\bdetox\b", combo, re.I) and ANDROIDISH_RE.search(combo):
        flags["detox_androidish"] = True

    if re.search(r"(connectedcheck|connectedandroidtest|androidtest)", combo, re.I):
        flags["gradle_androidtest"] = True

    if re.search(r"(baseline profile|baselineprofile|macrobenchmark)", combo, re.I):
        flags["baseline_profile"] = True

    if ANDROIDISH_RE.search(combo):
        flags["explicit_instru"] = True
        flags["explicit_instru_reason"] = "androidish"

    if normalize_style_label(target_style) == "Third-Party":
        if flags["third_party_instru_invoke"] or (
            flags["third_party_provider"] and (flags["third_party_provider_action"] or flags["third_party_config_hint"])
        ):
            flags["yaml_match"] = True
            flags["yaml_match_reason"] = "third_party_provider"
    elif normalize_style_label(target_style) == "GMD":
        if flags["gmd_setup"] or flags["gmd_lifecycle_task"]:
            flags["yaml_match"] = True
            flags["yaml_match_reason"] = "gmd"
    elif normalize_style_label(target_style) == "Custom":
        if flags["gradle_androidtest"] or flags["detox_androidish"] or flags["flutter_integration_androidish"] or flags["baseline_profile"]:
            flags["yaml_match"] = True
            flags["yaml_match_reason"] = "custom_exec"
    elif normalize_style_label(target_style) == "Real-Devices":
        if flags["explicit_instru"]:
            flags["yaml_match"] = True
            flags["yaml_match_reason"] = "real_devices_exec"
    else:
        if flags["explicit_instru"]:
            flags["yaml_match"] = True
            flags["yaml_match_reason"] = "generic_androidish"

    return flags

def mark_stage1_anchor_matches(
    steps: List[Dict[str, str]],
    stage1_anchor_name: str,
    called_origin_step_names: Set[str],
) -> None:
    anchor_low = low(stage1_anchor_name)
    for st in steps:
        step_low = low(st.get("step_name"))
        if anchor_low and step_low == anchor_low:
            st["stage1_anchor_match"] = True
            st["stage1_anchor_match_reason"] = "exact_stage1_anchor_name"
        elif step_low and step_low in called_origin_step_names:
            st["stage1_anchor_match"] = True
            st["stage1_anchor_match_reason"] = "matched_stage1_called_origin"
        else:
            st["stage1_anchor_match"] = False
            st["stage1_anchor_match_reason"] = ""

def enrich_with_called_file_support(
    gh: GitHubClient,
    owner: str,
    repo: str,
    ref: str,
    steps: List[Dict[str, str]],
    confirmed_called_paths: List[str],
    called_origin_step_names: Set[str],
    target_style: str,
) -> None:
    if not confirmed_called_paths:
        return

    file_evidence_map: Dict[str, bool] = {}
    for p in confirmed_called_paths:
        txt = gh_contents_raw(gh, owner, repo, p, ref)
        if not txt:
            continue
        file_evidence_map[p] = bool(FILE_HINT_INSTRU_RE.search(txt))

    if not file_evidence_map:
        return

    for st in steps:
        step_low = low(st.get("step_name"))
        uses_low = low(st.get("uses"))
        run_low = low(st.get("run"))
        combo = " | ".join([step_low, uses_low, run_low])

        matched_origin = step_low in called_origin_step_names if step_low else False
        matched_local_ref = any(normalize_local_repo_path(p) in combo for p in file_evidence_map.keys())

        if matched_origin or matched_local_ref:
            if any(file_evidence_map.values()):
                st["custom_followed_file_instru"] = True
                if normalize_style_label(target_style) == "Custom":
                    if CUSTOM_SCRIPT_HINT_RE.search(combo) or FILE_HINT_INSTRU_RE.search(combo):
                        st["custom_stage1_supported_exec"] = True

# =========================
# Step list / jobs API
# =========================
def list_jobs_for_run(gh: GitHubClient, owner: str, repo: str, run_id: str) -> List[dict]:
    jobs: List[dict] = []
    page = 1
    while page <= MAX_PAGES_PER_LIST:
        url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}/jobs?per_page=100&page={page}"
        r = gh.get(url)
        if r.status_code != 200:
            break
        js = r.json() if r.text else {}
        arr = js.get("jobs", [])
        if not arr:
            break
        jobs.extend(arr)
        if len(arr) < 100:
            break
        page += 1
    return jobs

def step_rows_from_jobs(jobs: List[dict]) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for job in jobs:
        jname = norm(job.get("name"))
        for st in job.get("steps") or []:
            started = st.get("started_at") or ""
            completed = st.get("completed_at") or ""
            sdt = iso_to_dt(started)
            edt = iso_to_dt(completed)
            dur = dt_to_seconds(sdt, edt)
            rows.append({
                "job_name": jname,
                "step_name": norm(st.get("name")),
                "status": norm(st.get("status")),
                "conclusion": norm(st.get("conclusion")),
                "started_at": started,
                "completed_at": completed,
                "duration_seconds": dur,
            })
    return rows

# =========================
# Anchor selection + execution selection
# =========================
def pick_instru_anchor_from_candidates(
    candidates: List[Tuple[datetime, str, str, Dict[str, Union[bool, str]]]]
) -> Tuple[Optional[datetime], str, str, str, Dict[str, Union[bool, str]]]:
    if not candidates:
        return None, "", "", "", {}

    # Highest priority: Stage1 anchor match
    st1 = [(t, n, jn, f) for (t, n, jn, f) in candidates if f.get("stage1_anchor_match")]
    if st1:
        st1.sort(key=lambda x: x[0])
        t, n, jn, f = st1[0]
        return t, n, jn, f"stage1_anchor_name_match:{f.get('stage1_anchor_match_reason','')}", f

    # Next explicit instrumentation
    exp = [(t, n, jn, f) for (t, n, jn, f) in candidates if f.get("explicit_instru")]
    if exp:
        exp.sort(key=lambda x: x[0])
        t, n, jn, f = exp[0]
        return t, n, jn, "explicit_instru_step", f

    # Third-party provider fallback
    tp = [(t, n, jn, f) for (t, n, jn, f) in candidates if f.get("third_party_provider")]
    if tp:
        tp.sort(key=lambda x: x[0])
        t, n, jn, f = tp[0]
        return t, n, jn, "third_party_provider_fallback", f

    # Emulator runner / general runner action fallback
    runner_like = []
    for (t, n, jn, f) in candidates:
        combo = low(n)
        if re.search(r"(emulator|avd|managed device|gmd|detox|integration test|androidtest|connectedcheck)", combo):
            runner_like.append((t, n, jn, f))
    if runner_like:
        runner_like.sort(key=lambda x: x[0])
        t, n, jn, f = runner_like[0]
        return t, n, jn, "emu_runner_action_step", f

    candidates.sort(key=lambda x: x[0])
    t, n, jn, f = candidates[0]
    return t, n, jn, "earliest_candidate", f

def is_exec_step(flags: Dict[str, Union[bool, str]]) -> bool:
    return bool(
        flags.get("explicit_instru")
        or flags.get("third_party_instru_invoke")
        or flags.get("gmd_lifecycle_task")
        or flags.get("flutter_integration_androidish")
        or flags.get("detox_androidish")
        or flags.get("gradle_androidtest")
        or flags.get("baseline_profile")
        or flags.get("custom_stage1_supported_exec")
    )

STYLE_METRIC_KEYS = [
    "timeline_evidence_mode",
    "first_test_step_started_at",
    "ttfts_seconds",
    "ttfts_source",

    "modified_ttfts_seconds",
    "modified_ttfts_source",
    "modified_ttfts_quality",

    "jobs_to_anchor_job_count",
    "jobs_to_anchor_job_count_source",

    "instru_started_at",
    "instru_ended_at",
    "test_exec_started_at",
    "test_exec_ended_at",

    "instru_duration_seconds",
    "pre_test_overhead_seconds",
    "core_instru_window_seconds",
    "post_test_overhead_seconds",
    "instru_exec_sum_seconds",
    "instru_exec_window_seconds",
    "instru_exec_step_count",

    "env_setup_sum_seconds",
    "artifact_sum_seconds",

    "instru_started_at_direct",
    "instru_started_at_direct_source_type",
    "instru_started_at_direct_source_layer",
    "instru_started_at_direct_observational_quality",
    "instru_started_at_direct_semantic_quality",
    "instru_started_at_direct_step_name",
    "instru_started_at_direct_job_name",
    "instru_started_at_fallback1",
    "instru_started_at_fallback1_source_type",
    "instru_started_at_fallback1_source_layer",
    "instru_started_at_fallback1_rank",
    "instru_started_at_fallback2",
    "instru_started_at_fallback2_source_type",
    "instru_started_at_fallback2_source_layer",
    "instru_started_at_fallback2_rank",
    "instru_started_at_selected_source_type",
    "instru_started_at_selected_source_layer",
    "instru_started_at_selected_observational_quality",
    "instru_started_at_selected_semantic_quality",
    "instru_started_at_selected_step_name",
    "instru_started_at_selected_job_name",
    "instru_started_at_ambiguity_count",
    "instru_started_at_ambiguity_level",

    "test_exec_started_at_direct",
    "test_exec_started_at_direct_source_type",
    "test_exec_started_at_direct_source_layer",
    "test_exec_started_at_direct_observational_quality",
    "test_exec_started_at_direct_semantic_quality",
    "test_exec_started_at_direct_step_name",
    "test_exec_started_at_direct_job_name",
    "test_exec_started_at_fallback1",
    "test_exec_started_at_fallback1_source_type",
    "test_exec_started_at_fallback1_source_layer",
    "test_exec_started_at_fallback1_rank",
    "test_exec_started_at_fallback2",
    "test_exec_started_at_fallback2_source_type",
    "test_exec_started_at_fallback2_source_layer",
    "test_exec_started_at_fallback2_rank",
    "test_exec_started_at_selected_source_type",
    "test_exec_started_at_selected_source_layer",
    "test_exec_started_at_selected_observational_quality",
    "test_exec_started_at_selected_semantic_quality",
    "test_exec_started_at_selected_step_name",
    "test_exec_started_at_selected_job_name",
    "test_exec_started_at_ambiguity_count",
    "test_exec_started_at_ambiguity_level",

    "test_exec_ended_at_direct",
    "test_exec_ended_at_direct_source_type",
    "test_exec_ended_at_direct_source_layer",
    "test_exec_ended_at_direct_observational_quality",
    "test_exec_ended_at_direct_semantic_quality",
    "test_exec_ended_at_direct_step_name",
    "test_exec_ended_at_direct_job_name",
    "test_exec_ended_at_fallback1",
    "test_exec_ended_at_fallback1_source_type",
    "test_exec_ended_at_fallback1_source_layer",
    "test_exec_ended_at_fallback1_rank",
    "test_exec_ended_at_fallback2",
    "test_exec_ended_at_fallback2_source_type",
    "test_exec_ended_at_fallback2_source_layer",
    "test_exec_ended_at_fallback2_rank",
    "test_exec_ended_at_selected_source_type",
    "test_exec_ended_at_selected_source_layer",
    "test_exec_ended_at_selected_observational_quality",
    "test_exec_ended_at_selected_semantic_quality",
    "test_exec_ended_at_selected_step_name",
    "test_exec_ended_at_selected_job_name",
    "test_exec_ended_at_ambiguity_count",
    "test_exec_ended_at_ambiguity_level",

    "instru_ended_at_direct",
    "instru_ended_at_direct_source_type",
    "instru_ended_at_direct_source_layer",
    "instru_ended_at_direct_observational_quality",
    "instru_ended_at_direct_semantic_quality",
    "instru_ended_at_direct_step_name",
    "instru_ended_at_direct_job_name",
    "instru_ended_at_fallback1",
    "instru_ended_at_fallback1_source_type",
    "instru_ended_at_fallback1_source_layer",
    "instru_ended_at_fallback1_rank",
    "instru_ended_at_fallback2",
    "instru_ended_at_fallback2_source_type",
    "instru_ended_at_fallback2_source_layer",
    "instru_ended_at_fallback2_rank",
    "instru_ended_at_selected_source_type",
    "instru_ended_at_selected_source_layer",
    "instru_ended_at_selected_observational_quality",
    "instru_ended_at_selected_semantic_quality",
    "instru_ended_at_selected_step_name",
    "instru_ended_at_selected_job_name",
    "instru_ended_at_ambiguity_count",
    "instru_ended_at_ambiguity_level",

    "instru_test_window_direct_seconds",
    "instru_test_window_direct_source_type",
    "instru_test_window_direct_source_layer",
    "instru_test_window_fallback1_seconds",
    "instru_test_window_fallback1_source_type",
    "instru_test_window_fallback1_source_layer",
    "instru_test_window_fallback1_rank",
    "instru_test_window_fallback2_seconds",
    "instru_test_window_fallback2_source_type",
    "instru_test_window_fallback2_source_layer",
    "instru_test_window_fallback2_rank",
    "instru_test_window_stage3_selected_seconds",
    "instru_test_window_stage3_selected_source_type",
    "instru_test_window_stage3_selected_source_layer",
]

def compute_style_aware_instru_end(
    target_style: str,
    anchor_job_name: str,
    exec_job_names: Set[str],
    exec_first: Optional[datetime],
    exec_last: Optional[datetime],
    step_times: List[Tuple[Optional[datetime], Optional[datetime], str, str, Dict[str, Union[bool, str]]]],
) -> Optional[datetime]:
    style = normalize_style_label(target_style)

    # broader continuation logic:
    # Third-Party AND Custom may continue beyond anchor job
    if style in {"Third-Party", "Custom"}:
        terminal_steps: List[datetime] = []
        for st_start, st_end, job_name, step_name, flags in step_times:
            if not st_start or not st_end:
                continue
            if exec_first and st_start < exec_first:
                continue

            if style == "Third-Party":
                is_cont = bool(
                    flags.get("third_party_provider")
                    or flags.get("third_party_instru_invoke")
                    or flags.get("third_party_lifecycle")
                    or flags.get("artifact")
                    or is_exec_step(flags)
                )
            else:  # Custom
                is_cont = bool(
                    flags.get("custom_followed_file_instru")
                    or flags.get("custom_stage1_supported_exec")
                    or flags.get("explicit_instru")
                    or flags.get("flutter_integration_androidish")
                    or flags.get("detox_androidish")
                    or flags.get("gradle_androidtest")
                    or flags.get("baseline_profile")
                    or flags.get("artifact")
                    or INSTRU_TASK_NAME_HINT_RE.search(step_name or "")
                )

            if is_cont:
                terminal_steps.append(st_end)

        if terminal_steps:
            return max(terminal_steps)

        if exec_last:
            return exec_last
        if exec_job_names:
            same_jobs = [st_end for _, st_end, jn, _, _ in step_times if st_end and jn in exec_job_names]
            if same_jobs:
                return max(same_jobs)
        return None

    # same-job constrained logic for Community / GMD / Real-Devices
    terminal_steps_same_job: List[datetime] = []
    for st_start, st_end, job_name, step_name, flags in step_times:
        if not st_start or not st_end:
            continue
        if job_name != anchor_job_name:
            continue
        if exec_first and st_start < exec_first:
            continue
        if is_exec_step(flags) or flags.get("artifact") or flags.get("explicit_instru"):
            terminal_steps_same_job.append(st_end)

    if terminal_steps_same_job:
        return max(terminal_steps_same_job)

    if exec_last:
        return exec_last

    same_job_all = [st_end for _, st_end, jn, _, _ in step_times if st_end and jn == anchor_job_name]
    if same_job_all:
        return max(same_job_all)
    return None


def dt_to_iso_z(dt: Optional[datetime]) -> str:
    if not dt:
        return ""
    return dt.isoformat().replace("+00:00", "Z")


def semantic_quality_for_anchor(anchor_source: str, flags: Dict[str, Union[bool, str]]) -> str:
    src = (anchor_source or "").strip().lower()
    if src == "explicit_instru_step":
        return "high"
    if src.startswith("stage1_anchor_name_match"):
        return "high"
    if flags.get("stage1_anchor_match") or flags.get("explicit_instru"):
        return "high"
    if src in {"emu_runner_action_step", "third_party_provider_fallback", "fallback_instru_evidence"}:
        return "medium"
    return "medium" if src else "unknown"


def semantic_quality_for_instru_end(step_name: str, flags: Dict[str, Union[bool, str]]) -> str:
    s = (step_name or "").strip().lower()
    if is_exec_step(flags) or flags.get("explicit_instru") or flags.get("stage1_anchor_match"):
        return "high"
    if flags.get("artifact"):
        return "medium"
    if s.startswith("post ") or s == "complete job":
        return "low"
    return "medium"


def fallback_rank_from_source(source_type: str) -> str:
    s = (source_type or "").strip().lower()
    if not s or s == "missing":
        return "missing"
    if "workflow_label" in s:
        return "weak"
    if "stage2" in s or s.startswith("s2_") or s == "telemetry_stage2":
        return "strong"
    return "medium"


def ambiguity_level(count: int) -> str:
    if count <= 1:
        return "low"
    if count <= 3:
        return "medium"
    return "high"


def build_boundary_stub(prefix: str) -> Dict[str, Union[str, int, None]]:
    return {
        f"{prefix}_direct": "",
        f"{prefix}_direct_source_type": "missing",
        f"{prefix}_direct_source_layer": "missing",
        f"{prefix}_direct_observational_quality": "missing",
        f"{prefix}_direct_semantic_quality": "unknown",
        f"{prefix}_direct_step_name": "",
        f"{prefix}_direct_job_name": "",
        f"{prefix}_fallback1": "",
        f"{prefix}_fallback1_source_type": "missing",
        f"{prefix}_fallback1_source_layer": "missing",
        f"{prefix}_fallback1_rank": "missing",
        f"{prefix}_fallback2": "",
        f"{prefix}_fallback2_source_type": "missing",
        f"{prefix}_fallback2_source_layer": "missing",
        f"{prefix}_fallback2_rank": "missing",
        f"{prefix}_selected_source_type": "missing",
        f"{prefix}_selected_source_layer": "missing",
        f"{prefix}_selected_observational_quality": "missing",
        f"{prefix}_selected_semantic_quality": "unknown",
        f"{prefix}_selected_step_name": "",
        f"{prefix}_selected_job_name": "",
        f"{prefix}_ambiguity_count": 0,
        f"{prefix}_ambiguity_level": "low",
    }


def find_terminal_step_details(
    boundary_dt: Optional[datetime],
    step_times: List[Tuple[Optional[datetime], Optional[datetime], str, str, Dict[str, Union[bool, str]]]],
    side: str = "end",
) -> Tuple[str, str, str, int]:
    if boundary_dt is None:
        return "", "", "unknown", 0
    matches: List[Tuple[int, str, str, str]] = []
    for st_start, st_end, job_name, step_name, flags in step_times:
        cmp_dt = st_end if side == "end" else st_start
        if not cmp_dt or cmp_dt != boundary_dt:
            continue
        if side == "start":
            sem = semantic_quality_for_anchor("direct_step", flags)
        else:
            sem = semantic_quality_for_instru_end(step_name, flags)
        score = {"high": 3, "medium": 2, "low": 1}.get(sem, 0)
        matches.append((score, step_name or "", job_name or "", sem))
    if not matches:
        return "", "", "unknown", 0
    matches.sort(key=lambda x: (-x[0], x[1], x[2]))
    best = matches[0]
    return best[1], best[2], best[3], len(matches)

def compute_metrics_for_event_set(
    base_start: Optional[datetime],
    events: List[Tuple[Optional[datetime], Optional[datetime], Optional[int], str, str, Dict[str, Union[bool, str]]]],
    stage2_instru_detect_method: str,
    s2_fallback: Dict[str, str],
    target_style: str = "",
) -> Dict[str, Union[str, int, None]]:

    out: Dict[str, Union[str, int, None]] = {
        "timeline_evidence_mode": "missing",
        "first_test_step_started_at": "",
        "ttfts_seconds": None,
        "ttfts_source": "missing",
        "modified_ttfts_seconds": None,
        "modified_ttfts_source": "missing",
        "modified_ttfts_quality": "missing",
        "jobs_to_anchor_job_count": None,
        "jobs_to_anchor_job_count_source": "missing",
        "instru_started_at": "",
        "instru_ended_at": "",
        "test_exec_started_at": "",
        "test_exec_ended_at": "",
        "instru_duration_seconds": None,
        "pre_test_overhead_seconds": None,
        "core_instru_window_seconds": None,
        "post_test_overhead_seconds": None,
        "instru_exec_sum_seconds": None,
        "instru_exec_window_seconds": None,
        "instru_exec_step_count": None,
        "env_setup_sum_seconds": None,
        "artifact_sum_seconds": None,
        "instru_test_window_direct_seconds": None,
        "instru_test_window_direct_source_type": "missing",
        "instru_test_window_direct_source_layer": "missing",
        "instru_test_window_fallback1_seconds": None,
        "instru_test_window_fallback1_source_type": "missing",
        "instru_test_window_fallback1_source_layer": "missing",
        "instru_test_window_fallback1_rank": "missing",
        "instru_test_window_fallback2_seconds": None,
        "instru_test_window_fallback2_source_type": "missing",
        "instru_test_window_fallback2_source_layer": "missing",
        "instru_test_window_fallback2_rank": "missing",
        "instru_test_window_stage3_selected_seconds": None,
        "instru_test_window_stage3_selected_source_type": "missing",
        "instru_test_window_stage3_selected_source_layer": "missing",
    }
    for prefix in ["instru_started_at", "test_exec_started_at", "test_exec_ended_at", "instru_ended_at"]:
        out.update(build_boundary_stub(prefix))

    # Stage 2 fallback candidates always stay available to later layers.
    s2_first = (s2_fallback.get("S2_instru_first_started_at") or "").strip()
    s2_last = (s2_fallback.get("S2_instru_last_completed_at") or "").strip()
    s2_window = safe_int_from_str(s2_fallback.get("S2_instru_window_seconds", ""))
    s2_ttfi = (s2_fallback.get("S2_time_to_first_instru_seconds") or "").strip()

    if s2_first:
        out["instru_started_at_fallback1"] = s2_first
        out["instru_started_at_fallback1_source_type"] = "telemetry_stage2"
        out["instru_started_at_fallback1_source_layer"] = "first_degree_fallback"
        out["instru_started_at_fallback1_rank"] = "strong"
    if s2_last:
        out["instru_ended_at_fallback1"] = s2_last
        out["instru_ended_at_fallback1_source_type"] = "telemetry_stage2"
        out["instru_ended_at_fallback1_source_layer"] = "first_degree_fallback"
        out["instru_ended_at_fallback1_rank"] = "strong"
    if s2_window is not None:
        out["instru_test_window_fallback1_seconds"] = s2_window
        out["instru_test_window_fallback1_source_type"] = "telemetry_stage2"
        out["instru_test_window_fallback1_source_layer"] = "first_degree_fallback"
        out["instru_test_window_fallback1_rank"] = "strong"

    if not events:
        out["timeline_evidence_mode"] = "fallback_only"
        stage1_jobs_before_anchor_count = safe_int_from_str(s2_fallback.get("jobs_before_anchor_count", ""))

        if s2_ttfi:
            try:
                out["ttfts_seconds"] = int(float(s2_ttfi))
                out["ttfts_source"] = "S2_time_to_first_instru_seconds"
            except Exception:
                pass
        if out["ttfts_seconds"] is None and base_start and s2_first:
            tt = dt_to_seconds(base_start, iso_to_dt(s2_first))
            if tt is not None:
                out["ttfts_seconds"] = tt
                out["ttfts_source"] = "S2_instru_first_started_at"
        if s2_first:
            out["instru_started_at"] = s2_first
            out["first_test_step_started_at"] = s2_first
            out["instru_started_at_selected_source_type"] = "telemetry_stage2"
            out["instru_started_at_selected_source_layer"] = "first_degree_fallback"
            out["instru_started_at_selected_observational_quality"] = "derived"
            out["instru_started_at_selected_semantic_quality"] = "unknown"
        if s2_last:
            out["instru_ended_at"] = s2_last
            out["instru_ended_at_selected_source_type"] = "telemetry_stage2"
            out["instru_ended_at_selected_source_layer"] = "first_degree_fallback"
            out["instru_ended_at_selected_observational_quality"] = "derived"
            out["instru_ended_at_selected_semantic_quality"] = "unknown"
        if s2_first and s2_last:
            d = dt_to_seconds(iso_to_dt(s2_first), iso_to_dt(s2_last))
            out["instru_duration_seconds"] = d
            out["instru_test_window_stage3_selected_seconds"] = d
            out["instru_test_window_stage3_selected_source_type"] = "telemetry_stage2"
            out["instru_test_window_stage3_selected_source_layer"] = "first_degree_fallback"
        elif s2_window is not None:
            out["instru_test_window_stage3_selected_seconds"] = s2_window
            out["instru_test_window_stage3_selected_source_type"] = "telemetry_stage2"
            out["instru_test_window_stage3_selected_source_layer"] = "first_degree_fallback"

        s2_mod_ttfts = (s2_fallback.get("S2_time_to_first_instru_from_anchor_job_seconds") or "").strip()
        s2_mod_ttfts_quality = (s2_fallback.get("S2_time_to_first_instru_from_anchor_job_quality") or "").strip()
        stage2_mod_ttfts = (s2_fallback.get("time_to_first_instru_from_anchor_job_seconds") or "").strip()
        stage2_mod_ttfts_quality = (s2_fallback.get("time_to_first_instru_from_anchor_job_quality") or "").strip()
        s2_anchor_job_start = (s2_fallback.get("S2_anchor_job_started_at") or "").strip()

        if s2_mod_ttfts:
            try:
                out["modified_ttfts_seconds"] = int(float(s2_mod_ttfts))
                out["modified_ttfts_source"] = "S2_time_to_first_instru_from_anchor_job_seconds"
                out["modified_ttfts_quality"] = s2_mod_ttfts_quality or "S2"
            except Exception:
                pass
        if out["modified_ttfts_seconds"] is None and stage2_mod_ttfts:
            try:
                out["modified_ttfts_seconds"] = int(float(stage2_mod_ttfts))
                out["modified_ttfts_source"] = "time_to_first_instru_from_anchor_job_seconds"
                out["modified_ttfts_quality"] = stage2_mod_ttfts_quality or "stage2"
            except Exception:
                pass
        if out["modified_ttfts_seconds"] is None and s2_anchor_job_start and s2_first:
            d = dt_to_seconds(iso_to_dt(s2_anchor_job_start), iso_to_dt(s2_first))
            if d is not None:
                out["modified_ttfts_seconds"] = d
                out["modified_ttfts_source"] = "S2_anchor_job_started_at_plus_S2_instru_first_started_at"
                out["modified_ttfts_quality"] = "derived"
        if stage1_jobs_before_anchor_count is not None:
            out["jobs_to_anchor_job_count"] = stage1_jobs_before_anchor_count + 1
            out["jobs_to_anchor_job_count_source"] = "stage1_jobs_before_anchor_count_plus1"

        if (stage2_instru_detect_method or "").strip().lower().startswith("workflow_label"):
            out["timeline_evidence_mode"] = "fallback_only_style_proxy"
            if out["ttfts_seconds"] is None:
                out["ttfts_seconds"] = 0
                out["ttfts_source"] = "workflow_label_proxy"
                out["instru_started_at_fallback2_source_type"] = "workflow_label_proxy"
                out["instru_started_at_fallback2_source_layer"] = "second_degree_fallback"
                out["instru_started_at_fallback2_rank"] = "weak"
            if out["modified_ttfts_seconds"] is None:
                out["modified_ttfts_seconds"] = 0
                out["modified_ttfts_source"] = "workflow_label_proxy"
                out["modified_ttfts_quality"] = "workflow_label_proxy"
        return out

    total_env = 0
    total_art = 0
    exec_first: Optional[datetime] = None
    exec_last: Optional[datetime] = None
    exec_sum = 0
    exec_count = 0
    cands: List[Tuple[datetime, str, str, Dict[str, Union[bool, str]]]] = []
    job_first_step_start: Dict[str, datetime] = {}
    exec_job_names: Set[str] = set()
    step_times: List[Tuple[Optional[datetime], Optional[datetime], str, str, Dict[str, Union[bool, str]]]] = []

    for (st_start, st_end, st_dur, step_name, job_name, flags) in events:
        if flags.get("env_setup") and st_dur is not None:
            total_env += st_dur
        if flags.get("artifact") and st_dur is not None:
            total_art += st_dur
        if st_start:
            cands.append((st_start, step_name, job_name, flags))
            if job_name:
                prev = job_first_step_start.get(job_name)
                if prev is None or st_start < prev:
                    job_first_step_start[job_name] = st_start
        if is_exec_step(flags) and st_start and st_end:
            if exec_first is None or st_start < exec_first:
                exec_first = st_start
            if exec_last is None or st_end > exec_last:
                exec_last = st_end
            if st_dur is not None:
                exec_sum += st_dur
            exec_count += 1
            if job_name:
                exec_job_names.add(job_name)
        step_times.append((st_start, st_end, job_name or "", step_name, flags))

    out["timeline_evidence_mode"] = "step_based"
    out["env_setup_sum_seconds"] = total_env if total_env > 0 else None
    out["artifact_sum_seconds"] = total_art if total_art > 0 else None

    anchor_dt, anchor_name, anchor_job_name, anchor_source, anchor_flags = pick_instru_anchor_from_candidates(cands)
    if anchor_dt is None:
        fallback: List[Tuple[datetime, str, str, Dict[str, Union[bool, str]]]] = []
        for (t, n, jn, f) in cands:
            has_instru_evidence = bool(
                f.get("stage1_anchor_match") or f.get("explicit_instru") or f.get("third_party_instru_invoke")
                or f.get("third_party_lifecycle")
                or (f.get("third_party_provider") and (f.get("third_party_provider_action") or f.get("third_party_config_hint")))
                or f.get("gmd_setup") or f.get("gmd_lifecycle_task") or f.get("baseline_profile")
                or f.get("flutter_integration_androidish") or f.get("detox_androidish")
                or f.get("custom_followed_file_instru")
                or (normalize_style_label(target_style) == "Custom" and f.get("custom_stage1_supported_exec"))
                or INSTRU_TASK_NAME_HINT_RE.search(n or "")
            )
            if has_instru_evidence:
                fallback.append((t, n, jn, f))
        if fallback:
            fallback.sort(key=lambda x: x[0])
            anchor_dt, anchor_name, anchor_job_name, _f_source, anchor_flags = fallback[0]
            anchor_source = "fallback_instru_evidence"

    if anchor_dt:
        out["instru_started_at"] = dt_to_iso_z(anchor_dt)
        out["first_test_step_started_at"] = out["instru_started_at"]
        out["instru_started_at_direct"] = out["instru_started_at"]
        out["instru_started_at_direct_source_type"] = anchor_source or "direct_step"
        out["instru_started_at_direct_source_layer"] = "direct"
        out["instru_started_at_direct_observational_quality"] = "exact"
        out["instru_started_at_direct_semantic_quality"] = semantic_quality_for_anchor(anchor_source, anchor_flags)
        out["instru_started_at_direct_step_name"] = anchor_name or ""
        out["instru_started_at_direct_job_name"] = anchor_job_name or ""
        out["instru_started_at_selected_source_type"] = out["instru_started_at_direct_source_type"]
        out["instru_started_at_selected_source_layer"] = "direct"
        out["instru_started_at_selected_observational_quality"] = "exact"
        out["instru_started_at_selected_semantic_quality"] = out["instru_started_at_direct_semantic_quality"]
        out["instru_started_at_selected_step_name"] = anchor_name or ""
        out["instru_started_at_selected_job_name"] = anchor_job_name or ""
        step_name2, job_name2, _sem2, cnt2 = find_terminal_step_details(anchor_dt, step_times, side="start")
        out["instru_started_at_ambiguity_count"] = cnt2
        out["instru_started_at_ambiguity_level"] = ambiguity_level(cnt2)
        if base_start:
            out["ttfts_seconds"] = dt_to_seconds(base_start, anchor_dt)
            out["ttfts_source"] = anchor_source
    else:
        out["timeline_evidence_mode"] = "hybrid_fallback"
        if s2_first:
            out["instru_started_at"] = s2_first
            out["first_test_step_started_at"] = s2_first
            out["instru_started_at_selected_source_type"] = "telemetry_stage2"
            out["instru_started_at_selected_source_layer"] = "first_degree_fallback"
            out["instru_started_at_selected_observational_quality"] = "derived"
            out["instru_started_at_selected_semantic_quality"] = "unknown"
        if s2_ttfi:
            try:
                out["ttfts_seconds"] = int(float(s2_ttfi))
                out["ttfts_source"] = "S2_time_to_first_instru_seconds"
            except Exception:
                pass
        if out["ttfts_seconds"] is None and base_start and s2_first:
            tt = dt_to_seconds(base_start, iso_to_dt(s2_first))
            if tt is not None:
                out["ttfts_seconds"] = tt
                out["ttfts_source"] = "S2_instru_first_started_at"

    if anchor_job_name and anchor_job_name in job_first_step_start:
        anchor_job_start_dt = job_first_step_start[anchor_job_name]
        mod = dt_to_seconds(anchor_job_start_dt, anchor_dt)
        out["modified_ttfts_seconds"] = mod
        out["modified_ttfts_source"] = "runtime_anchor_job_earliest_step_to_anchor_step"
        out["modified_ttfts_quality"] = "runtime_observed"
    else:
        s2_mod_ttfts = (s2_fallback.get("S2_time_to_first_instru_from_anchor_job_seconds") or "").strip()
        s2_mod_ttfts_quality = (s2_fallback.get("S2_time_to_first_instru_from_anchor_job_quality") or "").strip()
        stage2_mod_ttfts = (s2_fallback.get("time_to_first_instru_from_anchor_job_seconds") or "").strip()
        stage2_mod_ttfts_quality = (s2_fallback.get("time_to_first_instru_from_anchor_job_quality") or "").strip()
        s2_anchor_job_start = (s2_fallback.get("S2_anchor_job_started_at") or "").strip()
        if s2_mod_ttfts:
            try:
                out["modified_ttfts_seconds"] = int(float(s2_mod_ttfts))
                out["modified_ttfts_source"] = "S2_time_to_first_instru_from_anchor_job_seconds"
                out["modified_ttfts_quality"] = s2_mod_ttfts_quality or "S2"
            except Exception:
                pass
        if out["modified_ttfts_seconds"] is None and stage2_mod_ttfts:
            try:
                out["modified_ttfts_seconds"] = int(float(stage2_mod_ttfts))
                out["modified_ttfts_source"] = "time_to_first_instru_from_anchor_job_seconds"
                out["modified_ttfts_quality"] = stage2_mod_ttfts_quality or "stage2"
            except Exception:
                pass
        if out["modified_ttfts_seconds"] is None and s2_anchor_job_start and s2_first:
            d = dt_to_seconds(iso_to_dt(s2_anchor_job_start), iso_to_dt(s2_first))
            if d is not None:
                out["modified_ttfts_seconds"] = d
                out["modified_ttfts_source"] = "S2_anchor_job_started_at_plus_S2_instru_first_started_at"
                out["modified_ttfts_quality"] = "derived"
        if out["modified_ttfts_seconds"] is None and (stage2_instru_detect_method or "").strip().lower().startswith("workflow_label"):
            out["modified_ttfts_seconds"] = 0
            out["modified_ttfts_source"] = "workflow_label_proxy"
            out["modified_ttfts_quality"] = "workflow_label_proxy"

    if anchor_job_name and job_first_step_start:
        ordered_jobs = sorted(job_first_step_start.items(), key=lambda kv: kv[1])
        idx = None
        for i, (jn, _dt) in enumerate(ordered_jobs):
            if jn == anchor_job_name:
                idx = i
                break
        if idx is not None:
            out["jobs_to_anchor_job_count"] = idx + 1
            out["jobs_to_anchor_job_count_source"] = "runtime_observed_job_order"
    if out["jobs_to_anchor_job_count"] is None:
        stage1_jobs_before_anchor_count = safe_int_from_str(s2_fallback.get("jobs_before_anchor_count", ""))
        if stage1_jobs_before_anchor_count is not None:
            out["jobs_to_anchor_job_count"] = stage1_jobs_before_anchor_count + 1
            out["jobs_to_anchor_job_count_source"] = "stage1_jobs_before_anchor_count_plus1"

    if exec_first:
        out["test_exec_started_at"] = dt_to_iso_z(exec_first)
        step_name2, job_name2, sem2, cnt2 = find_terminal_step_details(exec_first, step_times, side="start")
        out["test_exec_started_at_direct"] = out["test_exec_started_at"]
        out["test_exec_started_at_direct_source_type"] = "direct_exec_step"
        out["test_exec_started_at_direct_source_layer"] = "direct"
        out["test_exec_started_at_direct_observational_quality"] = "exact"
        out["test_exec_started_at_direct_semantic_quality"] = sem2 if sem2 != "unknown" else "high"
        out["test_exec_started_at_direct_step_name"] = step_name2
        out["test_exec_started_at_direct_job_name"] = job_name2
        out["test_exec_started_at_selected_source_type"] = "direct_exec_step"
        out["test_exec_started_at_selected_source_layer"] = "direct"
        out["test_exec_started_at_selected_observational_quality"] = "exact"
        out["test_exec_started_at_selected_semantic_quality"] = out["test_exec_started_at_direct_semantic_quality"]
        out["test_exec_started_at_selected_step_name"] = step_name2
        out["test_exec_started_at_selected_job_name"] = job_name2
        out["test_exec_started_at_ambiguity_count"] = cnt2
        out["test_exec_started_at_ambiguity_level"] = ambiguity_level(cnt2)
    if exec_last:
        out["test_exec_ended_at"] = dt_to_iso_z(exec_last)
        step_name2, job_name2, sem2, cnt2 = find_terminal_step_details(exec_last, step_times, side="end")
        out["test_exec_ended_at_direct"] = out["test_exec_ended_at"]
        out["test_exec_ended_at_direct_source_type"] = "direct_exec_step"
        out["test_exec_ended_at_direct_source_layer"] = "direct"
        out["test_exec_ended_at_direct_observational_quality"] = "exact"
        out["test_exec_ended_at_direct_semantic_quality"] = sem2 if sem2 != "unknown" else "high"
        out["test_exec_ended_at_direct_step_name"] = step_name2
        out["test_exec_ended_at_direct_job_name"] = job_name2
        out["test_exec_ended_at_selected_source_type"] = "direct_exec_step"
        out["test_exec_ended_at_selected_source_layer"] = "direct"
        out["test_exec_ended_at_selected_observational_quality"] = "exact"
        out["test_exec_ended_at_selected_semantic_quality"] = out["test_exec_ended_at_direct_semantic_quality"]
        out["test_exec_ended_at_selected_step_name"] = step_name2
        out["test_exec_ended_at_selected_job_name"] = job_name2
        out["test_exec_ended_at_ambiguity_count"] = cnt2
        out["test_exec_ended_at_ambiguity_level"] = ambiguity_level(cnt2)

    instru_end = compute_style_aware_instru_end(
        target_style=target_style,
        anchor_job_name=anchor_job_name,
        exec_job_names=exec_job_names,
        exec_first=exec_first,
        exec_last=exec_last,
        step_times=step_times,
    )
    if instru_end:
        out["instru_ended_at"] = dt_to_iso_z(instru_end)
        step_name2, job_name2, sem2, cnt2 = find_terminal_step_details(instru_end, step_times, side="end")
        out["instru_ended_at_direct"] = out["instru_ended_at"]
        out["instru_ended_at_direct_source_type"] = "direct_terminal_step"
        out["instru_ended_at_direct_source_layer"] = "direct"
        out["instru_ended_at_direct_observational_quality"] = "exact"
        out["instru_ended_at_direct_semantic_quality"] = sem2
        out["instru_ended_at_direct_step_name"] = step_name2
        out["instru_ended_at_direct_job_name"] = job_name2
        out["instru_ended_at_selected_source_type"] = "direct_terminal_step"
        out["instru_ended_at_selected_source_layer"] = "direct"
        out["instru_ended_at_selected_observational_quality"] = "exact"
        out["instru_ended_at_selected_semantic_quality"] = sem2
        out["instru_ended_at_selected_step_name"] = step_name2
        out["instru_ended_at_selected_job_name"] = job_name2
        out["instru_ended_at_ambiguity_count"] = cnt2
        out["instru_ended_at_ambiguity_level"] = ambiguity_level(cnt2)

    if anchor_dt and instru_end:
        out["instru_duration_seconds"] = dt_to_seconds(anchor_dt, instru_end)
        out["instru_test_window_direct_seconds"] = out["instru_duration_seconds"]
        out["instru_test_window_direct_source_type"] = "direct_instru_path_window"
        out["instru_test_window_direct_source_layer"] = "direct"
        out["instru_test_window_stage3_selected_seconds"] = out["instru_duration_seconds"]
        out["instru_test_window_stage3_selected_source_type"] = "direct_instru_path_window"
        out["instru_test_window_stage3_selected_source_layer"] = "direct"
    elif s2_window is not None:
        out["instru_test_window_stage3_selected_seconds"] = s2_window
        out["instru_test_window_stage3_selected_source_type"] = "telemetry_stage2"
        out["instru_test_window_stage3_selected_source_layer"] = "first_degree_fallback"

    if anchor_dt and exec_first:
        out["pre_test_overhead_seconds"] = dt_to_seconds(anchor_dt, exec_first)
    if exec_first and exec_last:
        out["core_instru_window_seconds"] = dt_to_seconds(exec_first, exec_last)
        out["instru_exec_window_seconds"] = out["core_instru_window_seconds"]
        out["instru_exec_sum_seconds"] = exec_sum if exec_sum > 0 else None
        out["instru_exec_step_count"] = exec_count if exec_count > 0 else None
    if exec_last and instru_end:
        out["post_test_overhead_seconds"] = dt_to_seconds(exec_last, instru_end)

    if not anchor_dt and (exec_first or exec_last or instru_end):
        out["timeline_evidence_mode"] = "hybrid_fallback"
    return out

def build_stage3_outputs_for_run(
    gh: GitHubClient,
    file_text_cache: Dict[str, str],
    row: Dict[str, str],
) -> Tuple[Dict[str, object], List[Dict[str, object]], List[Dict[str, object]]]:
    full_name = norm(row.get("full_name"))
    workflow_path = norm(row.get("workflow_path"))
    workflow_ref = norm(row.get("workflow_ref"))
    run_id = norm(row.get("run_id"))
    run_started_at = iso_to_dt(row.get("run_started_at"))

    declared_styles = split_styles(row.get("inferred_styles"))
    target_styles = declared_styles[:] if declared_styles else [normalize_style_label(row.get("inferred_style"))]

    owner, repo = parse_repo(full_name)

    jobs = list_jobs_for_run(gh, owner, repo, run_id)
    step_rows = step_rows_from_jobs(jobs)

    workflow_yaml = ""
    if FETCH_WORKFLOW_YAML and workflow_path and workflow_ref:
        workflow_yaml = gh_contents_raw(gh, owner, repo, workflow_path, workflow_ref) or ""

    yaml_steps = extract_steps_from_workflow_yaml(workflow_yaml) if workflow_yaml else []

    stage1_anchor_name = row.get("anchor_step_name") or row.get("instru_step_name") or ""
    called_paths = parse_stage1_confirmed_called_file_paths(row)
    called_origin_step_names = parse_stage1_confirmed_called_origins(row)

    yaml_by_stepname: Dict[str, List[Dict[str, str]]] = {}
    for ys in yaml_steps:
        yaml_by_stepname.setdefault(low(ys.get("step_name")), []).append(ys)

    merged_steps: List[Dict[str, object]] = []
    by_job_name_yaml: Dict[str, List[Dict[str, str]]] = {}
    for ys in yaml_steps:
        by_job_name_yaml.setdefault(low(ys.get("job_name")), []).append(ys)

    # map runtime step -> candidate YAML entry using same job + exact step name if possible
    for sr in step_rows:
        rt_job = low(sr["job_name"])
        rt_step = low(sr["step_name"])
        matched_yaml = None
        for ys in by_job_name_yaml.get(rt_job, []):
            if low(ys.get("step_name")) == rt_step:
                matched_yaml = ys
                break
        if matched_yaml is None:
            cands = yaml_by_stepname.get(rt_step, [])
            matched_yaml = cands[0] if cands else {"job_name": sr["job_name"], "step_name": sr["step_name"], "uses": "", "run": ""}

        merged = {
            "job_name": sr["job_name"],
            "step_name": sr["step_name"],
            "status": sr["status"],
            "conclusion": sr["conclusion"],
            "started_at": sr["started_at"],
            "completed_at": sr["completed_at"],
            "duration_seconds": sr["duration_seconds"],
            "uses": matched_yaml.get("uses", ""),
            "run": matched_yaml.get("run", ""),
        }
        merged_steps.append(merged)

    # stage1/2 pass-through file support only if Stage1 confirmed
    if called_paths:
        enrich_with_called_file_support(
            gh=gh,
            owner=owner,
            repo=repo,
            ref=workflow_ref,
            steps=merged_steps,
            confirmed_called_paths=called_paths,
            called_origin_step_names=called_origin_step_names,
            target_style=normalize_style_label(row.get("inferred_style")) or (target_styles[0] if target_styles else ""),
        )

    mark_stage1_anchor_matches(merged_steps, stage1_anchor_name, called_origin_step_names)

    stage2_instru_detect_method = norm(row.get("instru_detect_method"))
    s2_fallback = dict(row)

    all_step_breakdown_rows: List[Dict[str, object]] = []
    per_style_rows: List[Dict[str, object]] = []

    for target_style in target_styles:
        tstyle = normalize_style_label(target_style)
        events: List[Tuple[Optional[datetime], Optional[datetime], Optional[int], str, str, Dict[str, Union[bool, str]]]] = []

        for st in merged_steps:
            flags = infer_flags_from_step(
                step_name=st["step_name"],
                uses=st["uses"],
                run_cmd=st["run"],
                target_style=tstyle,
            )

            # merge carried support flags from earlier helpers
            if st.get("stage1_anchor_match"):
                flags["stage1_anchor_match"] = True
                flags["stage1_anchor_match_reason"] = st.get("stage1_anchor_match_reason", "")
            if st.get("custom_followed_file_instru"):
                flags["custom_followed_file_instru"] = True
            if st.get("custom_stage1_supported_exec"):
                flags["custom_stage1_supported_exec"] = True

            st_start = iso_to_dt(st["started_at"])
            st_end = iso_to_dt(st["completed_at"])
            st_dur = st["duration_seconds"] if isinstance(st["duration_seconds"], int) else safe_int_from_str(str(st["duration_seconds"]))
            events.append((st_start, st_end, st_dur, st["step_name"], st["job_name"], flags))

            br = {
                "full_name": full_name,
                "workflow_path": workflow_path,
                "workflow_ref": workflow_ref,
                "run_id": run_id,
                "target_style": tstyle,

                "job_name": st["job_name"],
                "step_name": st["step_name"],
                "status": st["status"],
                "conclusion": st["conclusion"],
                "started_at": st["started_at"],
                "completed_at": st["completed_at"],
                "duration_seconds": st_dur,
                "uses": st["uses"],
                "run": st["run"],

                "category": (
                    "artifact" if flags.get("artifact")
                    else "env_setup" if flags.get("env_setup")
                    else "test" if is_exec_step(flags)
                    else "other"
                ),
                "category_reason": (
                    "artifact" if flags.get("artifact")
                    else "env_setup" if flags.get("env_setup")
                    else "exec_step" if is_exec_step(flags)
                    else ""
                ),
                "step_style_tag": tstyle,
                "step_style_reason": (
                    "yaml_match" if flags.get("yaml_match")
                    else "stage1_anchor" if flags.get("stage1_anchor_match")
                    else "carried_called_file" if flags.get("custom_followed_file_instru")
                    else ""
                ),

                "yaml_match": str(bool(flags.get("yaml_match"))).lower(),
                "yaml_match_reason": flags.get("yaml_match_reason", ""),
                "stage1_anchor_match": str(bool(flags.get("stage1_anchor_match"))).lower(),
                "stage1_anchor_match_reason": flags.get("stage1_anchor_match_reason", ""),

                "explicit_instru": str(bool(flags.get("explicit_instru"))).lower(),
                "explicit_instru_reason": flags.get("explicit_instru_reason", ""),
                "env_setup": str(bool(flags.get("env_setup"))).lower(),
                "artifact": str(bool(flags.get("artifact"))).lower(),

                "third_party_provider": str(bool(flags.get("third_party_provider"))).lower(),
                "third_party_provider_name": flags.get("third_party_provider_name", ""),
                "third_party_provider_action": str(bool(flags.get("third_party_provider_action"))).lower(),
                "third_party_config_hint": str(bool(flags.get("third_party_config_hint"))).lower(),
                "third_party_instru_invoke": str(bool(flags.get("third_party_instru_invoke"))).lower(),
                "third_party_lifecycle": str(bool(flags.get("third_party_lifecycle"))).lower(),

                "gmd_setup": str(bool(flags.get("gmd_setup"))).lower(),
                "gmd_lifecycle_task": str(bool(flags.get("gmd_lifecycle_task"))).lower(),

                "flutter_integration_androidish": str(bool(flags.get("flutter_integration_androidish"))).lower(),
                "detox_androidish": str(bool(flags.get("detox_androidish"))).lower(),
                "gradle_androidtest": str(bool(flags.get("gradle_androidtest"))).lower(),
                "baseline_profile": str(bool(flags.get("baseline_profile"))).lower(),

                "custom_followed_file_instru": str(bool(flags.get("custom_followed_file_instru"))).lower(),
                "custom_stage1_supported_exec": str(bool(flags.get("custom_stage1_supported_exec"))).lower(),
            }
            all_step_breakdown_rows.append(br)

        metrics = compute_metrics_for_event_set(
            base_start=run_started_at,
            events=events,
            stage2_instru_detect_method=stage2_instru_detect_method,
            s2_fallback=s2_fallback,
            target_style=tstyle,
        )

        pr: Dict[str, object] = {
            "full_name": full_name,
            "workflow_path": workflow_path,
            "workflow_ref": workflow_ref,
            "run_id": run_id,
            "target_style": tstyle,
            "inferred_styles_all": safe_join_names(target_styles),
        }
        for k in STYLE_METRIC_KEYS:
            pr[k] = metrics.get(k, "")
        per_style_rows.append(pr)

    # build run-level rollup
    run_out: Dict[str, object] = dict(row)
    run_out["styles_count_stage3"] = len(per_style_rows)
    run_out["styles_stage3"] = safe_join_names([r["target_style"] for r in per_style_rows])

    # style-specific columns at run level
    for sty in STYLE_CANONICAL:
        pref = {
            "Community": "community",
            "Custom": "custom",
            "GMD": "gmd",
            "Third-Party": "third_party",
            "Real-Devices": "real_devices",
        }[sty]
        found = None
        for pr in per_style_rows:
            if pr["target_style"] == sty:
                found = pr
                break
        if found:
            for k in STYLE_METRIC_KEYS:
                run_out[f"{pref}_{k}"] = found.get(k, "")
        else:
            for k in STYLE_METRIC_KEYS:
                run_out[f"{pref}_{k}"] = ""

    return run_out, all_step_breakdown_rows, per_style_rows

# =========================
# CSV fields
# =========================
BASE_RUN_KEEP_FIELDS = [
    # existing Stage2/run fields preserved
    "full_name", "workflow_path", "workflow_ref", "workflow_id", "run_id",
    "run_started_at", "run_completed_at", "run_duration_seconds",
    "triggering_actor", "run_attempt", "run_status", "run_conclusion",
    "inferred_style", "inferred_styles", "instru_detect_method", "instru_step_name",
    "anchor_step_name", "anchor_job_started_at", "anchor_job_start_source",
    "time_to_first_instru_seconds", "time_to_first_instru_source",
    "time_to_first_instru_from_anchor_job_seconds", "time_to_first_instru_from_anchor_job_quality",
    "jobs_before_anchor_count",
    "called_instru_signal", "called_instru_file_paths", "called_instru_origin_refs",
    "called_instru_origin_step_names", "called_instru_file_types",
    "S2_anchor_job_started_at", "S2_time_to_first_instru_seconds",
    "S2_time_to_first_instru_from_anchor_job_seconds", "S2_time_to_first_instru_from_anchor_job_quality",
    "S2_instru_first_started_at", "S2_instru_last_completed_at", "S2_instru_window_seconds",
]

out_fieldnames_3a = BASE_RUN_KEEP_FIELDS + [
    "styles_count_stage3",
    "styles_stage3",
]
for sty, pref in [
    ("Community", "community"),
    ("Custom", "custom"),
    ("GMD", "gmd"),
    ("Third-Party", "third_party"),
    ("Real-Devices", "real_devices"),
]:
    for k in STYLE_METRIC_KEYS:
        out_fieldnames_3a.append(f"{pref}_{k}")

out_fieldnames_3b = [
    "full_name", "workflow_path", "workflow_ref", "run_id", "target_style",
    "job_name", "step_name", "status", "conclusion",
    "started_at", "completed_at", "duration_seconds",
    "uses", "run",
    "category", "category_reason",
    "step_style_tag", "step_style_reason",
    "yaml_match", "yaml_match_reason",
    "stage1_anchor_match", "stage1_anchor_match_reason",
    "explicit_instru", "explicit_instru_reason",
    "env_setup", "artifact",
    "third_party_provider", "third_party_provider_name",
    "third_party_provider_action", "third_party_config_hint",
    "third_party_instru_invoke", "third_party_lifecycle",
    "gmd_setup", "gmd_lifecycle_task",
    "flutter_integration_androidish", "detox_androidish",
    "gradle_androidtest", "baseline_profile",
    "custom_followed_file_instru", "custom_stage1_supported_exec",
]

per_style_fields = ["full_name", "workflow_path", "workflow_ref", "run_id", "target_style", "inferred_styles_all"] + STYLE_METRIC_KEYS

# =========================
# Main
# =========================
def main():
    tokens = read_env_tokens(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens=tokens)

    rows_in = read_csv_rows(IN_STAGE2_CSV)
    if PROCESS_ONLY_RELEVANT_ROWS:
        rows_in = [r for r in rows_in if norm(r.get("run_id"))]

    all_step_rows: List[Dict[str, object]] = []
    all_per_style_rows: List[Dict[str, object]] = []
    rows: List[Dict[str, object]] = []
    extracted_ts = now_utc_iso()

    iterator = tqdm(rows_in, desc="Stage3") if tqdm else rows_in
    file_text_cache: Dict[str, str] = {}

    for row in iterator:
        try:
            run_out, breakdown_rows, per_style_rows = build_stage3_outputs_for_run(gh, file_text_cache, row)
            run_out["stage3_extracted_at_utc"] = extracted_ts
            rows.append(run_out)

            for br in breakdown_rows:
                br["stage3_extracted_at_utc"] = extracted_ts
                all_step_rows.append(br)

            for pr in per_style_rows:
                pr2 = dict(pr)
                for k in STYLE_METRIC_KEYS:
                    if k in pr:
                        pr2[k] = pr[k]
                pr2["stage3_extracted_at_utc"] = extracted_ts
                all_per_style_rows.append(pr2)
        except Exception as e:
            fallback = dict(row)
            fallback["styles_count_stage3"] = 0
            fallback["styles_stage3"] = ""
            for sty, pref in [
                ("Community", "community"),
                ("Custom", "custom"),
                ("GMD", "gmd"),
                ("Third-Party", "third_party"),
                ("Real-Devices", "real_devices"),
            ]:
                for k in STYLE_METRIC_KEYS:
                    fallback[f"{pref}_{k}"] = ""
            fallback["stage3_extracted_at_utc"] = extracted_ts
            fallback["stage3_error"] = str(e)
            rows.append(fallback)

    write_csv(OUT_STAGE3B_STEPS_CSV, out_fieldnames_3b, all_step_rows)
    write_csv(OUT_STAGE3A_RUNS_CSV, out_fieldnames_3a, rows)
    write_csv(OUT_STAGE3C_RUN_PER_STYLE_CSV, per_style_fields, all_per_style_rows)

    print("[done] Run metrics:", OUT_STAGE3A_RUNS_CSV)
    print("[done] Step breakdown:", OUT_STAGE3B_STEPS_CSV)
    print("[done] Run x style:", OUT_STAGE3C_RUN_PER_STYLE_CSV)

if __name__ == "__main__":
    main()

Stage3: 100%|██████████| 22247/22247 [3:54:32<00:00,  1.58it/s]  


[done] Run metrics: C:\Android Mobile App\ICST2026_Ext\run_metrics_v16_stage3_enhanced.csv
[done] Step breakdown: C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv
[done] Run x style: C:\Android Mobile App\ICST2026_Ext\run_per_style_v1_stage3.csv


## Stage 4 — Build workload/test signature layer (artifact-first) and parse result/report artifacts

In [4]:
# -*- coding: utf-8 -*-
"""
Stage 4 (MIN SIGNATURE, FIXED + STAGE3-ONLY) — Style-agnostic workload signature for normalization

Fixes included:
1) full_name normalization (handle bad/missing columns / URLs / BOM / whitespace)
2) job_count_total_bucket "strange values" (e.g., dates) => stricter numeric parsing + safer source selection
3) Improve runner_os_bucket detection (prefer run/job telemetry if present, else step rows, else YAML runs-on parse)
4) Restore test_suite_size_bucket by extracting junit_cases via bounded artifact parsing (optional but enabled by default)
5) **NEW (critical): Stage 4 now fingerprints ONLY Stage 3 executed runs**
   - Stage 3 condition: instru_job_count > 0 (executed instrumentation evidence)
6) **NEW (recommended): emit BOTH base and full signature hashes**
   - base: OS + jobs + steps  (robust when suite size is unknown)
   - full: OS + jobs + steps + suite size (refines when known)
   - signature_hash column remains the FULL hash for backward compatibility

Signature hash inputs (style-agnostic):
- runner_os_bucket
- job_count_total_bucket
- step_count_total_bucket (executed if available else declared from YAML else unknown)
- test_suite_size_bucket (from parsed junit_cases if available else unknown)

Provenance/support fields included:
- signature_inputs (steps/yaml/artifacts presence)
- step_count_exec + declared + source
- junit_cases + source

Inputs (ROOT_DIR):
- run_metrics_v16_stage3_enhanced.csv
- run_steps_v16_stage3_breakdown.csv (must be CSV; unzip if needed)

Output:
- run_workload_signature_v2_min_fixed.csv
"""

import base64
import csv
import hashlib
import random
import re
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union
import xml.etree.ElementTree as ET

import requests

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_RUN_METRICS_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
IN_RUN_STEPS_CSV   = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"

OUT_STAGE4_SIGNATURE_CSV = ROOT_DIR / "run_workload_signature_v3.csv"

MAX_TOKENS_TO_USE = 7
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 90
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# YAML fetching (for declared step counts + runs-on fallback)
FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 5000

# Artifact parsing (to get junit_cases)
DOWNLOAD_AND_PARSE_ARTIFACTS = True
MAX_ARTIFACT_ZIP_BYTES = 15 * 1024 * 1024   # 15MB cap
MAX_ARTIFACTS_TO_PARSE = 3
MAX_XMLS_PER_ARTIFACT = 40   # slightly higher than before (still bounded)

# =========================
# Helpers
# =========================
BOM = "\ufeff"
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}", re.MULTILINE)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def safe_lower(s: str) -> str:
    return (s or "").strip().lower()

def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()

def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input CSV: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            row = {}
            for k, v in r.items():
                row[_clean_key(k)] = (v or "")
            rows.append(row)
    return rows, fields

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def first_nonempty(row: Dict[str, str], keys: List[str]) -> str:
    for k in keys:
        v = (row.get(k) or "").strip()
        if v:
            return v
    return ""

def to_int_loose(x: str, default: int = 0) -> int:
    """
    Loose numeric parsing for counters that might come as floats/strings.
    Returns default if empty/non-numeric.
    """
    s = (x or "").strip()
    if not s or s.lower() in {"nan", "none"}:
        return default
    try:
        return int(float(s))
    except Exception:
        return default

def parse_int_strict(s: str) -> Optional[int]:
    """
    Return int if string is clearly numeric; reject dates/timestamps/iso strings.

    Minimal formatting fix:
    - Accept comma/underscore formatted ints: "1,234", "1_234"
    - Accept integer-ish floats: "8.0"
    - Accept job-count strings with one number: "8 jobs", "jobs=8", "total_jobs: 8"
    - Still rejects datetime-like strings.
    """
    s = (s or "").strip()
    if not s:
        return None

    # reject obvious ISO/datetime formats
    if re.search(r"\d{4}-\d{2}-\d{2}", s):
        return None
    if re.search(r"T\d{2}:\d{2}:\d{2}", s):
        return None
    if re.search(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", s):
        return None

    # normalize thousands separators
    s2 = s.replace(",", "").replace("_", "").strip()

    # accept digits only
    if re.fullmatch(r"\d+", s2):
        try:
            return int(s2)
        except Exception:
            return None

    # accept digits with decimal .0
    if re.fullmatch(r"\d+\.0+", s2):
        try:
            return int(float(s2))
        except Exception:
            return None

    # accept a single number embedded in a job-count-looking string
    low = s2.lower()
    if any(k in low for k in ["job", "jobs", "total_jobs", "jobs_total", "job_count", "jobs_count"]):
        nums = re.findall(r"\d+", s2)
        if len(nums) == 1:
            try:
                return int(nums[0])
            except Exception:
                return None

    return None

def sanitize_gha_expr(text: str) -> str:
    if not text:
        return ""
    return GHA_EXPR_RE.sub("MATRIX", text)

# --- ONLY ADDITION (no other changes): stable job identity to avoid job_name collisions in fallback counting ---
def _job_identity_from_step_row(s: Dict[str, str]) -> str:
    """
    Prefer stable identifiers to avoid collapsing matrix jobs that share job_name.
    Falls back to the same fields you already have, but in a safer priority order.
    """
    job_id = (s.get("job_id") or "").strip()
    if job_id:
        return f"id:{job_id}"

    job_url = (s.get("job_url") or s.get("job_html_url") or s.get("html_url") or s.get("url") or "").strip()
    if job_url:
        return f"url:{job_url}"

    job_ordinal = (s.get("job_ordinal_in_run") or s.get("job_ordinal") or s.get("job_index") or s.get("job_number") or s.get("job_position") or "").strip()
    if job_ordinal:
        return f"ord:{job_ordinal}"

    job_name = (s.get("job_name") or "").strip()
    if job_name:
        attempt = (s.get("job_attempt") or s.get("attempt") or "").strip()
        matrix = (s.get("matrix") or s.get("strategy_matrix") or s.get("matrix_id") or s.get("job_matrix_id") or "").strip()
        return f"name:{job_name}|attempt:{attempt}|matrix:{matrix}"

    return ""

# -------------------------
# full_name normalization
# -------------------------
FULL_NAME_RE = re.compile(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$")

def normalize_full_name(v: str) -> str:
    v = (v or "").strip().replace(BOM, "")
    if not v:
        return ""
    # If URL, extract owner/repo
    m = re.search(r"github\.com/([A-Za-z0-9_.-]+)/([A-Za-z0-9_.-]+)", v)
    if m:
        return f"{m.group(1)}/{m.group(2)}"
    # If already owner/repo
    if FULL_NAME_RE.match(v):
        return v
    # If "owner repo" or "owner:repo"
    v2 = v.replace(":", "/").replace("\\", "/").strip()
    if FULL_NAME_RE.match(v2):
        return v2
    # Last resort: try splitting
    parts = [p for p in re.split(r"[\s/]+", v2) if p]
    if len(parts) >= 2:
        cand = f"{parts[-2]}/{parts[-1]}"
        if FULL_NAME_RE.match(cand):
            return cand
    return ""

# -------------------------
# Bucketing
# -------------------------
def bucket_runner_os(os_raw: str) -> str:
    s = safe_lower(os_raw)
    if "ubuntu" in s or "linux" in s:
        return "ubuntu"
    if "macos" in s or "osx" in s or (s.startswith("mac") and "machine" not in s):
        return "macos"
    if "windows" in s or s.startswith("win"):
        return "windows"
    if not s:
        return "unknown"
    return "mixed_or_unknown"

def bucket_job_count(n: Optional[int]) -> str:
    if n is None:
        return "unknown"
    if n <= 1:
        return "1"
    if 2 <= n <= 3:
        return "2_3"
    if 4 <= n <= 6:
        return "4_6"
    return ">6"

def bucket_step_count(n: Optional[int]) -> str:
    if n is None:
        return "unknown"
    if n <= 20:
        return "<=20"
    if 21 <= n <= 40:
        return "21_40"
    if 41 <= n <= 80:
        return "41_80"
    return ">80"

def bucket_suite_size(n: Optional[int]) -> str:
    if n is None or n <= 0:
        return "unknown"
    if n <= 100:
        return "1_100"
    if n <= 500:
        return "101_500"
    if n <= 2000:
        return "501_2000"
    return ">2000"

# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "stage4-signature-min-fixed/1.2",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request(self, method: str, url: str, params: Optional[Dict] = None, stream: bool = False) -> Optional[requests.Response]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S), stream=stream)
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            return resp

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        resp = self.request(method, url, params=params, stream=False)
        if resp is None:
            return None
        try:
            return resp.json()
        except Exception:
            return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# =========================
# GitHub endpoints
# =========================
def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        resp = gh.request("GET", dl, params=None, stream=False)
        if resp and resp.status_code == 200:
            return resp.text or ""
    return ""

def list_run_artifacts(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/artifacts"
    return list(gh.paginate(url, params={}, item_key="artifacts"))

def download_artifact_zip(gh: GitHubClient, full_name: str, artifact_id: int) -> Optional[bytes]:
    url = f"https://api.github.com/repos/{full_name}/actions/artifacts/{artifact_id}/zip"
    resp = gh.request("GET", url, params=None, stream=True)
    if resp is None or resp.status_code != 200:
        return None
    data = bytearray()
    try:
        for chunk in resp.iter_content(chunk_size=1024 * 128):
            if not chunk:
                continue
            data.extend(chunk)
            if len(data) > MAX_ARTIFACT_ZIP_BYTES:
                return None
    except Exception:
        return None
    return bytes(data)

# =========================
# Declared step counting (conservative)
# =========================
def count_declared_steps_from_yaml(yaml_text: str) -> Optional[int]:
    """
    Conservative YAML step count:
    - Does NOT expand matrix
    - Does NOT follow reusable workflows
    - Counts only literal list items under "steps:" blocks
    """
    if not yaml_text or not yaml_text.strip():
        return None
    y = sanitize_gha_expr(yaml_text)
    lines = y.splitlines()

    total_steps = 0
    i = 0
    while i < len(lines):
        line = lines[i]
        m = re.match(r"^(\s*)steps\s*:\s*$", line)
        if not m:
            i += 1
            continue
        base_indent = len(m.group(1))
        i += 1
        while i < len(lines):
            ln = lines[i]
            if not ln.strip():
                i += 1
                continue
            indent = len(ln) - len(ln.lstrip(" "))
            if indent <= base_indent:
                break
            if re.match(r"^\s*-\s+(name|uses|run)\s*:", ln):
                total_steps += 1
            i += 1
    return total_steps if total_steps > 0 else None

def parse_runs_on_from_yaml(yaml_text: str) -> str:
    """
    Very simple runs-on detector:
    - collects values of `runs-on:` occurrences
    - buckets them to ubuntu/macos/windows/mixed_or_unknown/unknown
    """
    if not yaml_text or not yaml_text.strip():
        return "unknown"
    y = sanitize_gha_expr(yaml_text)
    vals = re.findall(r"(?im)^\s*runs-on\s*:\s*([^\n#]+)", y)
    buckets: Set[str] = set()
    for v in vals:
        v = v.strip().strip('"').strip("'")
        if not v:
            continue
        buckets.add(bucket_runner_os(v))
    if not buckets:
        return "unknown"
    if len(buckets) == 1:
        return list(buckets)[0]
    return "mixed_or_unknown"

# =========================
# JUnit count parsing (bounded)
# =========================
_JUNIT_XML_HINTS = (
    "junit", "test", "tests", "result", "results", "report", "reports",
    "androidtest", "instrumentation", "connected", "surefire", "TEST-"
)

def _try_parse_junit_xml_counts(xml_bytes: bytes) -> int:
    """Return number of testcases from JUnit XML; 0 if not JUnit or cannot parse."""
    try:
        root = ET.fromstring(xml_bytes)
    except Exception:
        return 0
    tag = (root.tag or "").lower()
    if not (tag.endswith("testsuite") or tag.endswith("testsuites")):
        return 0

    if tag.endswith("testsuite"):
        nodes = [root]
    else:
        nodes = list(root.findall(".//testsuite"))

    total = 0
    for n in nodes:
        t = n.attrib.get("tests")
        if t and re.fullmatch(r"\d+", t.strip()):
            total += int(t.strip())
    if total > 0:
        return total

    tcs = root.findall(".//testcase")
    return len(tcs) if tcs else 0

def extract_junit_cases_from_artifacts(gh: GitHubClient, full_name: str, run_id: int) -> Tuple[Optional[int], str]:
    """
    Returns (junit_cases, source):
      source in {artifacts, none}
    """
    artifacts = list_run_artifacts(gh, full_name, run_id) or []
    if not artifacts:
        return None, "none"

    def score_name(name: str) -> int:
        n = safe_lower(name)
        score = 0
        for kw in ["junit", "test-results", "test_results", "test-result",
                   "androidtest", "instrumentation", "connected",
                   "reports", "report", "results", "surefire"]:
            if kw in n:
                score += 3
        return score

    scored = []
    for a in artifacts:
        nm = a.get("name") or ""
        scored.append((score_name(nm), a))
    scored.sort(key=lambda x: x[0], reverse=True)

    parse_list = [(s, a) for (s, a) in scored if s > 0][:MAX_ARTIFACTS_TO_PARSE]
    if not parse_list:
        return None, "none"

    total_cases = 0
    parsed_any = False

    for _, a in parse_list:
        try:
            aid = int(a.get("id"))
        except Exception:
            continue
        zip_bytes = download_artifact_zip(gh, full_name, aid)
        if not zip_bytes:
            continue
        try:
            zf = zipfile.ZipFile(BytesIO(zip_bytes))
            names = zf.namelist()
        except Exception:
            continue

        xmls = []
        for n in names:
            nl = (n or "").lower()
            if not nl.endswith(".xml"):
                continue
            if any(h.lower() in nl for h in _JUNIT_XML_HINTS) or re.search(r"(?i)/TEST-[^/]+\.xml$", n):
                xmls.append(n)

        seen = set()
        xmls2 = []
        for x in xmls:
            if x in seen:
                continue
            seen.add(x)
            xmls2.append(x)

        for xn in xmls2[:MAX_XMLS_PER_ARTIFACT]:
            try:
                raw = zf.read(xn)
            except Exception:
                continue
            if not raw or len(raw) > 2_000_000:
                continue
            c = _try_parse_junit_xml_counts(raw)
            if c > 0:
                total_cases += c
                parsed_any = True

    if parsed_any and total_cases > 0:
        return total_cases, "artifacts"
    return None, "none"

# =========================
# MAIN
# =========================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    run_rows, _ = read_csv_rows(IN_RUN_METRICS_CSV)
    step_rows, _ = read_csv_rows(IN_RUN_STEPS_CSV)

    # Index step rows by (full_name, run_id)
    steps_by_run: Dict[Tuple[str, str], List[Dict[str, str]]] = {}
    jobs_by_run: Dict[Tuple[str, str], Set[str]] = {}
    runner_os_by_run: Dict[Tuple[str, str], Set[str]] = {}

    step_runner_keys = ["runner_os", "runs_on", "runner_labels", "runner_name", "os"]
    for s in step_rows:
        fn_raw = first_nonempty(s, ["full_name", "repo_full_name", "repository", "repo", "repo_name"])
        fn = normalize_full_name(fn_raw)
        rid = (s.get("run_id") or "").strip()
        if not fn or not rid:
            continue
        key = (fn, rid)
        steps_by_run.setdefault(key, []).append(s)

        # --- ONLY CHANGE HERE: use stable identity instead of (job_id or job_name) ---
        jid = _job_identity_from_step_row(s)
        if jid:
            jobs_by_run.setdefault(key, set()).add(jid)

        ros_raw = first_nonempty(s, step_runner_keys)
        if ros_raw:
            runner_os_by_run.setdefault(key, set()).add(bucket_runner_os(ros_raw))

    yaml_cache: Dict[Tuple[str, str, str], str] = {}
    out_rows: List[Dict[str, str]] = []

    for r in run_rows:
        instru_job_count = to_int_loose(r.get("instru_job_count", ""), 0)
        if instru_job_count <= 0:
            continue

        fn_raw = first_nonempty(r, ["full_name", "repo_full_name", "repository", "repo", "repo_name", "repo_url", "html_url"])
        full_name = normalize_full_name(fn_raw)
        run_id_s = (r.get("run_id") or "").strip()

        workflow_path = (r.get("workflow_path") or "").strip()
        head_sha = (r.get("head_sha") or "").strip()

        if not full_name or not run_id_s:
            continue

        run_key = (full_name, run_id_s)
        sr_list = steps_by_run.get(run_key, [])
        has_steps_rows = bool(sr_list)

        yaml_text = ""
        has_yaml = False
        step_count_decl = None
        yaml_runner_bucket = "unknown"

        if FETCH_WORKFLOW_YAML and workflow_path and head_sha:
            ck = (full_name, workflow_path, head_sha)
            if ck in yaml_cache:
                yaml_text = yaml_cache[ck]
            else:
                yaml_text = fetch_workflow_yaml(gh, full_name, workflow_path, head_sha)
                if len(yaml_cache) < WORKFLOW_YAML_CACHE_MAX:
                    yaml_cache[ck] = yaml_text
            has_yaml = bool((yaml_text or "").strip())
            if has_yaml:
                step_count_decl = count_declared_steps_from_yaml(yaml_text)
                yaml_runner_bucket = parse_runs_on_from_yaml(yaml_text)

        run_os_raw = first_nonempty(r, ["runner_os", "runs_on", "os", "runner_labels"])
        if run_os_raw:
            runner_os_bucket = bucket_runner_os(run_os_raw)
            runner_os_source = "run_metrics"
        else:
            os_set = runner_os_by_run.get(run_key, set())
            if len(os_set) == 1:
                runner_os_bucket = list(os_set)[0]
                runner_os_source = "steps"
            elif len(os_set) > 1:
                runner_os_bucket = "mixed_or_unknown"
                runner_os_source = "steps"
            else:
                runner_os_bucket = yaml_runner_bucket
                runner_os_source = "yaml" if yaml_runner_bucket != "unknown" else "unknown"

        # job_count_total (parse formatted; else compute from step rows unique jobs)
        job_count_total = None
        job_count_raw = first_nonempty(r, ["job_count_total", "jobs_total", "total_jobs", "jobs_count"])
        job_count_total = parse_int_strict(job_count_raw)

        if job_count_total is None:
            if jobs_by_run.get(run_key):
                job_count_total = len(jobs_by_run[run_key])

        job_count_total_bucket = bucket_job_count(job_count_total)

        step_count_exec = len(sr_list) if has_steps_rows else None
        step_count_exec_bucket = bucket_step_count(step_count_exec) if step_count_exec is not None else "unknown"
        step_count_decl_bucket = bucket_step_count(step_count_decl) if step_count_decl is not None else "unknown"

        if step_count_exec is not None:
            step_count_total_bucket = step_count_exec_bucket
            step_count_source = "executed"
        elif step_count_decl is not None:
            step_count_total_bucket = step_count_decl_bucket
            step_count_source = "declared"
        else:
            step_count_total_bucket = "unknown"
            step_count_source = "unknown"

        junit_cases = None
        junit_source = "none"
        if DOWNLOAD_AND_PARSE_ARTIFACTS:
            try:
                junit_cases, junit_source = extract_junit_cases_from_artifacts(gh, full_name, int(run_id_s))
            except Exception:
                junit_cases, junit_source = (None, "none")
        test_suite_size_bucket = bucket_suite_size(junit_cases)

        signature_inputs_parts = []
        if has_steps_rows:
            signature_inputs_parts.append("steps")
        if has_yaml:
            signature_inputs_parts.append("yaml")
        if junit_source == "artifacts":
            signature_inputs_parts.append("artifacts")
        signature_inputs = "+".join(signature_inputs_parts) if signature_inputs_parts else ""

        sig_basis_base = "\n".join([
            f"runner_os_bucket={runner_os_bucket}",
            f"job_count_total_bucket={job_count_total_bucket}",
            f"step_count_total_bucket={step_count_total_bucket}",
        ])
        signature_hash_base = hashlib.sha256(sig_basis_base.encode("utf-8", errors="ignore")).hexdigest()[:16]

        sig_basis_full = "\n".join([
            f"runner_os_bucket={runner_os_bucket}",
            f"job_count_total_bucket={job_count_total_bucket}",
            f"step_count_total_bucket={step_count_total_bucket}",
            f"test_suite_size_bucket={test_suite_size_bucket}",
        ])
        signature_hash_full = hashlib.sha256(sig_basis_full.encode("utf-8", errors="ignore")).hexdigest()[:16]

        signature_hash = signature_hash_full

        out_rows.append({
            "full_name": full_name,
            "run_id": run_id_s,
            "workflow_identifier": r.get("workflow_identifier", ""),
            "workflow_path": workflow_path,
            "head_sha": head_sha,

            "signature_inputs": signature_inputs,

            "runner_os_bucket": runner_os_bucket,
            "runner_os_source": runner_os_source,

            "job_count_total": str(job_count_total) if job_count_total is not None else "",
            "job_count_total_bucket": job_count_total_bucket,

            "step_count_exec": str(step_count_exec) if step_count_exec is not None else "",
            "step_count_exec_bucket": step_count_exec_bucket,
            "step_count_decl": str(step_count_decl) if step_count_decl is not None else "",
            "step_count_decl_bucket": step_count_decl_bucket,
            "step_count_source": step_count_source,
            "step_count_total_bucket": step_count_total_bucket,

            "junit_cases": str(junit_cases) if junit_cases is not None else "",
            "junit_source": junit_source,
            "test_suite_size_bucket": test_suite_size_bucket,

            "sig_basis_base": sig_basis_base,
            "signature_hash_base": signature_hash_base,
            "sig_basis_full": sig_basis_full,
            "signature_hash_full": signature_hash_full,

            "signature_hash": signature_hash,

            "stage4_extracted_at_utc": now_utc_iso(),
        })

    out_fields = [
        "full_name","run_id","workflow_identifier","workflow_path","head_sha",
        "signature_inputs",
        "runner_os_bucket","runner_os_source",
        "job_count_total","job_count_total_bucket",
        "step_count_exec","step_count_exec_bucket",
        "step_count_decl","step_count_decl_bucket",
        "step_count_source","step_count_total_bucket",
        "junit_cases","junit_source","test_suite_size_bucket",
        "sig_basis_base","signature_hash_base",
        "sig_basis_full","signature_hash_full",
        "signature_hash",
        "stage4_extracted_at_utc",
    ]

    write_csv(OUT_STAGE4_SIGNATURE_CSV, out_fields, out_rows)
    print("[done] Stage 4 signature (Stage3-only, base+full):", OUT_STAGE4_SIGNATURE_CSV)

if __name__ == "__main__":
    main()

[done] Stage 4 signature (Stage3-only, base+full): C:\Android Mobile App\ICST2026_Ext\run_workload_signature_v3.csv


#merge the related datasets to form the Main Dataset which alingns with the Steps telemetry covers the need from RQ1 to RQ5.

In [5]:
# -*- coding: utf-8 -*-
"""
build_total_dataset_v13.py

Build MainDataset.csv for the study by merging:
- Stage 3 run×style dataset: run_per_style_v1_stage3.csv
- Stage 4 signature dataset: run_workload_signature_v3.csv

This version is aligned with the updated layered timeline decomposition model.

MainDataset keeps:
- identifiers and controllers
- robustness/fingerprint fields
- final study-facing timing variables
- canonical boundary provenance carried from Stage 3
- direct / fallback instrumentation-window candidates
- signature-residual candidate for later policy validation
- compact timing-consistency audit fields

Key updates in this version:
- Consumes the new Stage 3 boundary provenance structure:
    *_selected_source_type
    *_selected_source_layer
    *_selected_observational_quality
    *_selected_semantic_quality
    *_selected_step_name
    *_selected_job_name
    *_ambiguity_count
    *_ambiguity_level
- Carries forward:
    study_timeline_evidence_mode
- Treats Stage 2 telemetry fallback as first-degree fallback
- Preserves:
    study_instru_test_window_direct_seconds
    study_instru_test_window_fallback1_seconds
    study_instru_test_window_fallback2_seconds
    study_instru_test_window_stage3_selected_seconds
- Keeps the signature-conditioned residual fallback candidate separate
- Resolved study-facing instrumentation window remains direct-only until RQ1 validates fallback use

MainDataset is filtered to step-aligned / instrumentation-executed rows using:
- instru_job_count > 0 if available
- otherwise Stage 3 evidence availability heuristics
"""

from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Set

import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_STAGE3 = ROOT_DIR / "run_per_style_v1_stage3.csv"
IN_STAGE4 = ROOT_DIR / "run_workload_signature_v3.csv"
OUT_TOTAL = ROOT_DIR / "MainDataset.csv"

SIG_COL_CANDIDATES: List[str] = [
    "signature_hash_base",
    "hash_base",
    "sig_hash_base",
]

STUDY_IN_SCOPE_STYLES = {"Community", "Custom", "GMD", "Third-Party"}
VERDICT_COMPLETE = {"success", "failure"}

MAD_Z_THRESHOLD = 3.5
TUKEY_K = 1.5
EPS = 1e-9

WINDOW_FALLBACK_POLICY = {
    "Community": "pending_rq1_validation",
    "Custom": "pending_rq1_validation",
    "GMD": "pending_rq1_validation",
    "Third-Party": "pending_rq1_validation",
}

MINOR_NEGATIVE_OTHER_SEC = 30
MAJOR_NEGATIVE_OTHER_SEC = 120
MINOR_WINDOW_DECOMP_DIFF_SEC = 30

SIGNATURE_RESIDUAL_MIN_DONORS_ROBUST = 5
SIGNATURE_RESIDUAL_MIN_DONORS_BASE = 5


# =========================
# HELPERS
# =========================
def pick_first_existing(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def ensure_alias(df: pd.DataFrame, target: str, candidates: List[str]) -> None:
    if target in df.columns:
        return
    for c in candidates:
        if c in df.columns:
            df.rename(columns={c: target}, inplace=True)
            return


def to_stripped_str(df: pd.DataFrame, col: str) -> None:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()


def to_int(df: pd.DataFrame, col: str) -> None:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")


def to_num(df: pd.DataFrame, col: str) -> None:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


def mad(x: pd.Series) -> float:
    arr = x.dropna().astype(float).to_numpy()
    if arr.size == 0:
        return float("nan")
    med = float(np.median(arr))
    return float(np.median(np.abs(arr - med)))


def robust_z_mad(x: pd.Series) -> pd.Series:
    xx = x.astype(float)
    med = float(np.nanmedian(xx))
    d = mad(xx)
    if np.isnan(d) or d == 0:
        return pd.Series(np.zeros(len(xx)), index=xx.index, dtype=float)
    return 0.6745 * (xx - med) / d


def clip_tiny_negative_to_zero(series: pd.Series, eps: float = EPS) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    return s.mask((s < 0) & (s > -eps), 0.0)


def canonicalize_ttfts_source_value(x: object) -> Optional[str]:
    """
    Map existing TTFTS provenance labels into compact study-facing provenance.

    Final output labels:
    - direct_step
    - fallback_stage2
    - unknown
    - None (missing/empty before final cleanup)
    """
    if pd.isna(x):
        return None

    s = str(x).strip().lower()
    if s in {"", "none", "nan", "na"}:
        return None

    direct_exact = {
        "explicit_instru_step",
        "emu_runner_action_step",
        "runtime_anchor_job_earliest_step_to_anchor_step",
        "direct_step",
    }
    if s in direct_exact:
        return "direct_step"

    if s.startswith("stage1_anchor_name_match:"):
        return "direct_step"

    direct_tokens = [
        "anchor_step",
        "earliest_step_to_anchor_step",
        "direct",
        "step",
    ]
    if any(tok in s for tok in direct_tokens):
        return "direct_step"

    fallback_exact = {
        "fallback_stage2",
        "s2_time_to_first_instru_from_anchor_job_seconds",
        "s2_time_to_first_instru_seconds",
        "s2_instru_first_started_at",
        "telemetry_stage2",
    }
    if s in fallback_exact:
        return "fallback_stage2"

    fallback_tokens = [
        "fallback",
        "s2",
        "stage2",
        "telemetry",
        "from_anchor_job_seconds",
    ]
    if any(tok in s for tok in fallback_tokens):
        return "fallback_stage2"

    return "unknown"


def build_ttfts_source_final(df: pd.DataFrame) -> pd.Series:
    """
    Canonical TTFTS provenance aligned with the study.

    Priority:
      1) explicit ttfts_source / modified_ttfts_source
      2) infer direct_step if Stage 3 selected instrumentation start is direct
      3) infer fallback_stage2 if selected instrumentation start is first-degree fallback
      4) infer direct if first_test_step_started_at exists and final ttfts exists
      5) infer fallback if S2 fallback exists and final ttfts exists
      6) missing if final ttfts missing
      7) unknown otherwise
    """
    out = pd.Series(index=df.index, dtype="object")
    explicit = pd.Series(index=df.index, dtype="object")

    if "ttfts_source" in df.columns:
        explicit = explicit.combine_first(df["ttfts_source"].map(canonicalize_ttfts_source_value))
    if "modified_ttfts_source" in df.columns:
        explicit = explicit.combine_first(df["modified_ttfts_source"].map(canonicalize_ttfts_source_value))

    out = explicit

    if {
        "instru_started_at_selected_source_layer",
        "ttfts_seconds"
    }.issubset(df.columns):
        selected_layer = df["instru_started_at_selected_source_layer"].fillna("").astype(str).str.strip().str.lower()
        direct_mask = selected_layer.eq("direct") & df["ttfts_seconds"].notna()
        fb1_mask = selected_layer.eq("first_degree_fallback") & df["ttfts_seconds"].notna()

        out = out.where(~(out.isna() & direct_mask), "direct_step")
        out = out.where(~(out.isna() & fb1_mask), "fallback_stage2")

    if "first_test_step_started_at" in df.columns and "ttfts_seconds" in df.columns:
        mask_direct = df["first_test_step_started_at"].notna() & df["ttfts_seconds"].notna()
        out = out.where(~(out.isna() & mask_direct), "direct_step")

    if "S2_time_to_first_instru_seconds" in df.columns and "ttfts_seconds" in df.columns:
        no_direct_marker = (
            df["first_test_step_started_at"].isna()
            if "first_test_step_started_at" in df.columns
            else True
        )
        mask_fb = no_direct_marker & df["S2_time_to_first_instru_seconds"].notna() & df["ttfts_seconds"].notna()
        out = out.where(~(out.isna() & mask_fb), "fallback_stage2")

    if "ttfts_seconds" in df.columns:
        out = out.where(df["ttfts_seconds"].notna(), "missing")

    return out.fillna("unknown")


def direct_only_source(series: pd.Series, label: str) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    return pd.Series(np.where(s.notna(), label, "missing"), index=series.index, dtype="object")


def simple_available_source(series: pd.Series, label: str) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    return pd.Series(np.where(s.notna(), label, "missing"), index=series.index, dtype="object")


def canonicalize_style(x: object) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    low = s.lower().replace("_", "-")
    if low == "emu-community":
        return "Community"
    if low == "emu-custom":
        return "Custom"
    if low == "gmd":
        return "GMD"
    if low in {"third-party", "third party", "thirdparty"}:
        return "Third-Party"
    if low in {"real-device", "real-devices", "real devices"}:
        return "Real-Devices"
    return s


def canonicalize_layer_value(x: object) -> str:
    if pd.isna(x):
        return "missing"
    s = str(x).strip().lower()
    if s in {"", "none", "nan"}:
        return "missing"
    if s in {"direct", "first_degree_fallback", "second_degree_fallback", "missing"}:
        return s
    return s


def canonicalize_quality_value(x: object, fallback: str = "unknown") -> str:
    if pd.isna(x):
        return fallback
    s = str(x).strip().lower()
    if s in {"", "none", "nan"}:
        return fallback
    return s


def build_timing_consistency_flag(
    other_raw: pd.Series,
    window_decomp_diff: pd.Series,
    ttfts_source: pd.Series,
    window_source: pd.Series,
) -> pd.Series:
    """
    Compact row-level consistency flag for analysis/audit.

    Priority:
      - window decomp mismatches first
      - then negative "other" residuals
      - then mixed-source risk
      - otherwise ok / missing
    """
    other_raw_num = pd.to_numeric(other_raw, errors="coerce")
    diff_num = pd.to_numeric(window_decomp_diff, errors="coerce")

    ttfts_src = ttfts_source.fillna("missing").astype(str)
    win_src = window_source.fillna("missing").astype(str)

    out = pd.Series("ok", index=other_raw.index, dtype="object")

    both_missing = other_raw_num.isna() & diff_num.isna()
    out.loc[both_missing] = "missing"

    major_decomp = diff_num < -MINOR_WINDOW_DECOMP_DIFF_SEC
    out.loc[major_decomp] = "window_decomp_mismatch"

    major_neg_other = (other_raw_num < -MAJOR_NEGATIVE_OTHER_SEC) & ~major_decomp
    out.loc[major_neg_other] = "major_negative_other"

    minor_neg_other = (
        (other_raw_num < -MINOR_NEGATIVE_OTHER_SEC)
        & (other_raw_num >= -MAJOR_NEGATIVE_OTHER_SEC)
        & ~major_decomp
    )
    out.loc[minor_neg_other] = "minor_negative_other"

    mixed_source = (
        (ttfts_src == "fallback_stage2")
        & win_src.isin({"direct_instru_path_window", "direct_validated_fallback"})
        & out.eq("ok")
    )
    out.loc[mixed_source] = "mixed_source_risk"

    return out


def build_signature_residual_window_candidates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build signature-conditioned residual fallback candidates for the
    instrumentation-testing window.

    Candidate:
      run_duration - ttfts - median_other(signature donors)

    Kept as candidate only.
    """
    work = df.copy()

    required_mask = (
        pd.to_numeric(work["study_run_duration_seconds"], errors="coerce").notna()
        & pd.to_numeric(work["study_ttfts_seconds"], errors="coerce").notna()
        & pd.to_numeric(work["study_instru_test_window_direct_seconds"], errors="coerce").notna()
        & work["signature_hash_base"].notna()
        & work["style"].isin(STUDY_IN_SCOPE_STYLES)
    )

    trusted_other = pd.Series(np.nan, index=work.index, dtype=float)
    trusted_other.loc[required_mask] = (
        pd.to_numeric(work.loc[required_mask, "study_run_duration_seconds"], errors="coerce")
        - pd.to_numeric(work.loc[required_mask, "study_ttfts_seconds"], errors="coerce")
        - pd.to_numeric(work.loc[required_mask, "study_instru_test_window_direct_seconds"], errors="coerce")
    )

    donor_mask_base = (
        required_mask
        & work["Base"].fillna(False)
        & trusted_other.notna()
        & (trusted_other >= -MINOR_NEGATIVE_OTHER_SEC)
    )
    donor_mask_robust = donor_mask_base & work["Robust"].fillna(False)

    robust_stats = (
        pd.DataFrame({
            "signature_hash_base": work.loc[donor_mask_robust, "signature_hash_base"].astype(str),
            "trusted_other": trusted_other.loc[donor_mask_robust].astype(float),
        })
        .groupby("signature_hash_base", dropna=False)["trusted_other"]
        .agg(
            signature_residual_median_other_robust="median",
            signature_residual_donor_n_robust="count",
        )
        .reset_index()
    )

    base_stats = (
        pd.DataFrame({
            "signature_hash_base": work.loc[donor_mask_base, "signature_hash_base"].astype(str),
            "trusted_other": trusted_other.loc[donor_mask_base].astype(float),
        })
        .groupby("signature_hash_base", dropna=False)["trusted_other"]
        .agg(
            signature_residual_median_other_base="median",
            signature_residual_donor_n_base="count",
        )
        .reset_index()
    )

    work["signature_hash_base_str"] = work["signature_hash_base"].astype(str)
    work = work.merge(
        robust_stats,
        left_on="signature_hash_base_str",
        right_on="signature_hash_base",
        how="left",
    )
    work = work.merge(
        base_stats,
        left_on="signature_hash_base_str",
        right_on="signature_hash_base",
        how="left",
        suffixes=("_robustmerge", "_basemerge"),
    )

    robust_ok = pd.to_numeric(work["signature_residual_donor_n_robust"], errors="coerce").fillna(0) >= SIGNATURE_RESIDUAL_MIN_DONORS_ROBUST
    base_ok = pd.to_numeric(work["signature_residual_donor_n_base"], errors="coerce").fillna(0) >= SIGNATURE_RESIDUAL_MIN_DONORS_BASE

    selected_scope = pd.Series("missing", index=work.index, dtype="object")
    selected_other = pd.Series(np.nan, index=work.index, dtype=float)
    selected_n = pd.Series(np.nan, index=work.index, dtype=float)

    selected_scope.loc[robust_ok] = "same_signature_robust"
    selected_other.loc[robust_ok] = pd.to_numeric(
        work.loc[robust_ok, "signature_residual_median_other_robust"], errors="coerce"
    )
    selected_n.loc[robust_ok] = pd.to_numeric(
        work.loc[robust_ok, "signature_residual_donor_n_robust"], errors="coerce"
    )

    base_only = (~robust_ok) & base_ok
    selected_scope.loc[base_only] = "same_signature_base"
    selected_other.loc[base_only] = pd.to_numeric(
        work.loc[base_only, "signature_residual_median_other_base"], errors="coerce"
    )
    selected_n.loc[base_only] = pd.to_numeric(
        work.loc[base_only, "signature_residual_donor_n_base"], errors="coerce"
    )

    target_ready = (
        pd.to_numeric(work["study_run_duration_seconds"], errors="coerce").notna()
        & pd.to_numeric(work["study_ttfts_seconds"], errors="coerce").notna()
        & work["signature_hash_base"].notna()
        & selected_other.notna()
    )

    candidate = pd.Series(np.nan, index=work.index, dtype=float)
    candidate.loc[target_ready] = (
        pd.to_numeric(work.loc[target_ready, "study_run_duration_seconds"], errors="coerce")
        - pd.to_numeric(work.loc[target_ready, "study_ttfts_seconds"], errors="coerce")
        - pd.to_numeric(selected_other.loc[target_ready], errors="coerce")
    )

    candidate = candidate.mask(candidate < 0, np.nan)

    df["study_instru_test_window_signature_residual_candidate_seconds"] = candidate
    df["study_instru_test_window_signature_residual_donor_n"] = selected_n.astype("Float64")
    df["study_instru_test_window_signature_residual_scope"] = selected_scope

    df["study_instru_test_window_signature_residual_source"] = np.where(
        pd.to_numeric(df["study_instru_test_window_signature_residual_candidate_seconds"], errors="coerce").notna(),
        "signature_residual_other",
        "missing",
    )

    return df


def add_missing_columns(df: pd.DataFrame, cols: List[str], default=np.nan) -> None:
    for c in cols:
        if c not in df.columns:
            df[c] = default


# =========================
# MAIN
# =========================
def main() -> None:
    stage3 = pd.read_csv(IN_STAGE3, low_memory=False)
    stage4 = pd.read_csv(IN_STAGE4, low_memory=False)

    # Normalize repo/run keys
    for d in (stage3, stage4):
        if "repo_full_name" in d.columns and "full_name" not in d.columns:
            d.rename(columns={"repo_full_name": "full_name"}, inplace=True)
        if "repository" in d.columns and "full_name" not in d.columns:
            d.rename(columns={"repository": "full_name"}, inplace=True)

    for col in ["full_name", "run_id"]:
        if col not in stage3.columns:
            raise ValueError(f"Stage 3 missing required column: {col}")
        if col not in stage4.columns:
            raise ValueError(f"Stage 4 missing required column: {col}")

    to_stripped_str(stage3, "full_name")
    to_stripped_str(stage3, "run_id")
    to_stripped_str(stage4, "full_name")
    to_stripped_str(stage4, "run_id")

    # -------------------------------------------------
    # FILTER TO STEP-ALIGNED / INSTRUMENTATION-EXECUTED ROWS
    # -------------------------------------------------
    # Prefer instru_job_count if present.
    # Otherwise use Stage 3 evidence availability.
    if "instru_job_count" in stage3.columns:
        stage3["instru_job_count"] = pd.to_numeric(stage3["instru_job_count"], errors="coerce").fillna(0)
        stage3 = stage3.loc[stage3["instru_job_count"] > 0].copy()
    else:
        for c in [
            "instru_exec_step_count",
            "instru_test_window_stage3_selected_seconds",
            "instru_started_at",
            "test_exec_started_at",
            "instru_ended_at",
        ]:
            if c in stage3.columns:
                pass

        evidence_mask = pd.Series(False, index=stage3.index)
        if "instru_exec_step_count" in stage3.columns:
            evidence_mask = evidence_mask | (pd.to_numeric(stage3["instru_exec_step_count"], errors="coerce").fillna(0) > 0)
        if "instru_test_window_stage3_selected_seconds" in stage3.columns:
            evidence_mask = evidence_mask | pd.to_numeric(stage3["instru_test_window_stage3_selected_seconds"], errors="coerce").notna()
        if "instru_started_at" in stage3.columns:
            evidence_mask = evidence_mask | stage3["instru_started_at"].notna()
        if "test_exec_started_at" in stage3.columns:
            evidence_mask = evidence_mask | stage3["test_exec_started_at"].notna()
        if "instru_ended_at" in stage3.columns:
            evidence_mask = evidence_mask | stage3["instru_ended_at"].notna()

        stage3 = stage3.loc[evidence_mask].copy()
        stage3["instru_job_count"] = np.where(
            pd.to_numeric(stage3.get("instru_exec_step_count", np.nan), errors="coerce").notna(),
            pd.to_numeric(stage3.get("instru_exec_step_count", np.nan), errors="coerce"),
            np.nan,
        )

    # -------------------------------------------------
    # STAGE 4 SIGNATURE
    # -------------------------------------------------
    sig_col = pick_first_existing(stage4, SIG_COL_CANDIDATES)
    if sig_col is None:
        raise ValueError(f"Stage 4 must contain one of the BASE signature columns: {SIG_COL_CANDIDATES}")
    if sig_col != "signature_hash_base":
        stage4.rename(columns={sig_col: "signature_hash_base"}, inplace=True)

    keep4 = ["full_name", "run_id", "signature_hash_base"]
    for extra in ["runner_os_bucket", "job_count_total_bucket", "step_count_total_bucket"]:
        if extra in stage4.columns:
            keep4.append(extra)

    stage4_small = stage4[keep4].drop_duplicates(subset=["full_name", "run_id"])
    df = stage3.merge(stage4_small, on=["full_name", "run_id"], how="left", validate="m:1")

    # -------------------------------------------------
    # HARMONIZE NAMES
    # -------------------------------------------------
    ensure_alias(df, "event", ["trigger", "event_name", "run_event", "github_event_name"])
    ensure_alias(df, "run_attempt", ["attempt", "run_attempt_number"])
    to_int(df, "run_attempt")

    ensure_alias(df, "style", ["target_style", "execution_style", "provider_category", "style_label"])
    to_stripped_str(df, "style")
    df["style"] = df["style"].map(canonicalize_style)

    if "styles" not in df.columns and "inferred_styles_all" in df.columns:
        df["styles"] = df["inferred_styles_all"]

    ensure_alias(df, "run_conclusion", ["conclusion", "workflow_run_conclusion", "run_result"])
    to_stripped_str(df, "run_conclusion")

    if "instru_conclusion" not in df.columns:
        # In run×style Stage 3 files this may be absent; fall back to run_conclusion if needed.
        df["instru_conclusion"] = df["run_conclusion"] if "run_conclusion" in df.columns else ""

    to_stripped_str(df, "instru_conclusion")

    ensure_alias(df, "created_at", ["run_created_at", "workflow_created_at", "stage3_extracted_at_utc"])
    ensure_alias(df, "run_started_at", ["started_at", "workflow_run_started_at"])
    ensure_alias(df, "run_duration_seconds", ["duration_seconds"])
    ensure_alias(df, "queue_seconds", ["queue_duration_seconds"])

    ensure_alias(df, "ttfts_seconds", ["ttfts_seconds_modified"])
    ensure_alias(df, "instru_duration_seconds", ["instrumentation_duration_seconds"])
    ensure_alias(df, "core_instru_window_seconds", ["core_execution_window_seconds"])
    ensure_alias(df, "instru_exec_window_seconds", ["exec_window_seconds", "test_exec_window_seconds"])

    ensure_alias(df, "instru_window_seconds", ["instrumentation_window_seconds", "telemetry_instru_window_seconds"])
    ensure_alias(df, "instru_total_seconds", ["instrumentation_total_seconds"])

    ensure_alias(df, "instru_started_at", ["instrumentation_started_at", "instr_started_at"])
    ensure_alias(df, "instru_ended_at", ["instrumentation_ended_at", "instr_ended_at"])
    ensure_alias(df, "test_exec_started_at", ["first_exec_started_at"])
    ensure_alias(df, "test_exec_ended_at", ["last_exec_ended_at"])

    ensure_alias(df, "pre_test_overhead_seconds", ["pre_exec_overhead_seconds"])
    ensure_alias(df, "post_test_overhead_seconds", ["post_exec_overhead_seconds"])

    # Numeric conversion
    for c in [
        "run_duration_seconds",
        "queue_seconds",
        "ttfts_seconds",
        "instru_duration_seconds",
        "core_instru_window_seconds",
        "instru_exec_window_seconds",
        "instru_window_seconds",
        "instru_total_seconds",
        "pre_test_overhead_seconds",
        "post_test_overhead_seconds",
        "S2_time_to_first_instru_seconds",
        "instru_test_window_direct_seconds",
        "instru_test_window_fallback1_seconds",
        "instru_test_window_fallback2_seconds",
        "instru_test_window_stage3_selected_seconds",
        "instru_exec_step_count",
        "instru_started_at_ambiguity_count",
        "test_exec_started_at_ambiguity_count",
        "test_exec_ended_at_ambiguity_count",
        "instru_ended_at_ambiguity_count",
    ]:
        to_num(df, c)

    # Layer / quality cleanup if present
    for c in [
        "instru_started_at_selected_source_layer",
        "test_exec_started_at_selected_source_layer",
        "test_exec_ended_at_selected_source_layer",
        "instru_ended_at_selected_source_layer",
        "instru_test_window_stage3_selected_source_layer",
        "instru_test_window_fallback1_source_layer",
        "instru_test_window_fallback2_source_layer",
    ]:
        if c in df.columns:
            df[c] = df[c].map(canonicalize_layer_value)

    for c in [
        "instru_started_at_selected_observational_quality",
        "test_exec_started_at_selected_observational_quality",
        "test_exec_ended_at_selected_observational_quality",
        "instru_ended_at_selected_observational_quality",
        "instru_started_at_selected_semantic_quality",
        "test_exec_started_at_selected_semantic_quality",
        "test_exec_ended_at_selected_semantic_quality",
        "instru_ended_at_selected_semantic_quality",
        "instru_started_at_ambiguity_level",
        "test_exec_started_at_ambiguity_level",
        "test_exec_ended_at_ambiguity_level",
        "instru_ended_at_ambiguity_level",
        "timeline_evidence_mode",
    ]:
        if c in df.columns:
            df[c] = df[c].fillna("missing").astype(str).str.strip()

    # If only one execution-span field exists, synchronize
    if "core_instru_window_seconds" in df.columns and "instru_exec_window_seconds" in df.columns:
        fill_a = df["core_instru_window_seconds"].isna() & df["instru_exec_window_seconds"].notna()
        df.loc[fill_a, "core_instru_window_seconds"] = df.loc[fill_a, "instru_exec_window_seconds"]

        fill_b = df["instru_exec_window_seconds"].isna() & df["core_instru_window_seconds"].notna()
        df.loc[fill_b, "instru_exec_window_seconds"] = df.loc[fill_b, "core_instru_window_seconds"]

    # -------------------------------------------------
    # PAPER-ALIGNED CONTROLLER
    # -------------------------------------------------
    is_in_scope_style = df["style"].isin(STUDY_IN_SCOPE_STYLES)

    # If run_attempt missing in run×style input, default to 1.
    if "run_attempt" not in df.columns:
        df["run_attempt"] = 1
    is_attempt1 = pd.to_numeric(df["run_attempt"], errors="coerce").fillna(1).astype("Int64").eq(1)

    instr_concl = df["instru_conclusion"].fillna("").astype(str).str.lower()
    is_verdict_complete = instr_concl.isin(VERDICT_COMPLETE)

    df["Base"] = is_in_scope_style & is_attempt1 & is_verdict_complete

    # -------------------------------------------------
    # ROBUST FLAG
    # -------------------------------------------------
    base_runs = (
        df.loc[df["Base"] & df["signature_hash_base"].notna(), ["signature_hash_base", "full_name", "run_id"]]
        .drop_duplicates()
        .copy()
    )

    sig_n_runs = base_runs.groupby("signature_hash_base")["run_id"].nunique().rename("n_runs")
    sig_n_repos = base_runs.groupby("signature_hash_base")["full_name"].nunique().rename("n_repos")

    sig_stats = pd.concat([sig_n_runs, sig_n_repos], axis=1).reset_index()
    sig_stats["runs_per_repo"] = sig_stats["n_runs"] / sig_stats["n_repos"].replace(0, np.nan)

    q1 = sig_stats["runs_per_repo"].quantile(0.25)
    q3 = sig_stats["runs_per_repo"].quantile(0.75)
    iqr = q3 - q1
    tukey_upper = q3 + TUKEY_K * iqr
    sig_stats["out_tukey"] = sig_stats["runs_per_repo"] > tukey_upper

    sig_stats["z_mad"] = robust_z_mad(sig_stats["runs_per_repo"])
    sig_stats["out_mad"] = sig_stats["z_mad"].abs() > MAD_Z_THRESHOLD

    sig_stats["outlier_signature"] = sig_stats["out_tukey"] & sig_stats["out_mad"]
    outlier_sigs: Set[str] = set(sig_stats.loc[sig_stats["outlier_signature"], "signature_hash_base"].astype(str))

    df["Robust"] = (
        df["Base"]
        & df["signature_hash_base"].notna()
        & (~df["signature_hash_base"].astype(str).isin(outlier_sigs))
    )

    # -------------------------------------------------
    # STUDY-FACING CANONICAL VARIABLES
    # -------------------------------------------------
    df["study_run_duration_seconds"] = df["run_duration_seconds"] if "run_duration_seconds" in df.columns else np.nan
    df["study_run_duration_source_final"] = simple_available_source(
        df["study_run_duration_seconds"], "run_metadata_duration"
    )

    df["study_queue_seconds"] = df["queue_seconds"] if "queue_seconds" in df.columns else np.nan
    df["study_queue_source_final"] = simple_available_source(
        df["study_queue_seconds"], "trigger_to_run_start"
    )

    df["study_ttfts_seconds"] = df["ttfts_seconds"] if "ttfts_seconds" in df.columns else np.nan
    df["study_ttfts_source_final"] = build_ttfts_source_final(df)

    df["study_ttfts_overlap_valid"] = (
        pd.to_numeric(df["study_ttfts_seconds"], errors="coerce").notna()
        & pd.to_numeric(df.get("S2_time_to_first_instru_seconds", np.nan), errors="coerce").notna()
        & df["study_ttfts_source_final"].eq("direct_step")
    )

    # -------------------------------------------------
    # CARRY FORWARD STAGE 3 PROVENANCE
    # -------------------------------------------------
    carry_cols = [
        "timeline_evidence_mode",

        "instru_started_at_selected_source_type",
        "instru_started_at_selected_source_layer",
        "instru_started_at_selected_observational_quality",
        "instru_started_at_selected_semantic_quality",
        "instru_started_at_selected_step_name",
        "instru_started_at_selected_job_name",
        "instru_started_at_ambiguity_count",
        "instru_started_at_ambiguity_level",

        "test_exec_started_at_selected_source_type",
        "test_exec_started_at_selected_source_layer",
        "test_exec_started_at_selected_observational_quality",
        "test_exec_started_at_selected_semantic_quality",
        "test_exec_started_at_selected_step_name",
        "test_exec_started_at_selected_job_name",
        "test_exec_started_at_ambiguity_count",
        "test_exec_started_at_ambiguity_level",

        "test_exec_ended_at_selected_source_type",
        "test_exec_ended_at_selected_source_layer",
        "test_exec_ended_at_selected_observational_quality",
        "test_exec_ended_at_selected_semantic_quality",
        "test_exec_ended_at_selected_step_name",
        "test_exec_ended_at_selected_job_name",
        "test_exec_ended_at_ambiguity_count",
        "test_exec_ended_at_ambiguity_level",

        "instru_ended_at_selected_source_type",
        "instru_ended_at_selected_source_layer",
        "instru_ended_at_selected_observational_quality",
        "instru_ended_at_selected_semantic_quality",
        "instru_ended_at_selected_step_name",
        "instru_ended_at_selected_job_name",
        "instru_ended_at_ambiguity_count",
        "instru_ended_at_ambiguity_level",

        "instru_test_window_fallback1_seconds",
        "instru_test_window_fallback1_source_type",
        "instru_test_window_fallback1_source_layer",
        "instru_test_window_fallback1_rank",

        "instru_test_window_fallback2_seconds",
        "instru_test_window_fallback2_source_type",
        "instru_test_window_fallback2_source_layer",
        "instru_test_window_fallback2_rank",

        "instru_test_window_stage3_selected_seconds",
        "instru_test_window_stage3_selected_source_type",
        "instru_test_window_stage3_selected_source_layer",
    ]

    add_missing_columns(df, carry_cols, default=np.nan)

    for c in carry_cols:
        df[f"study_{c}"] = df[c]

    if "study_timeline_evidence_mode" in df.columns:
        df["study_timeline_evidence_mode"] = df["study_timeline_evidence_mode"].fillna("missing")

    # -------------------------------------------------
    # INSTRUMENTATION-TEST WINDOW: POLICY-READY LAYERING
    # -------------------------------------------------
    # Direct baseline from Stage 3 direct path window
    if "instru_duration_seconds" in df.columns:
        df["study_instru_test_window_direct_seconds"] = df["instru_duration_seconds"]
    else:
        df["study_instru_test_window_direct_seconds"] = np.nan

    # Backward-compat / legacy candidate
    if "instru_window_seconds" in df.columns:
        df["study_instru_test_window_fallback_candidate_seconds"] = df["instru_window_seconds"]
    else:
        df["study_instru_test_window_fallback_candidate_seconds"] = np.nan

    # Signature-conditioned residual candidate
    df = build_signature_residual_window_candidates(df)

    # Overlap helpers
    df["study_instru_test_window_overlap_valid"] = (
        pd.to_numeric(df["study_instru_test_window_direct_seconds"], errors="coerce").notna()
        & pd.to_numeric(df["study_instru_test_window_fallback_candidate_seconds"], errors="coerce").notna()
    )

    df["study_instru_test_window_signature_residual_overlap_valid"] = (
        pd.to_numeric(df["study_instru_test_window_direct_seconds"], errors="coerce").notna()
        & pd.to_numeric(df["study_instru_test_window_signature_residual_candidate_seconds"], errors="coerce").notna()
    )

    df["study_instru_test_window_resolution_policy"] = df["style"].map(
        lambda s: WINDOW_FALLBACK_POLICY.get(s, "pending_rq1_validation")
    )
    df["study_instru_test_window_fallback_eligible"] = "unknown"

    # Resolved study-facing field remains direct-only for now
    df["study_instru_test_window_seconds"] = df["study_instru_test_window_direct_seconds"]
    df["study_instru_test_window_source_final"] = np.where(
        pd.to_numeric(df["study_instru_test_window_direct_seconds"], errors="coerce").notna(),
        "direct_instru_path_window",
        "missing",
    )

    # -------------------------------------------------
    # OTHER / DECOMPOSITION
    # -------------------------------------------------
    df["study_other_raw_seconds"] = np.where(
        pd.to_numeric(df["study_run_duration_seconds"], errors="coerce").notna()
        & pd.to_numeric(df["study_ttfts_seconds"], errors="coerce").notna()
        & pd.to_numeric(df["study_instru_test_window_seconds"], errors="coerce").notna(),
        pd.to_numeric(df["study_run_duration_seconds"], errors="coerce")
        - pd.to_numeric(df["study_ttfts_seconds"], errors="coerce")
        - pd.to_numeric(df["study_instru_test_window_seconds"], errors="coerce"),
        np.nan,
    )
    df["study_other_seconds"] = clip_tiny_negative_to_zero(df["study_other_raw_seconds"])
    df["study_other_source_final"] = np.where(
        pd.to_numeric(df["study_other_seconds"], errors="coerce").notna(),
        "derived_from_run_ttfts_resolved_window",
        "missing",
    )

    df["study_pre_exec_seconds"] = (
        df["pre_test_overhead_seconds"] if "pre_test_overhead_seconds" in df.columns else np.nan
    )
    df["study_exec_span_seconds"] = (
        df["instru_exec_window_seconds"] if "instru_exec_window_seconds" in df.columns else np.nan
    )
    df["study_post_exec_seconds"] = (
        df["post_test_overhead_seconds"] if "post_test_overhead_seconds" in df.columns else np.nan
    )

    df["study_pre_exec_source_final"] = direct_only_source(
        df["study_pre_exec_seconds"], "direct_instru_path_window"
    )
    df["study_exec_span_source_final"] = direct_only_source(
        df["study_exec_span_seconds"], "direct_instru_exec_span"
    )
    df["study_post_exec_source_final"] = direct_only_source(
        df["study_post_exec_seconds"], "direct_instru_path_window"
    )

    df["study_window_decomp_sum_seconds"] = np.where(
        pd.to_numeric(df["study_pre_exec_seconds"], errors="coerce").notna()
        | pd.to_numeric(df["study_exec_span_seconds"], errors="coerce").notna()
        | pd.to_numeric(df["study_post_exec_seconds"], errors="coerce").notna(),
        pd.to_numeric(df["study_pre_exec_seconds"], errors="coerce").fillna(0)
        + pd.to_numeric(df["study_exec_span_seconds"], errors="coerce").fillna(0)
        + pd.to_numeric(df["study_post_exec_seconds"], errors="coerce").fillna(0),
        np.nan,
    )

    df["study_window_decomp_diff_seconds"] = np.where(
        pd.to_numeric(df["study_instru_test_window_direct_seconds"], errors="coerce").notna()
        & pd.to_numeric(df["study_window_decomp_sum_seconds"], errors="coerce").notna(),
        pd.to_numeric(df["study_instru_test_window_direct_seconds"], errors="coerce")
        - pd.to_numeric(df["study_window_decomp_sum_seconds"], errors="coerce"),
        np.nan,
    )
    df["study_window_decomp_diff_seconds"] = clip_tiny_negative_to_zero(df["study_window_decomp_diff_seconds"])

    df["study_timing_consistency_flag"] = build_timing_consistency_flag(
        other_raw=df["study_other_raw_seconds"],
        window_decomp_diff=df["study_window_decomp_diff_seconds"],
        ttfts_source=df["study_ttfts_source_final"],
        window_source=df["study_instru_test_window_source_final"],
    )

    # -------------------------------------------------
    # OUTPUT ORDERING
    # -------------------------------------------------
    id_cols = [
        "full_name",
        "run_id",
        "workflow_id",
        "workflow_identifier",
        "workflow_path",
        "workflow_ref",
        "run_attempt",
        "event",
        "head_branch",
        "default_branch",
        "head_sha",
        "style",
        "styles",
        "invocation_types",
        "third_party_provider_name",
        "instru_job_count",
    ]

    controller_cols = [
        "run_conclusion",
        "instru_conclusion",
        "Base",
        "Robust",
    ]

    essential_timestamps = [
        "created_at",
        "run_started_at",
        "first_test_step_started_at",
        "instru_started_at",
        "test_exec_started_at",
        "test_exec_ended_at",
        "instru_ended_at",
    ]

    robustness_cols = [
        "signature_hash_base",
        "runner_os_bucket",
        "job_count_total_bucket",
        "step_count_total_bucket",
    ]

    study_timeline_cols = [
        "study_run_duration_seconds",
        "study_run_duration_source_final",
        "study_queue_seconds",
        "study_queue_source_final",
        "study_ttfts_seconds",
        "study_ttfts_source_final",
        "study_ttfts_overlap_valid",

        "study_timeline_evidence_mode",

        "study_instru_started_at_selected_source_type",
        "study_instru_started_at_selected_source_layer",
        "study_instru_started_at_selected_observational_quality",
        "study_instru_started_at_selected_semantic_quality",
        "study_instru_started_at_selected_step_name",
        "study_instru_started_at_selected_job_name",
        "study_instru_started_at_ambiguity_count",
        "study_instru_started_at_ambiguity_level",

        "study_test_exec_started_at_selected_source_type",
        "study_test_exec_started_at_selected_source_layer",
        "study_test_exec_started_at_selected_observational_quality",
        "study_test_exec_started_at_selected_semantic_quality",
        "study_test_exec_started_at_selected_step_name",
        "study_test_exec_started_at_selected_job_name",
        "study_test_exec_started_at_ambiguity_count",
        "study_test_exec_started_at_ambiguity_level",

        "study_test_exec_ended_at_selected_source_type",
        "study_test_exec_ended_at_selected_source_layer",
        "study_test_exec_ended_at_selected_observational_quality",
        "study_test_exec_ended_at_selected_semantic_quality",
        "study_test_exec_ended_at_selected_step_name",
        "study_test_exec_ended_at_selected_job_name",
        "study_test_exec_ended_at_ambiguity_count",
        "study_test_exec_ended_at_ambiguity_level",

        "study_instru_ended_at_selected_source_type",
        "study_instru_ended_at_selected_source_layer",
        "study_instru_ended_at_selected_observational_quality",
        "study_instru_ended_at_selected_semantic_quality",
        "study_instru_ended_at_selected_step_name",
        "study_instru_ended_at_selected_job_name",
        "study_instru_ended_at_ambiguity_count",
        "study_instru_ended_at_ambiguity_level",

        "study_instru_test_window_direct_seconds",
        "study_instru_test_window_fallback1_seconds",
        "study_instru_test_window_fallback1_source_type",
        "study_instru_test_window_fallback1_source_layer",
        "study_instru_test_window_fallback1_rank",
        "study_instru_test_window_fallback2_seconds",
        "study_instru_test_window_fallback2_source_type",
        "study_instru_test_window_fallback2_source_layer",
        "study_instru_test_window_fallback2_rank",
        "study_instru_test_window_stage3_selected_seconds",
        "study_instru_test_window_stage3_selected_source_type",
        "study_instru_test_window_stage3_selected_source_layer",

        "study_instru_test_window_fallback_candidate_seconds",
        "study_instru_test_window_overlap_valid",
        "study_instru_test_window_signature_residual_candidate_seconds",
        "study_instru_test_window_signature_residual_donor_n",
        "study_instru_test_window_signature_residual_scope",
        "study_instru_test_window_signature_residual_source",
        "study_instru_test_window_signature_residual_overlap_valid",
        "study_instru_test_window_resolution_policy",
        "study_instru_test_window_fallback_eligible",
        "study_instru_test_window_seconds",
        "study_instru_test_window_source_final",

        "study_other_raw_seconds",
        "study_other_seconds",
        "study_other_source_final",

        "study_pre_exec_seconds",
        "study_pre_exec_source_final",
        "study_exec_span_seconds",
        "study_exec_span_source_final",
        "study_post_exec_seconds",
        "study_post_exec_source_final",

        "study_window_decomp_sum_seconds",
        "study_window_decomp_diff_seconds",
        "study_timing_consistency_flag",
    ]

    ordered_cols = [
        c for c in (
            id_cols
            + controller_cols
            + essential_timestamps
            + robustness_cols
            + study_timeline_cols
        )
        if c in df.columns
    ]

    df_out = df[ordered_cols].copy()

    # -------------------------------------------------
    # OUTPUT
    # -------------------------------------------------
    df_out.to_csv(OUT_TOTAL, index=False, encoding="utf-8")

    print(f"[done] wrote {OUT_TOTAL}")
    print(f"[info] rows kept after Stage 3 evidence filter: {len(df_out)}")
    print(f"[info] unique runs kept after Stage 3 evidence filter: {df_out[['full_name', 'run_id']].drop_duplicates().shape[0]}")
    print(f"[info] four-style emulator rows: {df_out['style'].isin(STUDY_IN_SCOPE_STYLES).sum()}")
    print(f"[info] Base rows: {int(df_out['Base'].sum())}")
    print(f"[info] Robust rows: {int(df_out['Robust'].sum())}")

    if "study_timeline_evidence_mode" in df_out.columns:
        print("[info] timeline evidence mode counts:")
        print(df_out["study_timeline_evidence_mode"].value_counts(dropna=False).to_string())

    print(f"[info] study_run_duration_seconds available: {df_out['study_run_duration_seconds'].notna().sum()}")
    print(f"[info] study_queue_seconds available: {df_out['study_queue_seconds'].notna().sum()}")
    print(f"[info] study_ttfts_seconds available: {df_out['study_ttfts_seconds'].notna().sum()}")
    print(f"[info] study_ttfts_overlap_valid count: {int(df_out['study_ttfts_overlap_valid'].sum())}")

    print(f"[info] study_instru_test_window_direct_seconds available: {df_out['study_instru_test_window_direct_seconds'].notna().sum()}")
    if "study_instru_test_window_fallback1_seconds" in df_out.columns:
        print(f"[info] study_instru_test_window_fallback1_seconds available: {df_out['study_instru_test_window_fallback1_seconds'].notna().sum()}")
    if "study_instru_test_window_fallback2_seconds" in df_out.columns:
        print(f"[info] study_instru_test_window_fallback2_seconds available: {df_out['study_instru_test_window_fallback2_seconds'].notna().sum()}")
    if "study_instru_test_window_stage3_selected_seconds" in df_out.columns:
        print(f"[info] study_instru_test_window_stage3_selected_seconds available: {df_out['study_instru_test_window_stage3_selected_seconds'].notna().sum()}")

    print(f"[info] study_instru_test_window_fallback_candidate_seconds available: {df_out['study_instru_test_window_fallback_candidate_seconds'].notna().sum()}")
    print(f"[info] study_instru_test_window_overlap_valid count: {int(df_out['study_instru_test_window_overlap_valid'].sum())}")
    print(f"[info] study_instru_test_window_signature_residual_candidate_seconds available: {df_out['study_instru_test_window_signature_residual_candidate_seconds'].notna().sum()}")
    print(f"[info] study_instru_test_window_signature_residual_overlap_valid count: {int(df_out['study_instru_test_window_signature_residual_overlap_valid'].sum())}")
    print(f"[info] study_instru_test_window_seconds available: {df_out['study_instru_test_window_seconds'].notna().sum()}")

    print(f"[info] study_other_seconds available: {df_out['study_other_seconds'].notna().sum()}")
    print(f"[info] study_pre_exec_seconds available: {df_out['study_pre_exec_seconds'].notna().sum()}")
    print(f"[info] study_exec_span_seconds available: {df_out['study_exec_span_seconds'].notna().sum()}")
    print(f"[info] study_post_exec_seconds available: {df_out['study_post_exec_seconds'].notna().sum()}")

    if "study_window_decomp_diff_seconds" in df_out.columns:
        diff_nonnull = pd.to_numeric(df_out["study_window_decomp_diff_seconds"], errors="coerce").dropna()
        exact_zero = int((diff_nonnull == 0).sum()) if len(diff_nonnull) else 0
        print(f"[info] study_window_decomp_diff_seconds non-null: {len(diff_nonnull)}")
        print(f"[info] exact decomposition matches: {exact_zero}")

    if "study_timing_consistency_flag" in df_out.columns:
        print("[info] timing consistency flag counts:")
        print(df_out["study_timing_consistency_flag"].value_counts(dropna=False).to_string())

    print(f"[info] outlier signatures (Tukey & MAD): {len(outlier_sigs)}")
    if outlier_sigs:
        print("       ", sorted(outlier_sigs))


if __name__ == "__main__":
    main()

c:\GitHub\.PCvenv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


AttributeError: 'float' object has no attribute 'notna'